<a href="https://colab.research.google.com/github/Ashu-design/Microbiome-representation-shapes-cross-cohort-transportability-CRC/blob/main/Gut_Publication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Section 1. Data acquisition

In [ ]:
!pip -q install pandas numpy requests tqdm beautifulsoup4 lxml pyreadr pyarrow fastparquet

import os, re, time
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import pyreadr

# -------------------------
# CONFIG
# -------------------------
OUT_DIR = "/content/curatedMetagenomicData_EH"
os.makedirs(OUT_DIR, exist_ok=True)

SAMPLE_METADATA_RDA_URL = "https://huggingface.co/datasets/wwydmanski/metagenomic_curated/resolve/main/sampleMetadata.rda"

TITLE_PAGE = "https://experimenthub.bioconductor.org/title/{study}"
EHID_META_URL = "https://experimenthub.bioconductor.org/ehid/{ehid}"

# You enabled these:
DATA_TYPES = {
    "relative_abundance",
    "pathway_abundance",
    "pathway_coverage",
    "marker_abundance",
    "marker_presence",
    "gene_families",
}

# Original strict set (kept)
CRC_RELEVANT_CONDITIONS = {"CRC", "adenoma", "control"}
# More robust matching (covers variants)
CRC_KEYWORDS = ["crc", "colorectal", "colon", "rectal", "adenoma", "control", "healthy"]

# Your relevant CRC case-control cohorts (union = 11 studies)
species_cc = {
    "YachidaS_2019": 509,
    "WirbelJ_2018": 125,
    "ZellerG_2014": 114,
    "VogtmannE_2016": 104,
    "ThomasAM_2019_c": 80,
    "ThomasAM_2018b": 60,
    "GuptaA_2019": 60,
    "ThomasAM_2018a": 53,
    "HanniganGD_2017": 55,
}
pathways_cc = {
    "YachidaS_2019": 509,
    "YuJ_2015": 128,
    "WirbelJ_2018": 125,
    "ZellerG_2014": 114,
    "VogtmannE_2016": 104,
    "FengQ_2015": 107,
    "ThomasAM_2019_c": 80,
    "ThomasAM_2018b": 60,
    "GuptaA_2019": 60,
    "ThomasAM_2018a": 53,
    "HanniganGD_2017": 55,
}
RELEVANT_STUDIES = sorted(set(species_cc) | set(pathways_cc))
STUDY_PRIORITY = {**pathways_cc, **species_cc}  # n descending preferred

# Budget in GB
BUDGET_GB = 80.0

# Priority of what to include under budget
CORE_DTYPES = ["relative_abundance", "pathway_abundance", "pathway_coverage"]
OPTIONAL_DTYPES = ["marker_abundance", "marker_presence", "gene_families"]

# Networking / politeness
SLEEP_BETWEEN_REQUESTS = 0.2
TIMEOUT = (30, 600)       # connect=30s, read=10min
HEAD_TIMEOUT = (15, 30)
MAX_RETRIES = 5

# -------------------------
# Robust HTTP helpers
# -------------------------
def http_get(url: str, stream: bool = False):
    last = None
    for i in range(MAX_RETRIES):
        try:
            r = requests.get(
                url,
                stream=stream,
                timeout=TIMEOUT,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(min(2 ** i, 10))
    raise last

def http_head(url: str):
    last = None
    for i in range(MAX_RETRIES):
        try:
            r = requests.head(
                url,
                allow_redirects=True,
                timeout=HEAD_TIMEOUT,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            if r.status_code >= 400:
                return None
            return r
        except Exception as e:
            last = e
            time.sleep(min(2 ** i, 10))
    return None

def download_file(url: str, out_path: str, chunk_size: int = 1 << 20):
    """
    Atomic download: writes to .part then renames.
    Skips if file exists and is non-empty.
    """
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return out_path

    tmp_path = out_path + ".part"
    r = http_get(url, stream=True)
    total = int(r.headers.get("content-length", 0))

    with open(tmp_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc=os.path.basename(out_path)
    ) as pbar:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))

    os.replace(tmp_path, out_path)
    return out_path

def safe_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

def guess_ext_from_url(url: str) -> str:
    clean = str(url).split("?", 1)[0]
    ext = os.path.splitext(clean)[1].lower()
    if ext in {".rds", ".rda", ".gz"}:
        return ext
    return ".rds"

# -------------------------
# Load sampleMetadata
# -------------------------
def load_sample_metadata() -> pd.DataFrame:
    local_rda = os.path.join(OUT_DIR, "sampleMetadata.rda")
    download_file(SAMPLE_METADATA_RDA_URL, local_rda)
    objs = pyreadr.read_r(local_rda)
    if "sampleMetadata" not in objs:
        raise ValueError(f"sampleMetadata not found. Keys: {list(objs.keys())}")
    return objs["sampleMetadata"]

# -------------------------
# Scrape EHIDs + resolve
# -------------------------
EHID_RE = re.compile(r"\bEH\d+\b")

def get_ehids_for_study(study_name: str) -> list[str]:
    url = TITLE_PAGE.format(study=study_name)
    r = http_get(url)
    soup = BeautifulSoup(r.text, "lxml")
    text = soup.get_text(" ", strip=True)
    return sorted(set(EHID_RE.findall(text)))

def resolve_ehid_to_url_and_title(ehid: str) -> tuple[str, str]:
    meta = http_get(EHID_META_URL.format(ehid=ehid)).json()
    url = meta["location_prefix"] + meta["rdatapaths"][0]["rdatapath"]
    title = meta.get("title", "")
    return url, title

def normalize_dt_match(title: str, allowed: set[str]) -> str | None:
    """
    Robust dtype match:
    - normalize title and look for key
    - allow suffix variants: key_*
    """
    t = str(title).lower()
    t_norm = re.sub(r"[\s\-\.\:\/]+", "_", t)

    for dt in allowed:
        key = dt.lower()
        if key in t_norm:
            return dt
        if re.search(rf"\b{re.escape(key)}_[a-z0-9]+\b", t_norm):
            return dt
    return None

def head_content_length(url: str) -> int:
    r = http_head(url)
    if r is None:
        return 0
    try:
        return int(r.headers.get("content-length", 0) or 0)
    except Exception:
        return 0

# -------------------------
# Budget selection helpers
# -------------------------
def ensure_sizes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "bytes" not in df.columns or "gb" not in df.columns:
        sizes = []
        for url in tqdm(df["url"].tolist(), desc="Preflighting sizes (HEAD)"):
            sizes.append(head_content_length(url) if isinstance(url, str) and url else 0)
        df["bytes"] = sizes
        df["gb"] = df["bytes"] / (1024**3)
    df["unknown_size"] = df["bytes"].fillna(0).astype(int).eq(0)
    return df

def pick_one_per_dtype(sub: pd.DataFrame, dtype: str) -> pd.Series | None:
    cand = sub[sub["data_type"] == dtype].copy()
    if cand.empty:
        return None
    # prefer known size, then smallest file
    cand = cand.sort_values(["unknown_size", "gb", "ehid"], ascending=[True, True, True])
    return cand.iloc[0]

def budget_select(
    df: pd.DataFrame,
    studies: list[str],
    budget_gb: float,
    core_dtypes: list[str],
    optional_dtypes: list[str],
    priority: dict[str, int],
):
    df = df[df["study_name"].isin(studies)].copy()
    df = ensure_sizes(df)

    ordered_studies = sorted(df["study_name"].unique(), key=lambda s: (-priority.get(s, 0), s))

    chosen_rows = []
    used = 0.0
    chosen_studies = []

    # CORE: include all core matrices for a study or skip the study
    for st in ordered_studies:
        sub = df[df["study_name"] == st]
        core_picks = []
        add_gb = 0.0
        ok = True

        for dt in core_dtypes:
            row = pick_one_per_dtype(sub, dt)
            if row is None:
                ok = False
                break
            core_picks.append(row)
            add_gb += float(row["gb"])

        if ok and (used + add_gb) <= budget_gb:
            chosen_rows.extend(core_picks)
            used += add_gb
            chosen_studies.append(st)

    # OPTIONAL: add per dtype per already-kept study if budget allows
    for dt in optional_dtypes:
        for st in chosen_studies:
            sub = df[df["study_name"] == st]
            row = pick_one_per_dtype(sub, dt)
            if row is None:
                continue
            add_gb = float(row["gb"])
            if (used + add_gb) <= budget_gb:
                chosen_rows.append(row)
                used += add_gb

    if not chosen_rows:
        return pd.DataFrame(), used, chosen_studies

    chosen = pd.DataFrame(chosen_rows).drop_duplicates(subset=["ehid"]).copy()
    return chosen, used, chosen_studies

# -------------------------
# 1) Identify CRC-relevant studies from sampleMetadata (robust)
# -------------------------
md = load_sample_metadata()

required = {"study_name", "study_condition"}
missing = required - set(md.columns)
if missing:
    raise ValueError(f"sampleMetadata missing required cols: {missing}")

cond = md["study_condition"].astype(str)

md_crc_strict = md[cond.isin(CRC_RELEVANT_CONDITIONS)].copy()

cond_l = cond.str.lower()
mask_kw = False
for kw in CRC_KEYWORDS:
    mask_kw = mask_kw | cond_l.str.contains(re.escape(kw), na=False)
md_crc_kw = md[mask_kw].copy()

md_crc = pd.concat([md_crc_strict, md_crc_kw], axis=0).drop_duplicates()

# Restrict to your relevant cohorts (intersection)
crc_studies = sorted(set(md_crc["study_name"].dropna().unique().tolist()) & set(RELEVANT_STUDIES))

print(f"Relevant CRC cohorts requested: {len(RELEVANT_STUDIES)} -> found in metadata: {len(crc_studies)}")
print("Will use studies:", crc_studies)

# -------------------------
# 2) Scrape EHIDs and select resources matching DATA_TYPES
# -------------------------
selected = []
errors = []

for study in tqdm(crc_studies, desc="Scraping study title pages"):
    try:
        time.sleep(SLEEP_BETWEEN_REQUESTS)
        ehids = get_ehids_for_study(study)

        for ehid in ehids:
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            try:
                url, title = resolve_ehid_to_url_and_title(ehid)
                matched_dt = normalize_dt_match(title, DATA_TYPES)
                if matched_dt is not None:
                    selected.append({
                        "study_name": study,
                        "ehid": ehid,
                        "data_type": matched_dt,
                        "title": title,
                        "url": url
                    })
            except Exception as e:
                errors.append({"study_name": study, "ehid": ehid, "error": str(e)})

    except Exception as e:
        errors.append({"study_name": study, "ehid": None, "error": str(e)})

sel_df = pd.DataFrame(selected)
err_df = pd.DataFrame(errors)

if len(sel_df):
    sel_df = sel_df.sort_values(["study_name", "data_type", "ehid"])

print(f"Selected resources (pre-budget): {len(sel_df)}")
print(f"Errors while scanning: {len(err_df)}")

sel_df.to_csv(os.path.join(OUT_DIR, "selected_resources_prebudget.csv"), index=False)
err_df.to_csv(os.path.join(OUT_DIR, "scan_errors.csv"), index=False)

display(sel_df.head(30))

if len(sel_df) == 0:
    raise RuntimeError(
        "No resources matched your DATA_TYPES. "
        "This can happen because ExperimentHub title pages can be incomplete/paginated for some studies. "
        "If this occurs, switch to the sqlite-based metadata listing approach."
    )

# -------------------------
# 3) Preflight sizes + apply 80GB budget using core-first strategy
# -------------------------
sel_df = ensure_sizes(sel_df)

print("Estimated GB across selected (these studies only):", float(sel_df["gb"].sum()))
print("Unknown-size files:", int(sel_df["unknown_size"].sum()))
display(sel_df.groupby("data_type", as_index=False)["gb"].sum().sort_values("gb", ascending=False))
display(sel_df.sort_values("gb", ascending=False).head(20)[["study_name","data_type","ehid","gb","url"]])

budget_df, used_gb, kept_studies = budget_select(
    sel_df,
    crc_studies,
    BUDGET_GB,
    CORE_DTYPES,
    OPTIONAL_DTYPES,
    STUDY_PRIORITY
)

print("---- BUDGET SELECTION ----")
print("Budget GB:", BUDGET_GB)
print("Used GB (estimated; unknown sizes count as 0):", used_gb)
print("Studies kept (core matrices):", len(kept_studies))
print(kept_studies)

budget_df.to_csv(os.path.join(OUT_DIR, "selected_resources_budget80_crc.csv"), index=False)
print("Saved:", os.path.join(OUT_DIR, "selected_resources_budget80_crc.csv"))

display(budget_df.groupby("data_type", as_index=False)["gb"].sum().sort_values("gb", ascending=False))
display(budget_df.groupby("study_name", as_index=False)["gb"].sum().sort_values("gb", ascending=False))
display(budget_df.head(30))

if len(budget_df) == 0:
    raise RuntimeError("Budget selection produced no files. Increase budget or relax required CORE_DTYPES.")

# -------------------------
# 4) Download budgeted resources
# -------------------------
downloads = []
for _, row in tqdm(budget_df.iterrows(), total=len(budget_df), desc="Downloading EH resources (budgeted CRC)"):
    ehid = row["ehid"]
    study = row["study_name"]
    dtype = row["data_type"]
    url = row["url"]

    ext = guess_ext_from_url(url)
    out_name = safe_filename(f"{study}__{dtype}__{ehid}{ext}")
    out_path = os.path.join(OUT_DIR, out_name)

    try:
        download_file(url, out_path)
        downloads.append({**row.to_dict(), "local_path": out_path})
    except Exception as e:
        downloads.append({**row.to_dict(), "local_path": None, "download_error": str(e)})

dl_df = pd.DataFrame(downloads)
dl_df.to_csv(os.path.join(OUT_DIR, "download_manifest.csv"), index=False)

print("Saved:", os.path.join(OUT_DIR, "download_manifest.csv"))
print("Download failures:", int(dl_df["local_path"].isna().sum()))
display(dl_df.head(20))

# -------------------------
# 5) Build the concrete dataset table (counts + metadata completeness) from sampleMetadata
# -------------------------
def completeness(series: pd.Series) -> float:
    s = series.replace(["NA", "NaN", "nan", ""], np.nan)
    return float(100.0 * s.notna().mean())

covariate_candidates = ["age", "Age", "host_age", "age_years", "sex", "Sex", "gender", "BMI", "bmi", "host_bmi"]
covariate_cols = [c for c in covariate_candidates if c in md.columns]

rows = []
for study in sorted(md["study_name"].dropna().unique()):
    sub = md[md["study_name"] == study].copy()
    counts = sub["study_condition"].astype(str).value_counts().to_dict()

    rec = {
        "study_name": study,
        "n_total": int(len(sub)),
        "n_crc": int(counts.get("CRC", 0) + counts.get("crc", 0)),
        "n_control": int(counts.get("control", 0) + counts.get("Control", 0) + counts.get("healthy", 0)),
        "n_adenoma": int(counts.get("adenoma", 0) + counts.get("Adenoma", 0)),
    }

    for col in covariate_cols:
        rec[f"pct_complete_{col}"] = completeness(sub[col].astype(str))

    rows.append(rec)

study_table = pd.DataFrame(rows).sort_values(["n_crc", "n_total"], ascending=False)
study_table.to_csv(os.path.join(OUT_DIR, "study_table_from_sampleMetadata.csv"), index=False)

print("Saved:", os.path.join(OUT_DIR, "study_table_from_sampleMetadata.csv"))
display(study_table.head(30))

print("\nDONE. Output dir:", OUT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 776.2/776.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.4 MB/s eta 0:00:00


sampleMetadata.rda:   0%|          | 0.00/610k [00:00<?, ?B/s]

Relevant CRC cohorts requested: 11 -> found in metadata: 11
Will use studies: ['FengQ_2015', 'GuptaA_2019', 'HanniganGD_2017', 'ThomasAM_2018a', 'ThomasAM_2018b', 'ThomasAM_2019_c', 'VogtmannE_2016', 'WirbelJ_2018', 'YachidaS_2019', 'YuJ_2015', 'ZellerG_2014']


Scraping study title pages:   0%|          | 0/11 [00:00<?, ?it/s]

Selected resources (pre-budget): 110
Errors while scanning: 0


...

/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = series.replace(["NA", "NaN", "nan", ""], np.nan)
/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = series.replace(["NA", "NaN", "nan", ""], np.nan)
/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set

Saved: /content/curatedMetagenomicData_EH/study_table_from_sampleMetadata.csv


/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = series.replace(["NA", "NaN", "nan", ""], np.nan)
/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s = series.replace(["NA", "NaN", "nan", ""], np.nan)
/tmp/ipykernel_7809/1089833977.py:447: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set

,study_name,n_total,n_crc,n_control,n_adenoma,pct_complete_age,pct_complete_gender,pct_complete_BMI
85,YachidaS_2019,616,258,251,67,100.000000,100.000000,99.837662
89,YuJ_2015,128,74,54,0,100.000000,100.000000,99.218750
83,WirbelJ_2018,125,60,65,0,100.000000,100.000000,100.000000
91,ZellerG_2014,156,53,61,42,100.000000,100.000000,96.794872
80,VogtmannE_2016,110,52,52,0,94.545455,94.545455,91.818182
15,FengQ_2015,154,46,61,47,100.000000,100.000000,100.000000
76,ThomasAM_2019_c,80,40,40,0,100.000000,100.000000,100.000000
75,ThomasAM_2018b,60,32,28,0,98.333333,100.000000,96.666667
20,GuptaA_2019,60,30,30,0,100.000000,100.000000,100.000000
74,ThomasAM_2018a,80,29,24,27,100.000000,100.000000,95.000000



DONE. Output dir: /content/curatedMetagenomicData_EH


##2. Main final CRC benchmark pipeline

In [ ]:
# Outputs per group in: /content/curatedMetagenomicData_EH/analysis_outputs_crc
# Figures in: .../analysis_outputs_crc/figures
# ============================================================

!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn tqdm pyreadr rdata rds2py

import os, re, glob, time, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import pyreadr
from scipy.stats import linregress
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

import matplotlib.pyplot as plt
import seaborn as sns

# rds2py for sparse matrices in .rds
from rds2py import read_rds, parse_rds
from scipy import sparse as sp

warnings.filterwarnings("ignore")
sns.set_context("paper")

# -------------------------
# CONFIG
# -------------------------
DATA_DIR = "/content/curatedMetagenomicData_EH"
OUT_DIR = os.path.join(DATA_DIR, "analysis_outputs_crc")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 7

# Penalized logistic regression
REG_TYPE = "l1"      # "l1" or "elasticnet"
L1_RATIO = 0.5       # used only if REG_TYPE == "elasticnet"
MAX_ITER = 2500
TOL = 1e-3

# Inner CV
N_INNER_CV = 3
C_GRID = np.logspace(-2, 2, 7)

# Labels: CRC vs control only (adenoma excluded)
POS_LABELS = ["crc", "colorectal", "colon", "rectal"]
NEG_LABELS = ["control", "healthy", "normal"]

# Case-control cohort filter
MIN_POS = 20
MIN_NEG = 20

# Coefficient stability thresholds
COEF_EPS = 1e-8
STABLE_FREQ_NONZERO_MIN = 0.66
STABLE_SIGN_CONSISTENCY_MIN = 0.90

# Analysis groups -> data types (matches your downloaded file naming)
ANALYSIS_GROUPS = {
    "species": ["relative_abundance"],
    "pathways": ["pathway_abundance", "pathway_coverage"],
    "markers_abundance": ["marker_abundance"],
    "markers_presence": ["marker_presence"],
    "gene_families": ["gene_families"],
}

# Group-specific preprocessing
GROUP_PREP = {
    "species": {
        "transform": "clr",
        "pseudocount": 1e-6,
        "use_scaler": False,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 2500,
    },
    "pathways": {
        "transform": "log1p",
        "pseudocount": 1e-6,   # not used for log1p
        "use_scaler": True,
        "min_prev": 0.02,
        "min_var": 1e-12,
        "max_features": 6000,
    },
    "markers_abundance": {
        "transform": "log1p",
        "pseudocount": 1e-6,
        "use_scaler": True,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 12000,
    },
    "markers_presence": {
        "transform": "none",     # already 0/1
        "pseudocount": 1e-6,
        "use_scaler": False,
        "min_prev": 0.001,
        "min_var": 1e-12,
        "max_features": 20000,
    },
    "gene_families": {
        "transform": "log1p",
        "pseudocount": 1e-6,
        "use_scaler": True,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 15000,
    }
}

# IMPORTANT safeguard: cap per-study features BEFORE union (prevents union explosion)
PREUNION_MAX_FEATURES = {
    "species": None,
    "pathways": 15000,
    "markers_abundance": 20000,
    "markers_presence": 30000,
    "gene_families": 30000,     # key for feasibility
}

# Output filenames (auto-generated)
def build_fn(group: str):
    return {
        "loso_metrics": f"loso_{group}_metrics_next.csv",
        "loso_preds": f"loso_{group}_predictions_next.csv",
        "pairwise": f"train_test_auc_{group}.csv",
        "shift_loso": f"shift_vs_auc_loso_{group}.csv",
        "shift_pairwise": f"shift_vs_auc_pairwise_{group}.csv",
        "stability": f"{group}_stability_summary.csv",
        "signature": f"{group}_signature_stable.csv",
        "coefs_long": f"loso_{group}_coefs_by_heldout.csv",
        "coefs_global_long": f"{group}_coefs_globalunion_by_heldout_long.csv",
        "load_errors": f"load_errors_{group}.csv",
    }

FN = {g: build_fn(g) for g in ANALYSIS_GROUPS.keys()}

# -------------------------
# HELPERS
# -------------------------
def clip_probs(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return np.clip(np.asarray(p, dtype=float), eps, 1 - eps)

def logit(p: np.ndarray) -> np.ndarray:
    p = clip_probs(p)
    return np.log(p / (1 - p))

def has_both_classes(y: np.ndarray) -> bool:
    y = np.asarray(y)
    u = np.unique(y[~pd.isna(y)])
    return len(u) == 2

def clr_transform(X: np.ndarray, pseudocount: float = 1e-6) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0) + np.float32(pseudocount)
    L = np.log(X).astype(np.float32)
    return (L - L.mean(axis=1, keepdims=True)).astype(np.float32)

def apply_transform(X: np.ndarray, mode: str, pseudocount: float) -> np.ndarray:
    if mode == "clr":
        return clr_transform(X, pseudocount=pseudocount)
    if mode == "log1p":
        X = np.asarray(X, dtype=np.float32)
        return np.log1p(np.maximum(X, 0.0)).astype(np.float32)
    if mode == "none":
        return np.asarray(X, dtype=np.float32)
    raise ValueError(f"Unknown transform: {mode}")

def sens_at_spec(y_true, y_prob, spec_target=0.90):
    if not has_both_classes(y_true):
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    ok = np.where(spec >= spec_target)[0]
    if len(ok) == 0:
        return np.nan
    idx = ok[np.argmax(tpr[ok])]
    return float(tpr[idx])

def spec_at_sens(y_true, y_prob, sens_target=0.90):
    if not has_both_classes(y_true):
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    sens = tpr
    ok = np.where(sens >= sens_target)[0]
    if len(ok) == 0:
        return np.nan
    spec = 1 - fpr
    idx = ok[np.argmax(spec[ok])]
    return float(spec[idx])

def calibration_intercept_slope(y_true, y_prob):
    """
    Robust ridge-logistic calibration fit: y ~ intercept + slope * logit(p).
    """
    if not has_both_classes(y_true):
        return (np.nan, np.nan)
    z = logit(y_prob).reshape(-1, 1)
    y = np.asarray(y_true).astype(int)
    try:
        lr = LogisticRegression(
            penalty="l2", C=10.0, solver="lbfgs",
            max_iter=2000, random_state=RANDOM_STATE
        )
        lr.fit(z, y)
        return (float(lr.intercept_[0]), float(lr.coef_[0, 0]))
    except Exception:
        return (np.nan, np.nan)

def choose_solver(n_features: int) -> str:
    if REG_TYPE == "l1" and n_features <= 20000:
        return "liblinear"
    return "saga"

def build_model(C: float, use_scaler: bool, n_features: int):
    solver = choose_solver(n_features)

    if REG_TYPE == "l1":
        clf = LogisticRegression(
            penalty="l1",
            solver=solver,
            C=C,
            max_iter=MAX_ITER,
            tol=TOL,
            random_state=RANDOM_STATE,
        )
    elif REG_TYPE == "elasticnet":
        clf = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=L1_RATIO,
            C=C,
            max_iter=MAX_ITER,
            tol=TOL,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError("REG_TYPE must be 'l1' or 'elasticnet'")

    if use_scaler:
        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("clf", clf),
        ])
    else:
        pipe = Pipeline([
            ("clf", clf),
        ])
    return pipe

def tune_C_nested_cv(X: np.ndarray, y: np.ndarray, use_scaler: bool, C_grid=C_GRID, n_splits=N_INNER_CV):
    y = np.asarray(y).astype(int)
    class_counts = np.bincount(y)
    if len(class_counts) < 2:
        return 1.0, pd.DataFrame([{"C": 1.0, "mean_cv_auc": np.nan, "n_folds_used": 0}])

    min_class = int(class_counts.min())
    splits = min(n_splits, max(2, min_class))
    if splits < 2:
        return 1.0, pd.DataFrame([{"C": 1.0, "mean_cv_auc": np.nan, "n_folds_used": 0}])

    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)

    n_features = int(X.shape[1])
    rows = []
    for C in C_grid:
        aucs = []
        for tr, va in skf.split(X, y):
            model = build_model(float(C), use_scaler=use_scaler, n_features=n_features)
            model.fit(X[tr], y[tr])
            p = clip_probs(model.predict_proba(X[va])[:, 1])
            if has_both_classes(y[va]):
                aucs.append(roc_auc_score(y[va], p))
        rows.append({
            "C": float(C),
            "mean_cv_auc": float(np.mean(aucs)) if len(aucs) else np.nan,
            "n_folds_used": int(len(aucs))
        })
    cv_df = pd.DataFrame(rows).sort_values("mean_cv_auc", ascending=False)
    best_C = float(cv_df.iloc[0]["C"])
    return best_C, cv_df

def mean_shift_distance(X_train: np.ndarray, X_test: np.ndarray) -> float:
    mu_tr = np.mean(X_train, axis=0)
    mu_te = np.mean(X_test, axis=0)
    return float(np.linalg.norm(mu_tr - mu_te))

# -------------------------
# METADATA + LABELS
# -------------------------
def load_sample_metadata(data_dir=DATA_DIR) -> pd.DataFrame:
    rda_path = os.path.join(data_dir, "sampleMetadata.rda")
    if not os.path.exists(rda_path):
        raise FileNotFoundError(f"sampleMetadata.rda not found at: {rda_path}")
    objs = pyreadr.read_r(rda_path)
    if "sampleMetadata" not in objs:
        raise ValueError(f"sampleMetadata not found. Keys: {list(objs.keys())}")
    return objs["sampleMetadata"].copy()

def pick_col(md: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in md.columns:
            return c
    return None

def get_sample_ids(md: pd.DataFrame) -> pd.Series:
    for c in ["sampleID", "sample_id", "sample", "Sample", "sample_name", "run_accession", "SRR", "SRS"]:
        if c in md.columns:
            s = md[c].astype(str)
            if s.nunique() > 100:
                return s
    return md.index.astype(str)

def map_binary_label(cond_str: str):
    s = str(cond_str).strip().lower()
    if "adenoma" in s:
        return None
    if any(kw in s for kw in POS_LABELS):
        return 1
    if any(kw in s for kw in NEG_LABELS):
        return 0
    return None

# -------------------------
# MATRIX DISCOVERY
# -------------------------
FILE_RE = re.compile(
    r"^(?P<study>.+?)__(?P<dtype>[^_]+(?:_[^_]+)*)__(?:(?P<date>\d{4}-\d{2}-\d{2})__)?(?P<ehid>EH\d+)\.(?P<ext>rds|rda)$"
)

def discover_latest_file_per_study(dtype: str, data_dir=DATA_DIR):
    paths = glob.glob(os.path.join(data_dir, f"*__{dtype}__*.rd*"))
    best = {}
    for p in paths:
        name = os.path.basename(p)
        m = FILE_RE.match(name)
        if not m:
            continue
        study = m.group("study")
        date = m.group("date")
        dt = pd.to_datetime(date) if date else pd.NaT
        key = (dt, name)
        if study not in best:
            best[study] = (key, p)
        else:
            prev_key, _ = best[study]
            if pd.isna(prev_key[0]) and not pd.isna(key[0]):
                best[study] = (key, p)
            elif (not pd.isna(prev_key[0]) and not pd.isna(key[0]) and key[0] > prev_key[0]):
                best[study] = (key, p)
            elif (pd.isna(prev_key[0]) and pd.isna(key[0]) and key[1] > prev_key[1]):
                best[study] = (key, p)
    return {k: v[1] for k, v in best.items()}

# -------------------------
# ROBUST RDS LOADING (pyreadr -> rds2py fallback)
# -------------------------
def _find_dimnames_recursive(o):
    if isinstance(o, dict):
        for k, v in o.items():
            if "dimname" in str(k).lower():
                return v
        for v in o.values():
            out = _find_dimnames_recursive(v)
            if out is not None:
                return out
    elif isinstance(o, (list, tuple)):
        for v in o:
            out = _find_dimnames_recursive(v)
            if out is not None:
                return out
    return None

def _extract_str_vector(x):
    if x is None:
        return None
    if isinstance(x, (pd.Index, pd.Series, np.ndarray)):
        return [str(i) for i in list(x)]
    if isinstance(x, str):
        return [x]
    if isinstance(x, (list, tuple)):
        out = []
        for e in x:
            ev = _extract_str_vector(e)
            if ev is None:
                continue
            out.extend(ev if isinstance(ev, list) else [str(ev)])
        return [str(i) for i in out] if len(out) else None
    if isinstance(x, dict):
        for k in ["data", "value", "values", "elements", "payload", "strings"]:
            if k in x:
                return _extract_str_vector(x[k])
        return _extract_str_vector(list(x.values()))
    return None

def _make_unique(names: list[str]) -> list[str]:
    seen = {}
    out = []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}__dup{seen[n]}")
    return out

def read_r_any(path: str):
    """
    Returns (obj, rownames, colnames) where obj can be:
      - pd.DataFrame
      - scipy sparse matrix
      - np.ndarray
    """
    # 1) try pyreadr (best for data.frame-like)
    try:
        robj = pyreadr.read_r(path)
        for k in robj.keys():
            v = robj[k]
            if isinstance(v, pd.DataFrame):
                df = v.copy()
                df.index = df.index.astype(str)
                df.columns = df.columns.astype(str)
                return df, df.index.astype(str).tolist(), df.columns.astype(str).tolist()
    except Exception:
        pass

    # 2) fallback to rds2py
    robj = read_rds(path)

    # sometimes rds2py already gives DataFrame
    if isinstance(robj, pd.DataFrame):
        df = robj.copy()
        df.index = df.index.astype(str)
        df.columns = df.columns.astype(str)
        return df, df.index.astype(str).tolist(), df.columns.astype(str).tolist()

    rownames = None
    colnames = None

    # try attribute access (if exposed)
    if rownames is None and hasattr(robj, "rownames"):
        rownames = _extract_str_vector(getattr(robj, "rownames"))
    if colnames is None and hasattr(robj, "colnames"):
        colnames = _extract_str_vector(getattr(robj, "colnames"))

    # parse_rds dimnames (robust)
    try:
        d = parse_rds(path)
        dn = _find_dimnames_recursive(d)
        if isinstance(dn, (list, tuple)) and len(dn) >= 2:
            rownames = rownames or _extract_str_vector(dn[0])
            colnames = colnames or _extract_str_vector(dn[1])
    except Exception:
        pass

    return robj, rownames, colnames

def orient_matrix_any(X, rownames, colnames, sample_id_set: set[str]):
    if rownames is None or colnames is None:
        return X, rownames, colnames

    idx = set(map(str, rownames))
    cols = set(map(str, colnames))
    idx_hit = len(idx & sample_id_set)
    col_hit = len(cols & sample_id_set)

    # if columns look like sample IDs, transpose
    if col_hit > idx_hit:
        if sp.issparse(X):
            X = X.T
        else:
            X = np.asarray(X).T
        rownames, colnames = colnames, rownames

    return X, rownames, colnames

def preunion_trim_matrix(X, colnames: list[str], cap: int):
    """
    Keep top-variance columns up to cap.
    Works for sparse and dense matrices.
    """
    if cap is None:
        return X, colnames
    p = int(X.shape[1])
    if p <= cap:
        return X, colnames

    n = int(X.shape[0])

    if sp.issparse(X):
        Xc = X.tocsc(copy=False)
        col_sum = np.asarray(Xc.sum(axis=0)).ravel()
        col_sq_sum = np.asarray(Xc.multiply(Xc).sum(axis=0)).ravel()
        mean = col_sum / max(1, n)
        var = (col_sq_sum / max(1, n)) - (mean ** 2)
    else:
        Xd = np.asarray(X, dtype=np.float64)
        mean = Xd.mean(axis=0)
        var = (Xd ** 2).mean(axis=0) - mean ** 2

    idx = np.argsort(var)[::-1][:cap]
    idx = np.sort(idx)
    if sp.issparse(X):
        X2 = X[:, idx]
    else:
        X2 = np.asarray(X)[:, idx]
    col2 = [colnames[i] for i in idx]
    return X2, col2

def coerce_to_sample_feature_df(obj, rownames, colnames, sample_id_set: set[str], preunion_cap: int | None):
    """
    Returns pd.DataFrame of shape (samples x features), with string index/columns.
    Raises if dimnames missing for non-DataFrame objects.
    """
    # DataFrame path
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
        df.index = df.index.astype(str)
        df.columns = df.columns.astype(str)
        # orient by sample id hits (may transpose)
        idx_hit = len(set(df.index) & sample_id_set)
        col_hit = len(set(df.columns) & sample_id_set)
        if col_hit > idx_hit and col_hit > 0:
            df = df.T
            df.index = df.index.astype(str)
            df.columns = df.columns.astype(str)
        # preunion trim
        if preunion_cap is not None and df.shape[1] > preunion_cap:
            var = df.apply(pd.to_numeric, errors="coerce").fillna(0.0).var(axis=0)
            top = var.sort_values(ascending=False).index[:preunion_cap]
            df = df.loc[:, top]
        return df

    # Sparse or ndarray path
    if rownames is None or colnames is None:
        raise ValueError("Missing dimnames (samples/features) for non-DataFrame object.")

    X = obj
    if not sp.issparse(X):
        X = np.asarray(X)

    X, rownames, colnames = orient_matrix_any(X, rownames, colnames, sample_id_set)

    rownames = [str(x) for x in rownames]
    colnames = [str(x) for x in colnames]

    if len(rownames) != int(X.shape[0]) or len(colnames) != int(X.shape[1]):
        raise ValueError(f"Dimnames length mismatch: rows={len(rownames)} vs {X.shape[0]}, cols={len(colnames)} vs {X.shape[1]}")

    rownames = _make_unique(rownames)
    colnames = _make_unique(colnames)

    # preunion trim BEFORE densifying
    X, colnames = preunion_trim_matrix(X, colnames, preunion_cap)

    # densify after trimming (keeps pipeline simple)
    if sp.issparse(X):
        Xd = X.toarray().astype(np.float32)
    else:
        Xd = np.asarray(X, dtype=np.float32)

    df = pd.DataFrame(Xd, index=np.asarray(rownames, dtype=str), columns=np.asarray(colnames, dtype=str))
    return df

# -------------------------
# GROUP MATRIX LOADING (multi-dtype with sample alignment)
# -------------------------
def load_study_group_matrix(
    study: str,
    group: str,
    dtypes: list[str],
    dtype_to_file: dict[str, dict[str, str]],
    sample_id_set: set[str],
):
    preunion_cap = PREUNION_MAX_FEATURES.get(group, None)

    mats = []
    common_samples = None

    for dt in dtypes:
        fmap = dtype_to_file.get(dt, {})
        if study not in fmap:
            continue

        path = fmap[study]
        obj, rn, cn = read_r_any(path)
        df = coerce_to_sample_feature_df(obj, rn, cn, sample_id_set, preunion_cap=preunion_cap)

        df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)

        # Prefix features to avoid collisions
        df.columns = [f"{dt}::{c}" for c in df.columns.astype(str)]
        mats.append(df)

        if common_samples is None:
            common_samples = set(df.index.astype(str))
        else:
            common_samples &= set(df.index.astype(str))

    if not mats:
        return None

    # keep intersection of samples across dtypes (preferred)
    if common_samples and len(common_samples) > 0:
        keep = sorted(list(common_samples))
    else:
        keep = sorted(list(set().union(*[set(m.index.astype(str)) for m in mats])))

    mats = [m.reindex(index=keep, fill_value=0.0) for m in mats]
    X = pd.concat(mats, axis=1)

    # Drop all-zero features early
    nz = (X.sum(axis=0) != 0)
    X = X.loc[:, nz]
    return X

# -------------------------
# FEATURE FILTERING
# -------------------------
def filter_features_df(X_df: pd.DataFrame, min_prev: float, min_var: float, max_features: int | None):
    prev = (X_df > 0).mean(axis=0)
    var = X_df.var(axis=0)

    keep = (prev >= min_prev) & (var >= min_var)
    if int(keep.sum()) == 0:
        keep = var > 0

    X_f = X_df.loc[:, keep]
    if (max_features is not None) and (X_f.shape[1] > max_features):
        var_f = X_f.var(axis=0).sort_values(ascending=False)
        top = var_f.index[:max_features]
        X_f = X_f.loc[:, top]
    return X_f

# -------------------------
# ALIGNMENT + STACKING
# -------------------------
def align_stack_filtered(
    study_dict: dict,
    studies: list[str],
    min_prev: float,
    min_var: float,
    max_features: int | None,
    transform: str,
    pseudocount: float,
):
    feats = set()
    for st in studies:
        feats |= set(study_dict[st]["X_raw"].columns)
    union = sorted(list(feats))

    X_list, y_list, study_labels, sample_ids = [], [], [], []
    for st in studies:
        Xs = study_dict[st]["X_raw"].reindex(columns=union, fill_value=0.0)
        ys = study_dict[st]["y"]
        sids = study_dict[st]["samples"]
        X_list.append(Xs)
        y_list.append(ys)
        study_labels.extend([st] * Xs.shape[0])
        sample_ids.extend(list(sids))

    X_df = pd.concat(X_list, axis=0)
    y = np.concatenate(y_list, axis=0).astype(int)
    study_labels = np.asarray(study_labels, dtype=str)
    sample_ids = np.asarray(sample_ids, dtype=str)

    # filter using training data only
    X_df = filter_features_df(X_df, min_prev=min_prev, min_var=min_var, max_features=max_features)
    feat_list = X_df.columns.astype(str).tolist()

    X = X_df.to_numpy(dtype=np.float32, copy=False)
    X = apply_transform(X, mode=transform, pseudocount=pseudocount)

    return X, y, study_labels, sample_ids, feat_list

def align_single_filtered(study_dict: dict, study: str, feat_list: list[str], transform: str, pseudocount: float):
    X_df = study_dict[study]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
    X = X_df.to_numpy(dtype=np.float32, copy=False)
    X = apply_transform(X, mode=transform, pseudocount=pseudocount)
    y = study_dict[study]["y"].astype(int)
    sids = study_dict[study]["samples"]
    return X, y, sids

def filter_case_control_studies(study_dict: dict, min_pos=MIN_POS, min_neg=MIN_NEG):
    eligible = []
    for st, info in study_dict.items():
        if info["pos"] >= min_pos and info["neg"] >= min_neg:
            eligible.append(st)
    return sorted(eligible)

# -------------------------
# LOSO
# -------------------------
def run_loso(group: str, study_dict: dict, studies: list[str]):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    use_scaler = prep["use_scaler"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    metrics_rows, pred_rows, coef_rows = [], [], []
    fold_coefs = []

    global_union = sorted(list(set().union(*[set(study_dict[s]["X_raw"].columns) for s in studies])))
    global_index = {f: i for i, f in enumerate(global_union)}

    for heldout in tqdm(studies, desc=f"LOSO ({group})"):
        t0 = time.time()
        train_studies = [s for s in studies if s != heldout]
        if len(train_studies) == 0:
            continue

        X_train, y_train, _, _, feat_list = align_stack_filtered(
            study_dict, train_studies,
            min_prev=min_prev, min_var=min_var, max_features=max_features,
            transform=transform, pseudocount=pseudocount
        )
        X_test, y_test, test_sample_ids = align_single_filtered(
            study_dict, heldout, feat_list,
            transform=transform, pseudocount=pseudocount
        )

        n_feat = int(X_train.shape[1])
        n_test = int(len(y_test))

        if not has_both_classes(y_test):
            metrics_rows.append({
                "heldout": heldout, "n_test": n_test, "n_feat": n_feat,
                "roc_auc": np.nan, "pr_auc": np.nan,
                "sens_at_90spec": np.nan, "spec_at_90sens": np.nan,
                "brier": np.nan, "cal_intercept": np.nan, "cal_slope": np.nan,
                "lambda": np.nan, "C_best": np.nan, "note": "skipped_one_class_test"
            })
            continue

        best_C, _cv = tune_C_nested_cv(X_train, y_train, use_scaler=use_scaler, C_grid=C_GRID, n_splits=N_INNER_CV)
        model = build_model(best_C, use_scaler=use_scaler, n_features=n_feat)
        model.fit(X_train, y_train)

        p_hat = clip_probs(model.predict_proba(X_test)[:, 1])

        roc_auc = float(roc_auc_score(y_test, p_hat))
        pr_auc = float(average_precision_score(y_test, p_hat))
        s90 = sens_at_spec(y_test, p_hat, spec_target=0.90)
        sp90 = spec_at_sens(y_test, p_hat, sens_target=0.90)
        brier = float(brier_score_loss(y_test, p_hat))
        cal_i, cal_s = calibration_intercept_slope(y_test, p_hat)

        metrics_rows.append({
            "heldout": heldout,
            "n_test": n_test,
            "n_feat": n_feat,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "sens_at_90spec": s90,
            "spec_at_90sens": sp90,
            "brier": brier,
            "cal_intercept": cal_i,
            "cal_slope": cal_s,
            "lambda": float(1.0 / best_C),
            "C_best": float(best_C),
            "note": ""
        })

        for sid, yy, pp in zip(test_sample_ids, y_test, p_hat):
            pred_rows.append({"heldout": heldout, "sample_id": str(sid), "y_true": int(yy), "p_hat": float(pp)})

        clf = model.named_steps["clf"] if "clf" in model.named_steps else model
        coefs = clf.coef_.reshape(-1)
        for f, c in zip(feat_list, coefs):
            if abs(float(c)) > COEF_EPS:
                coef_rows.append({"heldout": heldout, "feature": f, "coef": float(c)})

        vec = np.zeros(len(global_union), dtype=np.float32)
        for f, c in zip(feat_list, coefs):
            j = global_index.get(f, None)
            if j is not None:
                vec[j] = np.float32(c)
        fold_coefs.append((heldout, vec))

        dt = time.time() - t0
        print(f"[{group}] heldout={heldout} n_feat={n_feat} done in {dt/60:.1f} min")

    metrics_df = pd.DataFrame(metrics_rows).sort_values("roc_auc", ascending=False)
    preds_df = pd.DataFrame(pred_rows)
    coefs_long_df = pd.DataFrame(coef_rows)
    return metrics_df, preds_df, coefs_long_df, fold_coefs, global_union

# -------------------------
# PAIRWISE TRAIN->TEST
# -------------------------
def run_pairwise_transfer(group: str, study_dict: dict, studies: list[str]):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    use_scaler = prep["use_scaler"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    rows = []
    for train_study in tqdm(studies, desc=f"Pairwise train->test ({group})"):
        y_tr = study_dict[train_study]["y"]
        if not has_both_classes(y_tr):
            continue

        X_tr_df = study_dict[train_study]["X_raw"]
        X_tr_df = filter_features_df(X_tr_df, min_prev=min_prev, min_var=min_var, max_features=max_features)
        feat_list = X_tr_df.columns.astype(str).tolist()

        X_train = apply_transform(X_tr_df.to_numpy(dtype=np.float32, copy=False), mode=transform, pseudocount=pseudocount)
        y_train = y_tr.astype(int)

        n_feat = int(X_train.shape[1])
        best_C, _ = tune_C_nested_cv(X_train, y_train, use_scaler=use_scaler, C_grid=C_GRID, n_splits=N_INNER_CV)
        model = build_model(best_C, use_scaler=use_scaler, n_features=n_feat)
        model.fit(X_train, y_train)

        for test_study in studies:
            X_te_df = study_dict[test_study]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
            X_test = apply_transform(X_te_df.to_numpy(dtype=np.float32, copy=False), mode=transform, pseudocount=pseudocount)
            y_test = study_dict[test_study]["y"].astype(int)

            if not has_both_classes(y_test):
                rows.append({"train": train_study, "test": test_study, "roc_auc": np.nan})
                continue

            p = clip_probs(model.predict_proba(X_test)[:, 1])
            rows.append({"train": train_study, "test": test_study, "roc_auc": float(roc_auc_score(y_test, p))})

    return pd.DataFrame(rows)

# -------------------------
# DOMAIN SHIFT
# -------------------------
def run_shift_vs_auc_loso(group: str, study_dict: dict, studies: list[str], loso_metrics: pd.DataFrame):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    rows = []
    for heldout in studies:
        train_studies = [s for s in studies if s != heldout]
        if len(train_studies) == 0:
            continue

        X_train, y_train, _, _, feat_list = align_stack_filtered(
            study_dict, train_studies,
            min_prev=min_prev, min_var=min_var, max_features=max_features,
            transform=transform, pseudocount=pseudocount
        )
        X_test, y_test, _ = align_single_filtered(
            study_dict, heldout, feat_list,
            transform=transform, pseudocount=pseudocount
        )

        dist = mean_shift_distance(X_train, X_test)
        auc_row = loso_metrics[loso_metrics["heldout"] == heldout]
        roc_auc = float(auc_row["roc_auc"].values[0]) if len(auc_row) else np.nan

        rows.append({
            "heldout": heldout,
            "shift_distance": dist,
            "roc_auc": roc_auc,
            "n_train": int(X_train.shape[0]),
            "n_test": int(X_test.shape[0]),
            "n_feat": int(X_train.shape[1]),
        })
    return pd.DataFrame(rows)

def run_shift_vs_auc_pairwise(group: str, study_dict: dict, pairwise_df: pd.DataFrame):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    rows = []
    for _, r in pairwise_df.iterrows():
        tr = r["train"]
        te = r["test"]

        X_tr_df = study_dict[tr]["X_raw"]
        X_tr_df = filter_features_df(X_tr_df, min_prev=min_prev, min_var=min_var, max_features=max_features)
        feat_list = X_tr_df.columns.astype(str).tolist()

        X_train = apply_transform(X_tr_df.to_numpy(dtype=np.float32, copy=False), mode=transform, pseudocount=pseudocount)

        X_te_df = study_dict[te]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
        X_test = apply_transform(X_te_df.to_numpy(dtype=np.float32, copy=False), mode=transform, pseudocount=pseudocount)

        dist = mean_shift_distance(X_train, X_test)
        rows.append({"train": tr, "test": te, "shift_distance": dist, "roc_auc": r["roc_auc"]})
    return pd.DataFrame(rows)

# -------------------------
# STABILITY
# -------------------------
def stability_summary(fold_coefs, global_union):
    C = np.vstack([v for _, v in fold_coefs]).astype(np.float32)

    nonzero = (np.abs(C) > COEF_EPS).astype(np.float32)
    freq_nonzero = nonzero.mean(axis=0)

    mean_c = C.mean(axis=0)
    sd_c = C.std(axis=0)

    mean_sign = np.sign(mean_c)
    sign_match = (np.sign(C) == mean_sign).astype(np.float32)
    sign_match[:, mean_sign == 0] = 0.0
    sign_consistency = sign_match.mean(axis=0)

    stable = (
        (freq_nonzero >= STABLE_FREQ_NONZERO_MIN) &
        (sign_consistency >= STABLE_SIGN_CONSISTENCY_MIN) &
        (np.abs(mean_c) > COEF_EPS)
    )

    df = pd.DataFrame({
        "feature": global_union,
        "freq_nonzero": freq_nonzero,
        "mean_coef": mean_c,
        "sd_coef": sd_c,
        "sign_consistency": sign_consistency,
        "stable": stable
    }).sort_values(
        ["stable", "freq_nonzero", "sign_consistency", "mean_coef"],
        ascending=[False, False, False, False]
    )
    return df

def coefs_by_heldout_long(fold_coefs, global_union):
    rows = []
    for heldout, vec in fold_coefs:
        nz = np.where(np.abs(vec) > COEF_EPS)[0]
        for j in nz:
            rows.append({"heldout": heldout, "feature": global_union[j], "coef": float(vec[j])})
    return pd.DataFrame(rows)

# -------------------------
# PLOTTING
# -------------------------
def save_fig(path_base: str):
    plt.tight_layout()
    plt.savefig(path_base + ".png", dpi=300)
    plt.savefig(path_base + ".pdf")
    plt.close()

def plot_loso_auc(metrics_df: pd.DataFrame, title: str, outname: str):
    df = metrics_df.copy().sort_values("roc_auc", ascending=True)
    plt.figure(figsize=(8, max(4, 0.28 * max(1, len(df)))))
    plt.scatter(df["roc_auc"], df["heldout"])
    plt.axvline(0.5, linestyle="--", linewidth=1)
    plt.xlabel("ROC-AUC")
    plt.ylabel("Held-out cohort")
    plt.title(title)
    save_fig(os.path.join(FIG_DIR, outname))

def plot_heatmap_auc(pairwise_df: pd.DataFrame, title: str, outname: str):
    if len(pairwise_df) == 0:
        return
    mat = pairwise_df.pivot_table(index="train", columns="test", values="roc_auc", aggfunc="mean")
    plt.figure(figsize=(10.5, 8.5))
    sns.heatmap(mat, cmap="viridis", vmin=0.4, vmax=1.0, linewidths=0.2)
    plt.title(title)
    plt.xlabel("Test cohort")
    plt.ylabel("Train cohort")
    save_fig(os.path.join(FIG_DIR, outname))

def plot_shift_scatter(df: pd.DataFrame, title: str, outname: str):
    d = df.dropna(subset=["shift_distance", "roc_auc"]).copy()
    if len(d) == 0:
        return
    plt.figure(figsize=(6.8, 5.2))
    sns.regplot(data=d, x="shift_distance", y="roc_auc", scatter_kws={"s": 35}, line_kws={"linewidth": 2})
    plt.title(title)
    plt.xlabel("Mean-shift distance (Euclidean)")
    plt.ylabel("ROC-AUC")
    if len(d) >= 3:
        lr = linregress(d["shift_distance"].values, d["roc_auc"].values)
        txt = f"slope={lr.slope:.3f}, r={lr.rvalue:.3f}, p={lr.pvalue:.2g}"
        plt.gca().text(0.02, 0.02, txt, transform=plt.gca().transAxes, fontsize=10, va="bottom")
    save_fig(os.path.join(FIG_DIR, outname))

# -------------------------
# MAIN
# -------------------------
print("Loading sampleMetadata...")
md = load_sample_metadata(DATA_DIR)

study_col = "study_name" if "study_name" in md.columns else None
if study_col is None:
    raise ValueError("sampleMetadata missing required column: study_name")

cond_col = pick_col(md, ["study_condition", "condition", "disease", "diagnosis"])
if cond_col is None:
    raise ValueError("No condition column found in sampleMetadata. Tried: study_condition/condition/disease/diagnosis")

md = md.copy()
md["_sample_id"] = get_sample_ids(md).astype(str)
md["_study"] = md[study_col].astype(str)
md["_cond_raw"] = md[cond_col].astype(str)
md["_y"] = md["_cond_raw"].apply(map_binary_label)

md_bin = md.dropna(subset=["_y"]).copy()
md_bin["_y"] = md_bin["_y"].astype(int)

print(f"Total samples in sampleMetadata: {len(md)}")
print(f"Binary-labeled (CRC/control) samples: {len(md_bin)}")
print(f"Studies with binary-labeled samples: {md_bin['_study'].nunique()}")

sample_id_set = set(md["_sample_id"].astype(str)) | set(md.index.astype(str))

# Discover files for all dtypes used
dtype_to_file = {}
all_dtypes = sorted(set(dt for dts in ANALYSIS_GROUPS.values() for dt in dts))
for dt in all_dtypes:
    dtype_to_file[dt] = discover_latest_file_per_study(dt, DATA_DIR)
    print(f"Discovered files for dtype={dt}: {len(dtype_to_file[dt])}")

# Build per-group study dictionaries (only downloaded studies intersecting metadata)
group_study_data = {}
group_studies = {}

for group, dtypes in ANALYSIS_GROUPS.items():
    data = {}
    load_errors = []

    # limit to studies we actually have files for (big speedup)
    candidate = set()
    for dt in dtypes:
        candidate |= set(dtype_to_file.get(dt, {}).keys())
    studies_in_md = sorted(list(candidate & set(md_bin["_study"].unique().tolist())))

    for study in tqdm(studies_in_md, desc=f"Loading matrices for group={group}"):
        try:
            X = load_study_group_matrix(study, group, dtypes, dtype_to_file, sample_id_set)
            if X is None:
                continue

            md_s = md_bin[md_bin["_study"] == study].copy()
            md_s = md_s.set_index("_sample_id", drop=False)

            common = sorted(list(set(X.index.astype(str)) & set(md_s.index.astype(str))))
            if len(common) == 0:
                load_errors.append({"study": study, "error": "no_common_samples_between_matrix_and_metadata"})
                continue

            X = X.loc[common].copy()
            y = md_s.loc[common, "_y"].astype(int).values

            data[study] = {
                "X_raw": X,
                "y": y,
                "samples": np.array(common, dtype=str),
                "n": int(len(common)),
                "pos": int(np.sum(y == 1)),
                "neg": int(np.sum(y == 0)),
            }
        except Exception as e:
            load_errors.append({"study": study, "error": str(e)})

    group_study_data[group] = data
    eligible = filter_case_control_studies(data, min_pos=MIN_POS, min_neg=MIN_NEG)
    group_studies[group] = eligible

    print(f"\nGroup={group}: loaded {len(data)} studies with matrix+labels.")
    print(f"Group={group}: using {len(eligible)} CRC case-control cohorts (min_pos={MIN_POS}, min_neg={MIN_NEG}).")

    # Always write load errors if any
    if len(load_errors):
        errp = os.path.join(OUT_DIR, FN[group]["load_errors"])
        pd.DataFrame(load_errors).to_csv(errp, index=False)
        print("Load errors saved:", errp)

    # Safe display table (no crash on empty)
    if len(eligible) == 0:
        tmp = pd.DataFrame(columns=["study", "n", "pos", "neg"])
        display(tmp)
        continue

    tmp = pd.DataFrame(
        [{"study": s, "n": data[s]["n"], "pos": data[s]["pos"], "neg": data[s]["neg"]} for s in eligible]
    ).sort_values(["pos", "n"], ascending=False)
    display(tmp)

# Run analyses for groups with enough cohorts
run_groups = [g for g in ANALYSIS_GROUPS.keys() if len(group_studies.get(g, [])) >= 3]
skip_groups = [g for g in ANALYSIS_GROUPS.keys() if g not in run_groups]

if len(skip_groups):
    print("\nSkipping groups with <3 eligible cohorts:", skip_groups)

for group in run_groups:
    print("\n====================")
    print(f"RUNNING GROUP: {group}")
    print(f"Data types: {ANALYSIS_GROUPS[group]}")
    print(f"Cohorts: {len(group_studies[group])}")
    print("Prep:", GROUP_PREP[group])
    print("====================")

    study_dict = group_study_data[group]
    studies = group_studies[group]

    # LOSO
    loso_metrics, loso_preds, loso_coefs_long, fold_coefs, global_union = run_loso(group, study_dict, studies)
    loso_metrics.to_csv(os.path.join(OUT_DIR, FN[group]["loso_metrics"]), index=False)
    loso_preds.to_csv(os.path.join(OUT_DIR, FN[group]["loso_preds"]), index=False)
    loso_coefs_long.to_csv(os.path.join(OUT_DIR, FN[group]["coefs_long"]), index=False)

    print("Saved:", os.path.join(OUT_DIR, FN[group]["loso_metrics"]))
    print("Saved:", os.path.join(OUT_DIR, FN[group]["loso_preds"]))
    print("Saved:", os.path.join(OUT_DIR, FN[group]["coefs_long"]))
    print("LOSO summary (top 10 by ROC-AUC):")
    display(loso_metrics.head(10))

    # Pairwise
    pairwise_df = run_pairwise_transfer(group, study_dict, studies)
    pairwise_df.to_csv(os.path.join(OUT_DIR, FN[group]["pairwise"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["pairwise"]))
    display(pairwise_df.head(12))

    # Domain shift
    shift_loso_df = run_shift_vs_auc_loso(group, study_dict, studies, loso_metrics)
    shift_loso_df.to_csv(os.path.join(OUT_DIR, FN[group]["shift_loso"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["shift_loso"]))
    display(shift_loso_df.head(12))

    shift_pair_df = run_shift_vs_auc_pairwise(group, study_dict, pairwise_df)
    shift_pair_df.to_csv(os.path.join(OUT_DIR, FN[group]["shift_pairwise"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["shift_pairwise"]))
    display(shift_pair_df.head(12))

    # Stability + signature
    stab_df = stability_summary(fold_coefs, global_union)
    stab_df.to_csv(os.path.join(OUT_DIR, FN[group]["stability"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["stability"]))
    display(stab_df.head(25))

    sig_df = stab_df[stab_df["stable"]].copy()
    sig_df.to_csv(os.path.join(OUT_DIR, FN[group]["signature"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["signature"]))
    print(f"Stable signature features: {len(sig_df)}")

    coefs_global_long = coefs_by_heldout_long(fold_coefs, global_union)
    coefs_global_long.to_csv(os.path.join(OUT_DIR, FN[group]["coefs_global_long"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["coefs_global_long"]))

    # Figures
    plot_loso_auc(loso_metrics, title=f"{group} LOSO ROC-AUC (robust)", outname=f"{group}_loso_rocauc")
    plot_heatmap_auc(pairwise_df, title=f"{group} train -> test ROC-AUC", outname=f"{group}_train_test_heatmap")
    plot_shift_scatter(shift_loso_df, title=f"{group}: shift distance vs ROC-AUC", outname=f"{group}_shift_vs_auc")

    print("Figures saved in:", FIG_DIR)

print("\nDone.")
print("Outputs:", OUT_DIR)
print("Figures:", FIG_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.1/188.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 1.6 MB/s eta 0:00:00
Loading sampleMetadata...
Total samples in sampleMetadata: 22588
Binary-labeled (CRC/control) samples: 15822
Studies with binary-labeled samples: 82
Discovered files for dtype=gene_families: 11
Discovered files for dtype=marker_abundance: 11
Discovered files for dtype=marker_presence: 11
Discovered files for dtype=pathway_abundance: 11
Discovered files for dtype=pathway_coverage: 11
Discovered files for dtype=relative_abundance: 11


Loading matrices for group=species:   0%|          | 0/11 [00:00<?, ?it/s]


Group=species: loaded 9 studies with matrix+labels.
Group=species: using 9 CRC case-control cohorts (min_pos=20, min_neg=20).
Load errors saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/load_errors_species.csv


,study,n,pos,neg
7,YachidaS_2019,509,258,251
6,WirbelJ_2018,125,60,65
8,ZellerG_2014,114,53,61
5,VogtmannE_2016,104,52,52
4,ThomasAM_2019_c,80,40,40
3,ThomasAM_2018b,60,32,28
0,GuptaA_2019,60,30,30
2,ThomasAM_2018a,53,29,24
1,HanniganGD_2017,55,27,28


Loading matrices for group=pathways:   0%|          | 0/11 [00:00<?, ?it/s]

...

Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/shift_vs_auc_pairwise_markers_presence.csv


,train,test,shift_distance,roc_auc
0,FengQ_2015,FengQ_2015,0.000000,1.000000
1,FengQ_2015,GuptaA_2019,46.467232,0.694444
2,FengQ_2015,HanniganGD_2017,40.887630,0.588624
3,FengQ_2015,ThomasAM_2018a,32.555046,0.682471
4,FengQ_2015,ThomasAM_2018b,21.051935,0.603795
5,FengQ_2015,ThomasAM_2019_c,27.892809,0.756875
6,FengQ_2015,VogtmannE_2016,22.771206,0.629808
7,FengQ_2015,WirbelJ_2018,20.520498,0.703077
8,FengQ_2015,YachidaS_2019,28.270460,0.643102
9,FengQ_2015,YuJ_2015,24.101719,0.740490


Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_stability_summary.csv


,feature,freq_nonzero,mean_coef,sd_coef,sign_consistency,stable
36315,marker_presence::384639__GCA_003096855.1_01599,1.0,0.647675,0.252787,1.0,True
32104,marker_presence::29391__GCA_900476045.1_01564,1.0,0.543771,0.143804,1.0,True
1290,marker_presence::1134687__A0A377SEG4__A225_2996,1.0,0.520965,0.176946,1.0,True
47749,marker_presence::712288__G6C5I4__HMPREF9093_01831,1.0,0.464370,0.289324,1.0,True
34691,marker_presence::341694__E0E128__HMPREF0634_0779,1.0,0.434599,0.118904,1.0,True
26334,marker_presence::209880__A0A1G5WHZ1__SAMN02910...,1.0,0.395262,0.119706,1.0,True
33420,marker_presence::33033__A0A0B4S0E7__NW74_01920,1.0,0.384214,0.131892,1.0,True
2713,marker_presence::1261__A0A379CEJ7__prsA1,1.0,0.363554,0.156842,1.0,True
48242,marker_presence::713008__F9PTA1__HMPREF9127_1220,1.0,0.350204,0.150979,1.0,True
34690,marker_presence::341694__E0E125__HMPREF0634_0776,1.0,0.349623,0.121525,1.0,True


Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_signature_stable.csv
Stable signature features: 321
Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_coefs_globalunion_by_heldout_long.csv
Figures saved in: /content/curatedMetagenomicData_EH/analysis_outputs_crc/figures

Done.
Outputs: /content/curatedMetagenomicData_EH/analysis_outputs_crc
Figures: /content/curatedMetagenomicData_EH/analysis_outputs_crc/figures


# COMBINATION TRANSFER


In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn tqdm pyreadr rdata rds2py

import os, re, glob, time, warnings
from itertools import combinations
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import pyreadr
from scipy.stats import linregress
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve

import matplotlib.pyplot as plt
import seaborn as sns

from rds2py import read_rds, parse_rds
from scipy import sparse as sp

warnings.filterwarnings("ignore")
sns.set_context("paper")

# -------------------------
# CONFIG
# -------------------------
DATA_DIR = "/content/curatedMetagenomicData_EH"
OUT_DIR = os.path.join(DATA_DIR, "analysis_outputs_crc")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 7

REG_TYPE = "l1"
L1_RATIO = 0.5
MAX_ITER = 2500
TOL = 1e-3

N_INNER_CV = 3
C_GRID = np.logspace(-2, 2, 7)

POS_LABELS = ["crc", "colorectal", "colon", "rectal"]
NEG_LABELS = ["control", "healthy", "normal"]

MIN_POS = 20
MIN_NEG = 20

COEF_EPS = 1e-8
STABLE_FREQ_NONZERO_MIN = 0.66
STABLE_SIGN_CONSISTENCY_MIN = 0.90

RUN_PUBLICATION_TRANSFER = True
RUN_PUB_SHIFT = True

# Disjoint only for publication-oriented transfer
ALLOW_TRAIN_TEST_OVERLAP = False

# Recommended family set for paper
# You can remove ("2->2") if still too slow
PUBLICATION_FAMILIES = [
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (3, 1),
    (2, 2),
]

# Optional emergency cap
MAX_PUBLICATION_EVALS = None

ANALYSIS_GROUPS = {
    "species": ["relative_abundance"],
    "pathways": ["pathway_abundance", "pathway_coverage"],
    "markers_abundance": ["marker_abundance"],
    "markers_presence": ["marker_presence"],
    "gene_families": ["gene_families"],
}

GROUP_PREP = {
    "species": {
        "transform": "clr",
        "pseudocount": 1e-6,
        "use_scaler": False,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 2500,
    },
    "pathways": {
        "transform": "log1p",
        "pseudocount": 1e-6,
        "use_scaler": True,
        "min_prev": 0.02,
        "min_var": 1e-12,
        "max_features": 6000,
    },
    "markers_abundance": {
        "transform": "log1p",
        "pseudocount": 1e-6,
        "use_scaler": True,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 12000,
    },
    "markers_presence": {
        "transform": "none",
        "pseudocount": 1e-6,
        "use_scaler": False,
        "min_prev": 0.001,
        "min_var": 1e-12,
        "max_features": 20000,
    },
    "gene_families": {
        "transform": "log1p",
        "pseudocount": 1e-6,
        "use_scaler": True,
        "min_prev": 0.01,
        "min_var": 1e-12,
        "max_features": 15000,
    }
}

PREUNION_MAX_FEATURES = {
    "species": None,
    "pathways": 15000,
    "markers_abundance": 20000,
    "markers_presence": 30000,
    "gene_families": 30000,
}

def build_fn(group: str):
    return {
        "loso_metrics": f"loso_{group}_metrics_next.csv",
        "loso_preds": f"loso_{group}_predictions_next.csv",
        "pub_transfer": f"publication_transfer_auc_{group}.csv",
        "pub_summary": f"publication_transfer_summary_{group}.csv",
        "shift_loso": f"shift_vs_auc_loso_{group}.csv",
        "shift_pub": f"shift_vs_auc_publication_{group}.csv",
        "stability": f"{group}_stability_summary.csv",
        "signature": f"{group}_signature_stable.csv",
        "coefs_long": f"loso_{group}_coefs_by_heldout.csv",
        "coefs_global_long": f"{group}_coefs_globalunion_by_heldout_long.csv",
        "load_errors": f"load_errors_{group}.csv",
    }

FN = {g: build_fn(g) for g in ANALYSIS_GROUPS.keys()}

# -------------------------
# HELPERS
# -------------------------
def clip_probs(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return np.clip(np.asarray(p, dtype=float), eps, 1 - eps)

def logit(p: np.ndarray) -> np.ndarray:
    p = clip_probs(p)
    return np.log(p / (1 - p))

def has_both_classes(y: np.ndarray) -> bool:
    y = np.asarray(y)
    u = np.unique(y[~pd.isna(y)])
    return len(u) == 2

def clr_transform(X: np.ndarray, pseudocount: float = 1e-6) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)
    X = np.maximum(X, 0.0) + np.float32(pseudocount)
    L = np.log(X).astype(np.float32)
    return (L - L.mean(axis=1, keepdims=True)).astype(np.float32)

def apply_transform(X: np.ndarray, mode: str, pseudocount: float) -> np.ndarray:
    if mode == "clr":
        return clr_transform(X, pseudocount=pseudocount)
    if mode == "log1p":
        X = np.asarray(X, dtype=np.float32)
        return np.log1p(np.maximum(X, 0.0)).astype(np.float32)
    if mode == "none":
        return np.asarray(X, dtype=np.float32)
    raise ValueError(f"Unknown transform: {mode}")

def sens_at_spec(y_true, y_prob, spec_target=0.90):
    if not has_both_classes(y_true):
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    ok = np.where(spec >= spec_target)[0]
    if len(ok) == 0:
        return np.nan
    idx = ok[np.argmax(tpr[ok])]
    return float(tpr[idx])

def spec_at_sens(y_true, y_prob, sens_target=0.90):
    if not has_both_classes(y_true):
        return np.nan
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    sens = tpr
    ok = np.where(sens >= sens_target)[0]
    if len(ok) == 0:
        return np.nan
    spec = 1 - fpr
    idx = ok[np.argmax(spec[ok])]
    return float(spec[idx])

def calibration_intercept_slope(y_true, y_prob):
    if not has_both_classes(y_true):
        return (np.nan, np.nan)
    z = logit(y_prob).reshape(-1, 1)
    y = np.asarray(y_true).astype(int)
    try:
        lr = LogisticRegression(
            penalty="l2", C=10.0, solver="lbfgs",
            max_iter=2000, random_state=RANDOM_STATE
        )
        lr.fit(z, y)
        return (float(lr.intercept_[0]), float(lr.coef_[0, 0]))
    except Exception:
        return (np.nan, np.nan)

def choose_solver(n_features: int) -> str:
    if REG_TYPE == "l1" and n_features <= 20000:
        return "liblinear"
    return "saga"

def build_model(C: float, use_scaler: bool, n_features: int):
    solver = choose_solver(n_features)

    if REG_TYPE == "l1":
        clf = LogisticRegression(
            penalty="l1",
            solver=solver,
            C=C,
            max_iter=MAX_ITER,
            tol=TOL,
            random_state=RANDOM_STATE,
        )
    elif REG_TYPE == "elasticnet":
        clf = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=L1_RATIO,
            C=C,
            max_iter=MAX_ITER,
            tol=TOL,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError("REG_TYPE must be 'l1' or 'elasticnet'")

    if use_scaler:
        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("clf", clf),
        ])
    else:
        pipe = Pipeline([("clf", clf)])
    return pipe

def tune_C_nested_cv(X: np.ndarray, y: np.ndarray, use_scaler: bool, C_grid=C_GRID, n_splits=N_INNER_CV):
    y = np.asarray(y).astype(int)
    class_counts = np.bincount(y)
    if len(class_counts) < 2:
        return 1.0, pd.DataFrame([{"C": 1.0, "mean_cv_auc": np.nan, "n_folds_used": 0}])

    min_class = int(class_counts.min())
    splits = min(n_splits, max(2, min_class))
    if splits < 2:
        return 1.0, pd.DataFrame([{"C": 1.0, "mean_cv_auc": np.nan, "n_folds_used": 0}])

    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)

    n_features = int(X.shape[1])
    rows = []
    for C in C_grid:
        aucs = []
        for tr, va in skf.split(X, y):
            model = build_model(float(C), use_scaler=use_scaler, n_features=n_features)
            model.fit(X[tr], y[tr])
            p = clip_probs(model.predict_proba(X[va])[:, 1])
            if has_both_classes(y[va]):
                aucs.append(roc_auc_score(y[va], p))
        rows.append({
            "C": float(C),
            "mean_cv_auc": float(np.mean(aucs)) if len(aucs) else np.nan,
            "n_folds_used": int(len(aucs))
        })
    cv_df = pd.DataFrame(rows).sort_values("mean_cv_auc", ascending=False)
    best_C = float(cv_df.iloc[0]["C"])
    return best_C, cv_df

def mean_shift_distance_from_means(mu_train: np.ndarray, mu_test: np.ndarray) -> float:
    return float(np.linalg.norm(mu_train - mu_test))

# -------------------------
# METADATA + LABELS
# -------------------------
def load_sample_metadata(data_dir=DATA_DIR) -> pd.DataFrame:
    rda_path = os.path.join(data_dir, "sampleMetadata.rda")
    if not os.path.exists(rda_path):
        raise FileNotFoundError(f"sampleMetadata.rda not found at: {rda_path}")
    objs = pyreadr.read_r(rda_path)
    if "sampleMetadata" not in objs:
        raise ValueError(f"sampleMetadata not found. Keys: {list(objs.keys())}")
    return objs["sampleMetadata"].copy()

def pick_col(md: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in md.columns:
            return c
    return None

def get_sample_ids(md: pd.DataFrame) -> pd.Series:
    for c in ["sampleID", "sample_id", "sample", "Sample", "sample_name", "run_accession", "SRR", "SRS"]:
        if c in md.columns:
            s = md[c].astype(str)
            if s.nunique() > 100:
                return s
    return md.index.astype(str)

def map_binary_label(cond_str: str):
    s = str(cond_str).strip().lower()
    if "adenoma" in s:
        return None
    if any(kw in s for kw in POS_LABELS):
        return 1
    if any(kw in s for kw in NEG_LABELS):
        return 0
    return None

# -------------------------
# MATRIX DISCOVERY
# -------------------------
FILE_RE = re.compile(
    r"^(?P<study>.+?)__(?P<dtype>[^_]+(?:_[^_]+)*)__(?:(?P<date>\d{4}-\d{2}-\d{2})__)?(?P<ehid>EH\d+)\.(?P<ext>rds|rda)$"
)

def discover_latest_file_per_study(dtype: str, data_dir=DATA_DIR):
    paths = glob.glob(os.path.join(data_dir, f"*__{dtype}__*.rd*"))
    best = {}
    for p in paths:
        name = os.path.basename(p)
        m = FILE_RE.match(name)
        if not m:
            continue
        study = m.group("study")
        date = m.group("date")
        dt = pd.to_datetime(date) if date else pd.NaT
        key = (dt, name)
        if study not in best:
            best[study] = (key, p)
        else:
            prev_key, _ = best[study]
            if pd.isna(prev_key[0]) and not pd.isna(key[0]):
                best[study] = (key, p)
            elif (not pd.isna(prev_key[0]) and not pd.isna(key[0]) and key[0] > prev_key[0]):
                best[study] = (key, p)
            elif (pd.isna(prev_key[0]) and pd.isna(key[0]) and key[1] > prev_key[1]):
                best[study] = (key, p)
    return {k: v[1] for k, v in best.items()}

# -------------------------
# ROBUST RDS LOADING
# -------------------------
def _find_dimnames_recursive(o):
    if isinstance(o, dict):
        for k, v in o.items():
            if "dimname" in str(k).lower():
                return v
        for v in o.values():
            out = _find_dimnames_recursive(v)
            if out is not None:
                return out
    elif isinstance(o, (list, tuple)):
        for v in o:
            out = _find_dimnames_recursive(v)
            if out is not None:
                return out
    return None

def _extract_str_vector(x):
    if x is None:
        return None
    if isinstance(x, (pd.Index, pd.Series, np.ndarray)):
        return [str(i) for i in list(x)]
    if isinstance(x, str):
        return [x]
    if isinstance(x, (list, tuple)):
        out = []
        for e in x:
            ev = _extract_str_vector(e)
            if ev is None:
                continue
            out.extend(ev if isinstance(ev, list) else [str(ev)])
        return [str(i) for i in out] if len(out) else None
    if isinstance(x, dict):
        for k in ["data", "value", "values", "elements", "payload", "strings"]:
            if k in x:
                return _extract_str_vector(x[k])
        return _extract_str_vector(list(x.values()))
    return None

def _make_unique(names: list[str]) -> list[str]:
    seen = {}
    out = []
    for n in names:
        if n not in seen:
            seen[n] = 0
            out.append(n)
        else:
            seen[n] += 1
            out.append(f"{n}__dup{seen[n]}")
    return out

def read_r_any(path: str):
    try:
        robj = pyreadr.read_r(path)
        for k in robj.keys():
            v = robj[k]
            if isinstance(v, pd.DataFrame):
                df = v.copy()
                df.index = df.index.astype(str)
                df.columns = df.columns.astype(str)
                return df, df.index.astype(str).tolist(), df.columns.astype(str).tolist()
    except Exception:
        pass

    robj = read_rds(path)

    if isinstance(robj, pd.DataFrame):
        df = robj.copy()
        df.index = df.index.astype(str)
        df.columns = df.columns.astype(str)
        return df, df.index.astype(str).tolist(), df.columns.astype(str).tolist()

    rownames = None
    colnames = None

    if rownames is None and hasattr(robj, "rownames"):
        rownames = _extract_str_vector(getattr(robj, "rownames"))
    if colnames is None and hasattr(robj, "colnames"):
        colnames = _extract_str_vector(getattr(robj, "colnames"))

    try:
        d = parse_rds(path)
        dn = _find_dimnames_recursive(d)
        if isinstance(dn, (list, tuple)) and len(dn) >= 2:
            rownames = rownames or _extract_str_vector(dn[0])
            colnames = colnames or _extract_str_vector(dn[1])
    except Exception:
        pass

    return robj, rownames, colnames

def orient_matrix_any(X, rownames, colnames, sample_id_set: set[str]):
    if rownames is None or colnames is None:
        return X, rownames, colnames

    idx = set(map(str, rownames))
    cols = set(map(str, colnames))
    idx_hit = len(idx & sample_id_set)
    col_hit = len(cols & sample_id_set)

    if col_hit > idx_hit:
        if sp.issparse(X):
            X = X.T
        else:
            X = np.asarray(X).T
        rownames, colnames = colnames, rownames

    return X, rownames, colnames

def preunion_trim_matrix(X, colnames: list[str], cap: int):
    if cap is None:
        return X, colnames
    p = int(X.shape[1])
    if p <= cap:
        return X, colnames

    n = int(X.shape[0])

    if sp.issparse(X):
        Xc = X.tocsc(copy=False)
        col_sum = np.asarray(Xc.sum(axis=0)).ravel()
        col_sq_sum = np.asarray(Xc.multiply(Xc).sum(axis=0)).ravel()
        mean = col_sum / max(1, n)
        var = (col_sq_sum / max(1, n)) - (mean ** 2)
    else:
        Xd = np.asarray(X, dtype=np.float64)
        mean = Xd.mean(axis=0)
        var = (Xd ** 2).mean(axis=0) - mean ** 2

    idx = np.argsort(var)[::-1][:cap]
    idx = np.sort(idx)
    if sp.issparse(X):
        X2 = X[:, idx]
    else:
        X2 = np.asarray(X)[:, idx]
    col2 = [colnames[i] for i in idx]
    return X2, col2

def coerce_to_sample_feature_df(obj, rownames, colnames, sample_id_set: set[str], preunion_cap: int | None):
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
        df.index = df.index.astype(str)
        df.columns = df.columns.astype(str)
        idx_hit = len(set(df.index) & sample_id_set)
        col_hit = len(set(df.columns) & sample_id_set)
        if col_hit > idx_hit and col_hit > 0:
            df = df.T
            df.index = df.index.astype(str)
            df.columns = df.columns.astype(str)
        if preunion_cap is not None and df.shape[1] > preunion_cap:
            var = df.apply(pd.to_numeric, errors="coerce").fillna(0.0).var(axis=0)
            top = var.sort_values(ascending=False).index[:preunion_cap]
            df = df.loc[:, top]
        return df

    if rownames is None or colnames is None:
        raise ValueError("Missing dimnames (samples/features) for non-DataFrame object.")

    X = obj
    if not sp.issparse(X):
        X = np.asarray(X)

    X, rownames, colnames = orient_matrix_any(X, rownames, colnames, sample_id_set)

    rownames = [str(x) for x in rownames]
    colnames = [str(x) for x in colnames]

    if len(rownames) != int(X.shape[0]) or len(colnames) != int(X.shape[1]):
        raise ValueError(f"Dimnames length mismatch: rows={len(rownames)} vs {X.shape[0]}, cols={len(colnames)} vs {X.shape[1]}")

    rownames = _make_unique(rownames)
    colnames = _make_unique(colnames)

    X, colnames = preunion_trim_matrix(X, colnames, preunion_cap)

    if sp.issparse(X):
        Xd = X.toarray().astype(np.float32)
    else:
        Xd = np.asarray(X, dtype=np.float32)

    df = pd.DataFrame(Xd, index=np.asarray(rownames, dtype=str), columns=np.asarray(colnames, dtype=str))
    return df

# -------------------------
# GROUP MATRIX LOADING
# -------------------------
def load_study_group_matrix(
    study: str,
    group: str,
    dtypes: list[str],
    dtype_to_file: dict[str, dict[str, str]],
    sample_id_set: set[str],
):
    preunion_cap = PREUNION_MAX_FEATURES.get(group, None)

    mats = []
    common_samples = None

    for dt in dtypes:
        fmap = dtype_to_file.get(dt, {})
        if study not in fmap:
            continue

        path = fmap[study]
        obj, rn, cn = read_r_any(path)
        df = coerce_to_sample_feature_df(obj, rn, cn, sample_id_set, preunion_cap=preunion_cap)
        df = df.apply(pd.to_numeric, errors="coerce").fillna(0.0)
        df.columns = [f"{dt}::{c}" for c in df.columns.astype(str)]
        mats.append(df)

        if common_samples is None:
            common_samples = set(df.index.astype(str))
        else:
            common_samples &= set(df.index.astype(str))

    if not mats:
        return None

    if common_samples and len(common_samples) > 0:
        keep = sorted(list(common_samples))
    else:
        keep = sorted(list(set().union(*[set(m.index.astype(str)) for m in mats])))

    mats = [m.reindex(index=keep, fill_value=0.0) for m in mats]
    X = pd.concat(mats, axis=1)

    nz = (X.sum(axis=0) != 0)
    X = X.loc[:, nz]
    return X

# -------------------------
# FEATURE FILTERING
# -------------------------
def filter_features_df(X_df: pd.DataFrame, min_prev: float, min_var: float, max_features: int | None):
    prev = (X_df > 0).mean(axis=0)
    var = X_df.var(axis=0)

    keep = (prev >= min_prev) & (var >= min_var)
    if int(keep.sum()) == 0:
        keep = var > 0

    X_f = X_df.loc[:, keep]
    if (max_features is not None) and (X_f.shape[1] > max_features):
        var_f = X_f.var(axis=0).sort_values(ascending=False)
        top = var_f.index[:max_features]
        X_f = X_f.loc[:, top]
    return X_f

# -------------------------
# ALIGNMENT + STACKING
# -------------------------
def align_stack_filtered(
    study_dict: dict,
    studies: list[str],
    min_prev: float,
    min_var: float,
    max_features: int | None,
    transform: str,
    pseudocount: float,
):
    feats = set()
    for st in studies:
        feats |= set(study_dict[st]["X_raw"].columns)
    union = sorted(list(feats))

    X_list, y_list, study_labels, sample_ids = [], [], [], []
    for st in studies:
        Xs = study_dict[st]["X_raw"].reindex(columns=union, fill_value=0.0)
        ys = study_dict[st]["y"]
        sids = study_dict[st]["samples"]
        X_list.append(Xs)
        y_list.append(ys)
        study_labels.extend([st] * Xs.shape[0])
        sample_ids.extend(list(sids))

    X_df = pd.concat(X_list, axis=0)
    y = np.concatenate(y_list, axis=0).astype(int)
    study_labels = np.asarray(study_labels, dtype=str)
    sample_ids = np.asarray(sample_ids, dtype=str)

    X_df = filter_features_df(X_df, min_prev=min_prev, min_var=min_var, max_features=max_features)
    feat_list = X_df.columns.astype(str).tolist()

    X = X_df.to_numpy(dtype=np.float32, copy=False)
    X = apply_transform(X, mode=transform, pseudocount=pseudocount)

    return X, y, study_labels, sample_ids, feat_list

def align_single_filtered(study_dict: dict, study: str, feat_list: list[str], transform: str, pseudocount: float):
    X_df = study_dict[study]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
    X = X_df.to_numpy(dtype=np.float32, copy=False)
    X = apply_transform(X, mode=transform, pseudocount=pseudocount)
    y = study_dict[study]["y"].astype(int)
    sids = study_dict[study]["samples"]
    return X, y, sids

def filter_case_control_studies(study_dict: dict, min_pos=MIN_POS, min_neg=MIN_NEG):
    eligible = []
    for st, info in study_dict.items():
        if info["pos"] >= min_pos and info["neg"] >= min_neg:
            eligible.append(st)
    return sorted(eligible)

# -------------------------
# PUBLICATION FAMILY HELPERS
# -------------------------
def combo_to_str(combo):
    return " | ".join(combo)

def family_to_str(a, b):
    return f"{a}->{b}"

def count_family_combinations(n, a, b):
    if a < 1 or b < 1 or a + b > n:
        return 0
    from math import comb
    return comb(n, a) * comb(n - a, b)

def build_publication_family_test_index(studies, families):
    """
    Build:
    - family-specific train combos
    - family-specific valid test combos for each train combo
    """
    family_index = {}
    n = len(studies)

    for a, b in families:
        if a + b > n:
            continue
        train_combos = list(combinations(studies, a))
        per_train_tests = {}
        for tr in train_combos:
            remaining = [s for s in studies if s not in set(tr)]
            per_train_tests[tr] = list(combinations(remaining, b))
        family_index[(a, b)] = {
            "family": family_to_str(a, b),
            "train_combos": train_combos,
            "test_combos_by_train": per_train_tests,
            "n_pairs": count_family_combinations(n, a, b),
        }
    return family_index

# -------------------------
# FAST STUDY CACHE UNDER TRAIN FEATURE SPACE
# -------------------------
def build_study_cache_for_feat_list(study_dict, studies, feat_list, transform, pseudocount):
    cache = {}
    for st in studies:
        X_df = study_dict[st]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
        X = X_df.to_numpy(dtype=np.float32, copy=False)
        X = apply_transform(X, mode=transform, pseudocount=pseudocount)
        y = study_dict[st]["y"].astype(int)
        n = int(X.shape[0])
        mu = np.mean(X, axis=0).astype(np.float32) if n > 0 else np.zeros(len(feat_list), dtype=np.float32)
        cache[st] = {
            "X": X,
            "y": y,
            "n": n,
            "pos": int(np.sum(y == 1)),
            "neg": int(np.sum(y == 0)),
            "mu": mu,
        }
    return cache

def weighted_mean_from_cache(studies_subset, study_cache):
    total_n = sum(study_cache[s]["n"] for s in studies_subset)
    if total_n == 0:
        dim = len(next(iter(study_cache.values()))["mu"])
        return np.zeros(dim, dtype=np.float32), 0
    mu = np.zeros_like(next(iter(study_cache.values()))["mu"], dtype=np.float64)
    for s in studies_subset:
        mu += study_cache[s]["mu"].astype(np.float64) * study_cache[s]["n"]
    mu /= total_n
    return mu.astype(np.float32), total_n

def concat_arrays_for_subset(studies_subset, per_study_values, key):
    return np.concatenate([per_study_values[s][key] for s in studies_subset], axis=0)

# -------------------------
# LOSO
# -------------------------
def run_loso(group: str, study_dict: dict, studies: list[str]):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    use_scaler = prep["use_scaler"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    metrics_rows, pred_rows, coef_rows = [], [], []
    fold_coefs = []

    global_union = sorted(list(set().union(*[set(study_dict[s]["X_raw"].columns) for s in studies])))
    global_index = {f: i for i, f in enumerate(global_union)}

    for heldout in tqdm(studies, desc=f"LOSO ({group})"):
        train_studies = [s for s in studies if s != heldout]
        if len(train_studies) == 0:
            continue

        X_train, y_train, _, _, feat_list = align_stack_filtered(
            study_dict, train_studies,
            min_prev=min_prev, min_var=min_var, max_features=max_features,
            transform=transform, pseudocount=pseudocount
        )
        X_test, y_test, test_sample_ids = align_single_filtered(
            study_dict, heldout, feat_list,
            transform=transform, pseudocount=pseudocount
        )

        n_feat = int(X_train.shape[1])
        n_test = int(len(y_test))

        if not has_both_classes(y_test):
            metrics_rows.append({
                "heldout": heldout, "n_test": n_test, "n_feat": n_feat,
                "roc_auc": np.nan, "pr_auc": np.nan,
                "sens_at_90spec": np.nan, "spec_at_90sens": np.nan,
                "brier": np.nan, "cal_intercept": np.nan, "cal_slope": np.nan,
                "lambda": np.nan, "C_best": np.nan, "note": "skipped_one_class_test"
            })
            continue

        best_C, _ = tune_C_nested_cv(X_train, y_train, use_scaler=use_scaler, C_grid=C_GRID, n_splits=N_INNER_CV)
        model = build_model(best_C, use_scaler=use_scaler, n_features=n_feat)
        model.fit(X_train, y_train)

        p_hat = clip_probs(model.predict_proba(X_test)[:, 1])

        roc_auc = float(roc_auc_score(y_test, p_hat))
        pr_auc = float(average_precision_score(y_test, p_hat))
        s90 = sens_at_spec(y_test, p_hat, spec_target=0.90)
        sp90 = spec_at_sens(y_test, p_hat, sens_target=0.90)
        brier = float(brier_score_loss(y_test, p_hat))
        cal_i, cal_s = calibration_intercept_slope(y_test, p_hat)

        metrics_rows.append({
            "heldout": heldout,
            "n_test": n_test,
            "n_feat": n_feat,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "sens_at_90spec": s90,
            "spec_at_90sens": sp90,
            "brier": brier,
            "cal_intercept": cal_i,
            "cal_slope": cal_s,
            "lambda": float(1.0 / best_C),
            "C_best": float(best_C),
            "note": ""
        })

        for sid, yy, pp in zip(test_sample_ids, y_test, p_hat):
            pred_rows.append({"heldout": heldout, "sample_id": str(sid), "y_true": int(yy), "p_hat": float(pp)})

        clf = model.named_steps["clf"] if "clf" in model.named_steps else model
        coefs = clf.coef_.reshape(-1)
        for f, c in zip(feat_list, coefs):
            if abs(float(c)) > COEF_EPS:
                coef_rows.append({"heldout": heldout, "feature": f, "coef": float(c)})

        vec = np.zeros(len(global_union), dtype=np.float32)
        for f, c in zip(feat_list, coefs):
            j = global_index.get(f, None)
            if j is not None:
                vec[j] = np.float32(c)
        fold_coefs.append((heldout, vec))

    metrics_df = pd.DataFrame(metrics_rows).sort_values("roc_auc", ascending=False)
    preds_df = pd.DataFrame(pred_rows)
    coefs_long_df = pd.DataFrame(coef_rows)
    return metrics_df, preds_df, coefs_long_df, fold_coefs, global_union

# -------------------------
# PUBLICATION-ORIENTED TRANSFER
# -------------------------
def run_publication_transfer(group: str, study_dict: dict, studies: list[str], families=PUBLICATION_FAMILIES):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    use_scaler = prep["use_scaler"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    n = len(studies)
    fam_index = build_publication_family_test_index(studies, families)

    print(f"\n[{group}] Publication-oriented family counts")
    total_expected = 0
    for (a, b), info in fam_index.items():
        print(f"  {info['family']}: {info['n_pairs']:,} combinations")
        total_expected += info["n_pairs"]
    print(f"  TOTAL: {total_expected:,} combinations")

    rows = []
    shift_rows = []
    eval_counter = 0

    for (a, b), info in fam_index.items():
        fam_label = info["family"]
        train_combos = info["train_combos"]
        test_combos_by_train = info["test_combos_by_train"]

        for train_combo in tqdm(train_combos, desc=f"{group} family {fam_label}"):
            train_combo = tuple(train_combo)
            train_combo_str = combo_to_str(train_combo)

            X_train, y_train, _, _, feat_list = align_stack_filtered(
                study_dict, list(train_combo),
                min_prev=min_prev, min_var=min_var, max_features=max_features,
                transform=transform, pseudocount=pseudocount
            )

            if not has_both_classes(y_train):
                continue

            n_feat = int(X_train.shape[1])
            train_mu = np.mean(X_train, axis=0).astype(np.float32)

            best_C, _ = tune_C_nested_cv(X_train, y_train, use_scaler=use_scaler, C_grid=C_GRID, n_splits=N_INNER_CV)
            model = build_model(best_C, use_scaler=use_scaler, n_features=n_feat)
            model.fit(X_train, y_train)

            study_cache = build_study_cache_for_feat_list(
                study_dict=study_dict,
                studies=studies,
                feat_list=feat_list,
                transform=transform,
                pseudocount=pseudocount
            )

            per_study_pred = {}
            for st in studies:
                Xs = study_cache[st]["X"]
                ys = study_cache[st]["y"]
                ps = clip_probs(model.predict_proba(Xs)[:, 1]).astype(np.float32)
                per_study_pred[st] = {"y": ys, "p": ps}

            for test_combo in test_combos_by_train[train_combo]:
                test_combo = tuple(test_combo)
                test_combo_str = combo_to_str(test_combo)

                y_test = concat_arrays_for_subset(test_combo, per_study_pred, "y")
                p_hat = concat_arrays_for_subset(test_combo, per_study_pred, "p")

                row = {
                    "family": fam_label,
                    "train_combo": train_combo_str,
                    "test_combo": test_combo_str,
                    "n_train_studies": a,
                    "n_test_studies": b,
                    "train_studies": "||".join(train_combo),
                    "test_studies": "||".join(test_combo),
                    "n_train_samples": int(X_train.shape[0]),
                    "n_test_samples": int(len(y_test)),
                    "n_feat": n_feat,
                    "train_pos": int(np.sum(y_train == 1)),
                    "train_neg": int(np.sum(y_train == 0)),
                    "test_pos": int(np.sum(y_test == 1)),
                    "test_neg": int(np.sum(y_test == 0)),
                    "lambda": float(1.0 / best_C),
                    "C_best": float(best_C),
                }

                if not has_both_classes(y_test):
                    row.update({
                        "roc_auc": np.nan,
                        "pr_auc": np.nan,
                        "sens_at_90spec": np.nan,
                        "spec_at_90sens": np.nan,
                        "brier": np.nan,
                        "cal_intercept": np.nan,
                        "cal_slope": np.nan,
                        "note": "skipped_one_class_test"
                    })
                else:
                    cal_i, cal_s = calibration_intercept_slope(y_test, p_hat)
                    row.update({
                        "roc_auc": float(roc_auc_score(y_test, p_hat)),
                        "pr_auc": float(average_precision_score(y_test, p_hat)),
                        "sens_at_90spec": sens_at_spec(y_test, p_hat, spec_target=0.90),
                        "spec_at_90sens": spec_at_sens(y_test, p_hat, sens_target=0.90),
                        "brier": float(brier_score_loss(y_test, p_hat)),
                        "cal_intercept": cal_i,
                        "cal_slope": cal_s,
                        "note": ""
                    })

                rows.append(row)

                if RUN_PUB_SHIFT:
                    test_mu, test_n = weighted_mean_from_cache(test_combo, study_cache)
                    shift_rows.append({
                        "family": fam_label,
                        "train_combo": train_combo_str,
                        "test_combo": test_combo_str,
                        "n_train_studies": a,
                        "n_test_studies": b,
                        "n_train_samples": int(X_train.shape[0]),
                        "n_test_samples": int(test_n),
                        "n_feat": n_feat,
                        "shift_distance": mean_shift_distance_from_means(train_mu, test_mu),
                        "roc_auc": row["roc_auc"],
                    })

                eval_counter += 1
                if (MAX_PUBLICATION_EVALS is not None) and (eval_counter >= MAX_PUBLICATION_EVALS):
                    print(f"[{group}] reached MAX_PUBLICATION_EVALS={MAX_PUBLICATION_EVALS:,}; stopping early.")
                    return pd.DataFrame(rows), pd.DataFrame(shift_rows)

    return pd.DataFrame(rows), pd.DataFrame(shift_rows)

# -------------------------
# LOSO SHIFT
# -------------------------
def run_shift_vs_auc_loso(group: str, study_dict: dict, studies: list[str], loso_metrics: pd.DataFrame):
    prep = GROUP_PREP[group]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    rows = []
    for heldout in studies:
        train_studies = [s for s in studies if s != heldout]
        if len(train_studies) == 0:
            continue

        X_train, y_train, _, _, feat_list = align_stack_filtered(
            study_dict, train_studies,
            min_prev=min_prev, min_var=min_var, max_features=max_features,
            transform=transform, pseudocount=pseudocount
        )
        X_test, y_test, _ = align_single_filtered(
            study_dict, heldout, feat_list,
            transform=transform, pseudocount=pseudocount
        )

        dist = float(np.linalg.norm(np.mean(X_train, axis=0) - np.mean(X_test, axis=0)))
        auc_row = loso_metrics[loso_metrics["heldout"] == heldout]
        roc_auc = float(auc_row["roc_auc"].values[0]) if len(auc_row) else np.nan

        rows.append({
            "heldout": heldout,
            "shift_distance": dist,
            "roc_auc": roc_auc,
            "n_train": int(X_train.shape[0]),
            "n_test": int(X_test.shape[0]),
            "n_feat": int(X_train.shape[1]),
        })
    return pd.DataFrame(rows)

# -------------------------
# STABILITY
# -------------------------
def stability_summary(fold_coefs, global_union):
    C = np.vstack([v for _, v in fold_coefs]).astype(np.float32)

    nonzero = (np.abs(C) > COEF_EPS).astype(np.float32)
    freq_nonzero = nonzero.mean(axis=0)

    mean_c = C.mean(axis=0)
    sd_c = C.std(axis=0)

    mean_sign = np.sign(mean_c)
    sign_match = (np.sign(C) == mean_sign).astype(np.float32)
    sign_match[:, mean_sign == 0] = 0.0
    sign_consistency = sign_match.mean(axis=0)

    stable = (
        (freq_nonzero >= STABLE_FREQ_NONZERO_MIN) &
        (sign_consistency >= STABLE_SIGN_CONSISTENCY_MIN) &
        (np.abs(mean_c) > COEF_EPS)
    )

    df = pd.DataFrame({
        "feature": global_union,
        "freq_nonzero": freq_nonzero,
        "mean_coef": mean_c,
        "sd_coef": sd_c,
        "sign_consistency": sign_consistency,
        "stable": stable
    }).sort_values(
        ["stable", "freq_nonzero", "sign_consistency", "mean_coef"],
        ascending=[False, False, False, False]
    )
    return df

def coefs_by_heldout_long(fold_coefs, global_union):
    rows = []
    for heldout, vec in fold_coefs:
        nz = np.where(np.abs(vec) > COEF_EPS)[0]
        for j in nz:
            rows.append({"heldout": heldout, "feature": global_union[j], "coef": float(vec[j])})
    return pd.DataFrame(rows)

# -------------------------
# PLOTTING
# -------------------------
def save_fig(path_base: str):
    plt.tight_layout()
    plt.savefig(path_base + ".png", dpi=300)
    plt.savefig(path_base + ".pdf")
    plt.close()

def plot_loso_auc(metrics_df: pd.DataFrame, title: str, outname: str):
    df = metrics_df.copy().sort_values("roc_auc", ascending=True)
    plt.figure(figsize=(8, max(4, 0.28 * max(1, len(df)))))
    plt.scatter(df["roc_auc"], df["heldout"])
    plt.axvline(0.5, linestyle="--", linewidth=1)
    plt.xlabel("ROC-AUC")
    plt.ylabel("Held-out cohort")
    plt.title(title)
    save_fig(os.path.join(FIG_DIR, outname))

def plot_shift_scatter(df: pd.DataFrame, title: str, outname: str):
    d = df.dropna(subset=["shift_distance", "roc_auc"]).copy()
    if len(d) == 0:
        return
    plt.figure(figsize=(6.8, 5.2))
    sns.regplot(data=d, x="shift_distance", y="roc_auc", scatter_kws={"s": 35}, line_kws={"linewidth": 2})
    plt.title(title)
    plt.xlabel("Mean-shift distance (Euclidean)")
    plt.ylabel("ROC-AUC")
    if len(d) >= 3:
        lr = linregress(d["shift_distance"].values, d["roc_auc"].values)
        txt = f"slope={lr.slope:.3f}, r={lr.rvalue:.3f}, p={lr.pvalue:.2g}"
        plt.gca().text(0.02, 0.02, txt, transform=plt.gca().transAxes, fontsize=10, va="bottom")
    save_fig(os.path.join(FIG_DIR, outname))

def plot_pub_auc_by_family(df: pd.DataFrame, title: str, outname: str):
    d = df.dropna(subset=["roc_auc"]).copy()
    if len(d) == 0:
        return
    order = sorted(d["family"].unique(), key=lambda x: (int(x.split("->")[0]), int(x.split("->")[1])))
    plt.figure(figsize=(max(8, 1.1 * len(order)), 5.5))
    sns.boxplot(data=d, x="family", y="roc_auc", order=order)
    plt.axhline(0.5, linestyle="--", linewidth=1)
    plt.xlabel("Transfer family")
    plt.ylabel("ROC-AUC")
    plt.title(title)
    save_fig(os.path.join(FIG_DIR, outname))

# -------------------------
# MAIN
# -------------------------
print("Loading sampleMetadata...")
md = load_sample_metadata(DATA_DIR)

study_col = "study_name" if "study_name" in md.columns else None
if study_col is None:
    raise ValueError("sampleMetadata missing required column: study_name")

cond_col = pick_col(md, ["study_condition", "condition", "disease", "diagnosis"])
if cond_col is None:
    raise ValueError("No condition column found in sampleMetadata. Tried: study_condition/condition/disease/diagnosis")

md = md.copy()
md["_sample_id"] = get_sample_ids(md).astype(str)
md["_study"] = md[study_col].astype(str)
md["_cond_raw"] = md[cond_col].astype(str)
md["_y"] = md["_cond_raw"].apply(map_binary_label)

md_bin = md.dropna(subset=["_y"]).copy()
md_bin["_y"] = md_bin["_y"].astype(int)

print(f"Total samples in sampleMetadata: {len(md)}")
print(f"Binary-labeled (CRC/control) samples: {len(md_bin)}")
print(f"Studies with binary-labeled samples: {md_bin['_study'].nunique()}")

sample_id_set = set(md["_sample_id"].astype(str)) | set(md.index.astype(str))

dtype_to_file = {}
all_dtypes = sorted(set(dt for dts in ANALYSIS_GROUPS.values() for dt in dts))
for dt in all_dtypes:
    dtype_to_file[dt] = discover_latest_file_per_study(dt, DATA_DIR)
    print(f"Discovered files for dtype={dt}: {len(dtype_to_file[dt])}")

group_study_data = {}
group_studies = {}

for group, dtypes in ANALYSIS_GROUPS.items():
    data = {}
    load_errors = []

    candidate = set()
    for dt in dtypes:
        candidate |= set(dtype_to_file.get(dt, {}).keys())
    studies_in_md = sorted(list(candidate & set(md_bin["_study"].unique().tolist())))

    for study in tqdm(studies_in_md, desc=f"Loading matrices for group={group}"):
        try:
            X = load_study_group_matrix(study, group, dtypes, dtype_to_file, sample_id_set)
            if X is None:
                continue

            md_s = md_bin[md_bin["_study"] == study].copy()
            md_s = md_s.set_index("_sample_id", drop=False)

            common = sorted(list(set(X.index.astype(str)) & set(md_s.index.astype(str))))
            if len(common) == 0:
                load_errors.append({"study": study, "error": "no_common_samples_between_matrix_and_metadata"})
                continue

            X = X.loc[common].copy()
            y = md_s.loc[common, "_y"].astype(int).values

            data[study] = {
                "X_raw": X,
                "y": y,
                "samples": np.array(common, dtype=str),
                "n": int(len(common)),
                "pos": int(np.sum(y == 1)),
                "neg": int(np.sum(y == 0)),
            }
        except Exception as e:
            load_errors.append({"study": study, "error": str(e)})

    group_study_data[group] = data
    eligible = filter_case_control_studies(data, min_pos=MIN_POS, min_neg=MIN_NEG)
    group_studies[group] = eligible

    print(f"\nGroup={group}: loaded {len(data)} studies with matrix+labels.")
    print(f"Group={group}: using {len(eligible)} CRC case-control cohorts (min_pos={MIN_POS}, min_neg={MIN_NEG}).")

    if len(load_errors):
        errp = os.path.join(OUT_DIR, FN[group]["load_errors"])
        pd.DataFrame(load_errors).to_csv(errp, index=False)
        print("Load errors saved:", errp)

    if len(eligible) == 0:
        tmp = pd.DataFrame(columns=["study", "n", "pos", "neg"])
        display(tmp)
        continue

    tmp = pd.DataFrame(
        [{"study": s, "n": data[s]["n"], "pos": data[s]["pos"], "neg": data[s]["neg"]} for s in eligible]
    ).sort_values(["pos", "n"], ascending=False)
    display(tmp)

run_groups = [g for g in ANALYSIS_GROUPS.keys() if len(group_studies.get(g, [])) >= 3]
skip_groups = [g for g in ANALYSIS_GROUPS.keys() if g not in run_groups]

if len(skip_groups):
    print("\nSkipping groups with <3 eligible cohorts:", skip_groups)

for group in run_groups:
    print("\n====================")
    print(f"RUNNING GROUP: {group}")
    print(f"Data types: {ANALYSIS_GROUPS[group]}")
    print(f"Cohorts: {len(group_studies[group])}")
    print("Prep:", GROUP_PREP[group])
    print("====================")

    study_dict = group_study_data[group]
    studies = group_studies[group]

    loso_metrics, loso_preds, loso_coefs_long, fold_coefs, global_union = run_loso(group, study_dict, studies)
    loso_metrics.to_csv(os.path.join(OUT_DIR, FN[group]["loso_metrics"]), index=False)
    loso_preds.to_csv(os.path.join(OUT_DIR, FN[group]["loso_preds"]), index=False)
    loso_coefs_long.to_csv(os.path.join(OUT_DIR, FN[group]["coefs_long"]), index=False)

    print("Saved:", os.path.join(OUT_DIR, FN[group]["loso_metrics"]))
    print("Saved:", os.path.join(OUT_DIR, FN[group]["loso_preds"]))
    print("Saved:", os.path.join(OUT_DIR, FN[group]["coefs_long"]))
    display(loso_metrics.head(10))

    if RUN_PUBLICATION_TRANSFER:
        pub_df, shift_pub_df = run_publication_transfer(group, study_dict, studies, families=PUBLICATION_FAMILIES)

        pub_df.to_csv(os.path.join(OUT_DIR, FN[group]["pub_transfer"]), index=False)
        print("Saved:", os.path.join(OUT_DIR, FN[group]["pub_transfer"]))
        display(pub_df.head(12))

        if len(pub_df):
            pub_summary = (
                pub_df.groupby("family", as_index=False)
                .agg(
                    n_pairs=("roc_auc", "size"),
                    n_valid_auc=("roc_auc", lambda x: int(np.sum(~pd.isna(x)))),
                    roc_auc_mean=("roc_auc", "mean"),
                    roc_auc_median=("roc_auc", "median"),
                    roc_auc_sd=("roc_auc", "std"),
                    roc_auc_min=("roc_auc", "min"),
                    roc_auc_max=("roc_auc", "max"),
                )
                .sort_values("family")
            )
        else:
            pub_summary = pd.DataFrame(columns=[
                "family", "n_pairs", "n_valid_auc",
                "roc_auc_mean", "roc_auc_median", "roc_auc_sd",
                "roc_auc_min", "roc_auc_max"
            ])

        pub_summary.to_csv(os.path.join(OUT_DIR, FN[group]["pub_summary"]), index=False)
        print("Saved:", os.path.join(OUT_DIR, FN[group]["pub_summary"]))
        display(pub_summary)

        if RUN_PUB_SHIFT:
            shift_pub_df.to_csv(os.path.join(OUT_DIR, FN[group]["shift_pub"]), index=False)
            print("Saved:", os.path.join(OUT_DIR, FN[group]["shift_pub"]))
            display(shift_pub_df.head(12))
    else:
        pub_df = pd.DataFrame()
        shift_pub_df = pd.DataFrame()

    shift_loso_df = run_shift_vs_auc_loso(group, study_dict, studies, loso_metrics)
    shift_loso_df.to_csv(os.path.join(OUT_DIR, FN[group]["shift_loso"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["shift_loso"]))
    display(shift_loso_df.head(12))

    stab_df = stability_summary(fold_coefs, global_union)
    stab_df.to_csv(os.path.join(OUT_DIR, FN[group]["stability"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["stability"]))
    display(stab_df.head(25))

    sig_df = stab_df[stab_df["stable"]].copy()
    sig_df.to_csv(os.path.join(OUT_DIR, FN[group]["signature"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["signature"]))
    print(f"Stable signature features: {len(sig_df)}")

    coefs_global_long = coefs_by_heldout_long(fold_coefs, global_union)
    coefs_global_long.to_csv(os.path.join(OUT_DIR, FN[group]["coefs_global_long"]), index=False)
    print("Saved:", os.path.join(OUT_DIR, FN[group]["coefs_global_long"]))

    plot_loso_auc(loso_metrics, title=f"{group} LOSO ROC-AUC", outname=f"{group}_loso_rocauc")
    plot_shift_scatter(shift_loso_df, title=f"{group}: LOSO shift vs ROC-AUC", outname=f"{group}_shift_vs_auc_loso")

    if len(pub_df):
        plot_pub_auc_by_family(
            pub_df,
            title=f"{group}: publication-oriented transfer families",
            outname=f"{group}_publication_transfer_by_family"
        )
    if len(shift_pub_df):
        plot_shift_scatter(
            shift_pub_df,
            title=f"{group}: publication transfer shift vs ROC-AUC",
            outname=f"{group}_shift_vs_auc_publication"
        )

    print("Figures saved in:", FIG_DIR)

print("\nDone.")
print("Outputs:", OUT_DIR)
print("Figures:", FIG_DIR)

Loading sampleMetadata...
Total samples in sampleMetadata: 22588
Binary-labeled (CRC/control) samples: 15822
Studies with binary-labeled samples: 82
Discovered files for dtype=gene_families: 11
Discovered files for dtype=marker_abundance: 11
Discovered files for dtype=marker_presence: 11
Discovered files for dtype=pathway_abundance: 11
Discovered files for dtype=pathway_coverage: 11
Discovered files for dtype=relative_abundance: 11


Loading matrices for group=species:   0%|          | 0/11 [00:00<?, ?it/s]


Group=species: loaded 9 studies with matrix+labels.
Group=species: using 9 CRC case-control cohorts (min_pos=20, min_neg=20).
Load errors saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/load_errors_species.csv


,study,n,pos,neg
7,YachidaS_2019,509,258,251
6,WirbelJ_2018,125,60,65
8,ZellerG_2014,114,53,61
5,VogtmannE_2016,104,52,52
4,ThomasAM_2019_c,80,40,40
3,ThomasAM_2018b,60,32,28
0,GuptaA_2019,60,30,30
2,ThomasAM_2018a,53,29,24
1,HanniganGD_2017,55,27,28


Loading matrices for group=pathways:   0%|          | 0/11 [00:00<?, ?it/s]

...

Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/shift_vs_auc_loso_markers_presence.csv


,heldout,shift_distance,roc_auc,n_train,n_test,n_feat
0,FengQ_2015,25.181168,0.810406,1288,107,20000
1,GuptaA_2019,39.785595,0.847778,1335,60,20000
2,HanniganGD_2017,34.961201,0.484127,1340,55,20000
3,ThomasAM_2018a,22.380648,0.824713,1342,53,20000
4,ThomasAM_2018b,25.331640,0.794643,1335,60,20000
5,ThomasAM_2019_c,14.152111,0.983125,1315,80,20000
6,VogtmannE_2016,21.648937,0.684541,1291,104,20000
7,WirbelJ_2018,21.938660,0.822564,1270,125,20000
8,YachidaS_2019,17.628933,0.742642,886,509,20000
9,YuJ_2015,20.113165,0.814064,1267,128,20000


Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_stability_summary.csv


,feature,freq_nonzero,mean_coef,sd_coef,sign_consistency,stable
36315,marker_presence::384639__GCA_003096855.1_01599,1.0,0.647675,0.252787,1.0,True
32104,marker_presence::29391__GCA_900476045.1_01564,1.0,0.543771,0.143804,1.0,True
1290,marker_presence::1134687__A0A377SEG4__A225_2996,1.0,0.520965,0.176946,1.0,True
47749,marker_presence::712288__G6C5I4__HMPREF9093_01831,1.0,0.464370,0.289324,1.0,True
34691,marker_presence::341694__E0E128__HMPREF0634_0779,1.0,0.434599,0.118904,1.0,True
26334,marker_presence::209880__A0A1G5WHZ1__SAMN02910...,1.0,0.395262,0.119706,1.0,True
33420,marker_presence::33033__A0A0B4S0E7__NW74_01920,1.0,0.384214,0.131892,1.0,True
2713,marker_presence::1261__A0A379CEJ7__prsA1,1.0,0.363554,0.156842,1.0,True
48242,marker_presence::713008__F9PTA1__HMPREF9127_1220,1.0,0.350204,0.150979,1.0,True
34690,marker_presence::341694__E0E125__HMPREF0634_0776,1.0,0.349623,0.121525,1.0,True


Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_signature_stable.csv
Stable signature features: 321
Saved: /content/curatedMetagenomicData_EH/analysis_outputs_crc/markers_presence_coefs_globalunion_by_heldout_long.csv
Figures saved in: /content/curatedMetagenomicData_EH/analysis_outputs_crc/figures

Done.
Outputs: /content/curatedMetagenomicData_EH/analysis_outputs_crc
Figures: /content/curatedMetagenomicData_EH/analysis_outputs_crc/figures


##3. Post-hoc augmentation / statistics

In [ ]:
# ============================================================
# POST-HOC AUGMENTATION
# ============================================================

!pip -q install statsmodels

import os
import re
import json
import math
import warnings
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import (
    mannwhitneyu,
    kruskal,
    spearmanr,
    pearsonr,
    linregress
)

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

import statsmodels.api as sm
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
sns.set_context("paper")

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------
_required_globals = [
    "group_study_data", "group_studies", "GROUP_PREP",
    "align_stack_filtered", "build_model", "tune_C_nested_cv",
    "apply_transform", "clip_probs", "has_both_classes",
    "calibration_intercept_slope", "sens_at_spec", "spec_at_sens",
    "RANDOM_STATE", "C_GRID", "N_INNER_CV"
]
_missing = [x for x in _required_globals if x not in globals()]
if len(_missing):
    raise RuntimeError(
        "This block must be run AFTER the main notebook. Missing globals:\n"
        + "\n".join(_missing)
    )

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_DIR = "/content/curatedMetagenomicData_EH"
OUT_DIR = os.path.join(DATA_DIR, "analysis_outputs_crc")
FIG_DIR = os.path.join(OUT_DIR, "figures")
AUG_DIR = os.path.join(OUT_DIR, "if10plus_augmented")
AUG_FIG_DIR = os.path.join(AUG_DIR, "figures")
AUG_TAB_DIR = os.path.join(AUG_DIR, "tables")
AUG_TXT_DIR = os.path.join(AUG_DIR, "reports")
os.makedirs(AUG_DIR, exist_ok=True)
os.makedirs(AUG_FIG_DIR, exist_ok=True)
os.makedirs(AUG_TAB_DIR, exist_ok=True)
os.makedirs(AUG_TXT_DIR, exist_ok=True)

PUBLICATION_FAMILIES = [
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (3, 1),
    (2, 2),
]

REPRESENTATIONS = ["species", "pathways", "markers_abundance", "markers_presence"]
NOVELTY_GROUPS = ["species", "pathways", "markers_abundance", "markers_presence"]
NOVELTY_FAMILIES = [(1, 1), (2, 1), (3, 1)]

MAX_TRAIN_COMBOS_PER_FAMILY = None
MAX_TESTS_PER_TRAIN_COMBO = None

CORAL_EPS = 1e-3
CORAL_MAX_FULL_FEATURES = 1500
USE_DIAGONAL_CORAL_FALLBACK = True

# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------
def fp_publication_transfer(group):
    return os.path.join(OUT_DIR, f"publication_transfer_auc_{group}.csv")

def fp_publication_summary(group):
    return os.path.join(OUT_DIR, f"publication_transfer_summary_{group}.csv")

def fp_shift_publication(group):
    return os.path.join(OUT_DIR, f"shift_vs_auc_publication_{group}.csv")

def fp_loso_metrics(group):
    return os.path.join(OUT_DIR, f"loso_{group}_metrics_next.csv")

def fp_loso_shift(group):
    return os.path.join(OUT_DIR, f"shift_vs_auc_loso_{group}.csv")

def fp_stability(group):
    return os.path.join(OUT_DIR, f"{group}_stability_summary.csv")

def fp_signature(group):
    return os.path.join(OUT_DIR, f"{group}_signature_stable.csv")

def safe_read_csv(path):
    if not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Failed to read {path}: {e}")
        return None

def has_cols(df, cols):
    return (df is not None) and (len(df) > 0) and set(cols).issubset(df.columns)

def safe_subset_dropna(df, cols):
    if not has_cols(df, cols):
        return pd.DataFrame(columns=list(cols))
    return df.dropna(subset=list(cols)).copy()

# ------------------------------------------------------------
# GENERAL HELPERS
# ------------------------------------------------------------
def bh_adjust(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    if n == 0:
        return np.array([])
    order = np.argsort(pvals)
    ranks = np.empty(n, dtype=int)
    ranks[order] = np.arange(1, n + 1)
    adj = pvals * n / ranks
    adj_sorted = adj[order]
    adj_sorted = np.minimum.accumulate(adj_sorted[::-1])[::-1]
    out = np.empty(n, dtype=float)
    out[order] = np.minimum(adj_sorted, 1.0)
    return out

def cliffs_delta(x, y):
    x = np.asarray(pd.Series(x).dropna(), dtype=float)
    y = np.asarray(pd.Series(y).dropna(), dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    return (gt - lt) / (len(x) * len(y))

def bootstrap_ci(values, stat_fn=np.median, n_boot=2000, alpha=0.05, seed=7):
    vals = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(vals) == 0:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boots = []
    n = len(vals)
    for _ in range(n_boot):
        samp = rng.choice(vals, size=n, replace=True)
        boots.append(stat_fn(samp))
    boots = np.asarray(boots)
    lo = np.quantile(boots, alpha / 2)
    hi = np.quantile(boots, 1 - alpha / 2)
    return (float(stat_fn(vals)), float(lo), float(hi))

def save_fig(path_base):
    plt.tight_layout()
    plt.savefig(path_base + ".png", dpi=300, bbox_inches="tight")
    plt.savefig(path_base + ".pdf", bbox_inches="tight")
    plt.close()

def combo_to_str(combo):
    return " | ".join(combo)

def family_to_str(a, b):
    return f"{a}->{b}"

def safe_corr(x, y, method="spearman"):
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)
    mask = x.notna() & y.notna()
    x = x[mask].values
    y = y[mask].values
    if len(x) < 3:
        return np.nan, np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    try:
        if method == "spearman":
            return spearmanr(x, y)
        elif method == "pearson":
            return pearsonr(x, y)
        else:
            raise ValueError("method must be 'spearman' or 'pearson'")
    except Exception:
        return np.nan, np.nan

# ------------------------------------------------------------
# LOAD SAVED RESULTS
# ------------------------------------------------------------
pub_results = {}
shift_results = {}
loso_metrics_results = {}
loso_shift_results = {}
stab_results = {}
sig_results = {}

for g in REPRESENTATIONS:
    pub_results[g] = safe_read_csv(fp_publication_transfer(g))
    shift_results[g] = safe_read_csv(fp_shift_publication(g))
    loso_metrics_results[g] = safe_read_csv(fp_loso_metrics(g))
    loso_shift_results[g] = safe_read_csv(fp_loso_shift(g))
    stab_results[g] = safe_read_csv(fp_stability(g))
    sig_results[g] = safe_read_csv(fp_signature(g))

available_groups = [
    g for g in REPRESENTATIONS
    if has_cols(pub_results[g], ["family", "roc_auc"])
]
print("Available groups:", available_groups)

# ------------------------------------------------------------
# 1) FORMAL STATISTICAL COMPARISON BETWEEN TRANSFER FAMILIES
# ------------------------------------------------------------
def run_family_statistics(pub_df, group_name):
    if not has_cols(pub_df, ["family", "roc_auc"]):
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df = pub_df.dropna(subset=["roc_auc"]).copy()
    fam_order = [family_to_str(a, b) for a, b in PUBLICATION_FAMILIES if family_to_str(a, b) in set(df["family"])]

    out_rows = []
    for fam in fam_order:
        vals = df.loc[df["family"] == fam, "roc_auc"].values
        med, lo, hi = bootstrap_ci(vals, stat_fn=np.median, n_boot=2000, alpha=0.05, seed=RANDOM_STATE)
        mean_, lo_m, hi_m = bootstrap_ci(vals, stat_fn=np.mean, n_boot=2000, alpha=0.05, seed=RANDOM_STATE)
        out_rows.append({
            "group": group_name,
            "family": fam,
            "n": int(len(vals)),
            "roc_auc_mean": float(np.mean(vals)) if len(vals) else np.nan,
            "roc_auc_median": float(np.median(vals)) if len(vals) else np.nan,
            "roc_auc_sd": float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan,
            "roc_auc_mean_ci_low": lo_m,
            "roc_auc_mean_ci_high": hi_m,
            "roc_auc_median_ci_low": lo,
            "roc_auc_median_ci_high": hi,
        })
    desc_df = pd.DataFrame(out_rows)

    family_groups = [df.loc[df["family"] == fam, "roc_auc"].dropna().values for fam in fam_order]
    family_groups = [x for x in family_groups if len(x) > 0]
    kw_stat, kw_p = kruskal(*family_groups) if len(family_groups) >= 2 else (np.nan, np.nan)

    kw_df = pd.DataFrame([{
        "group": group_name,
        "test": "Kruskal-Wallis",
        "n_families": len(fam_order),
        "statistic": kw_stat,
        "p_value": kw_p
    }])

    pair_rows = []
    for i in range(len(fam_order)):
        for j in range(i + 1, len(fam_order)):
            f1 = fam_order[i]
            f2 = fam_order[j]
            x = df.loc[df["family"] == f1, "roc_auc"].dropna().values
            y = df.loc[df["family"] == f2, "roc_auc"].dropna().values

            if len(x) == 0 or len(y) == 0:
                stat, p = np.nan, np.nan
            else:
                try:
                    stat, p = mannwhitneyu(x, y, alternative="two-sided")
                except Exception:
                    stat, p = np.nan, np.nan

            pair_rows.append({
                "group": group_name,
                "family_1": f1,
                "family_2": f2,
                "n1": len(x),
                "n2": len(y),
                "median_1": float(np.median(x)) if len(x) else np.nan,
                "median_2": float(np.median(y)) if len(y) else np.nan,
                "mean_1": float(np.mean(x)) if len(x) else np.nan,
                "mean_2": float(np.mean(y)) if len(y) else np.nan,
                "delta_median": float(np.median(x) - np.median(y)) if len(x) and len(y) else np.nan,
                "delta_mean": float(np.mean(x) - np.mean(y)) if len(x) and len(y) else np.nan,
                "cliffs_delta": cliffs_delta(x, y),
                "mw_statistic": stat,
                "p_value": p,
            })

    pair_df = pd.DataFrame(pair_rows)
    if len(pair_df):
        pair_df["p_adj_bh"] = bh_adjust(pair_df["p_value"].fillna(1.0).values)

    return desc_df, kw_df, pair_df

family_desc_all = []
family_kw_all = []
family_pair_all = []

for g in available_groups:
    d1, d2, d3 = run_family_statistics(pub_results[g], g)
    if len(d1): family_desc_all.append(d1)
    if len(d2): family_kw_all.append(d2)
    if len(d3): family_pair_all.append(d3)

family_desc_all = pd.concat(family_desc_all, ignore_index=True) if len(family_desc_all) else pd.DataFrame()
family_kw_all = pd.concat(family_kw_all, ignore_index=True) if len(family_kw_all) else pd.DataFrame()
family_pair_all = pd.concat(family_pair_all, ignore_index=True) if len(family_pair_all) else pd.DataFrame()

family_desc_all.to_csv(os.path.join(AUG_TAB_DIR, "family_descriptive_statistics.csv"), index=False)
family_kw_all.to_csv(os.path.join(AUG_TAB_DIR, "family_global_kruskal_tests.csv"), index=False)
family_pair_all.to_csv(os.path.join(AUG_TAB_DIR, "family_pairwise_mannwhitney_tests.csv"), index=False)

print("\nSaved family-level inferential statistics.")
display(family_desc_all.head(12))
display(family_kw_all)
display(family_pair_all.head(12))

for g in available_groups:
    d = safe_subset_dropna(pub_results[g], ["family", "roc_auc"])
    if len(d) == 0:
        continue
    fam_order = [family_to_str(a, b) for a, b in PUBLICATION_FAMILIES if family_to_str(a, b) in set(d["family"])]
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=d, x="family", y="roc_auc", order=fam_order)
    plt.axhline(0.5, ls="--", lw=1)
    plt.title(f"{g}: ROC-AUC distributions across transfer families")
    plt.xlabel("Transfer family")
    plt.ylabel("ROC-AUC")
    save_fig(os.path.join(AUG_FIG_DIR, f"{g}_family_rocauc_boxplot"))

# ------------------------------------------------------------
# 2) STRONGER SHIFT-PERFORMANCE ANALYSIS
# ------------------------------------------------------------
def run_shift_analysis(pub_df, shift_df, loso_shift_df, group_name):
    rows = []
    ols_df = pd.DataFrame(columns=["term", "coef", "p_value", "ci_low", "ci_high", "group"])

    if has_cols(shift_df, ["shift_distance", "roc_auc"]):
        d = shift_df.dropna(subset=["shift_distance", "roc_auc"]).copy()

        sp_r, sp_p = safe_corr(d["shift_distance"], d["roc_auc"], method="spearman")
        pr_r, pr_p = safe_corr(d["shift_distance"], d["roc_auc"], method="pearson")

        rows.append({
            "group": group_name,
            "scope": "publication_all",
            "n": len(d),
            "spearman_r": sp_r,
            "spearman_p": sp_p,
            "pearson_r": pr_r,
            "pearson_p": pr_p,
            "mean_shift": float(d["shift_distance"].mean()) if len(d) else np.nan,
            "median_shift": float(d["shift_distance"].median()) if len(d) else np.nan,
            "mean_auc": float(d["roc_auc"].mean()) if len(d) else np.nan,
            "median_auc": float(d["roc_auc"].median()) if len(d) else np.nan,
        })

        if "family" in d.columns:
            for fam in sorted(d["family"].dropna().unique()):
                dfam = d[d["family"] == fam].copy()
                sp_r, sp_p = safe_corr(dfam["shift_distance"], dfam["roc_auc"], method="spearman")
                pr_r, pr_p = safe_corr(dfam["shift_distance"], dfam["roc_auc"], method="pearson")
                rows.append({
                    "group": group_name,
                    "scope": fam,
                    "n": len(dfam),
                    "spearman_r": sp_r,
                    "spearman_p": sp_p,
                    "pearson_r": pr_r,
                    "pearson_p": pr_p,
                    "mean_shift": float(dfam["shift_distance"].mean()) if len(dfam) else np.nan,
                    "median_shift": float(dfam["shift_distance"].median()) if len(dfam) else np.nan,
                    "mean_auc": float(dfam["roc_auc"].mean()) if len(dfam) else np.nan,
                    "median_auc": float(dfam["roc_auc"].median()) if len(dfam) else np.nan,
                })

        d_mod = d.copy()
        if len(d_mod) >= 5 and d_mod["shift_distance"].nunique() > 1:
            if "family" in d_mod.columns:
                dummies = pd.get_dummies(d_mod["family"], prefix="family", drop_first=True, dtype=float)
                X = pd.concat([d_mod[["shift_distance"]].astype(float), dummies], axis=1)
            else:
                X = d_mod[["shift_distance"]].astype(float)

            X = sm.add_constant(X, has_constant="add")
            y = d_mod["roc_auc"].astype(float)

            try:
                ols = sm.OLS(y, X).fit()
                conf = ols.conf_int()
                ols_df = pd.DataFrame({
                    "term": ols.params.index,
                    "coef": ols.params.values,
                    "p_value": ols.pvalues.values,
                    "ci_low": conf.iloc[:, 0].values,
                    "ci_high": conf.iloc[:, 1].values,
                    "group": group_name
                })
            except Exception:
                pass

    out_df = pd.DataFrame(rows)

    if has_cols(loso_shift_df, ["shift_distance", "roc_auc"]):
        d = loso_shift_df.dropna(subset=["shift_distance", "roc_auc"]).copy()
        sp_r, sp_p = safe_corr(d["shift_distance"], d["roc_auc"], method="spearman")
        pr_r, pr_p = safe_corr(d["shift_distance"], d["roc_auc"], method="pearson")

        loso_shift_summary_df = pd.DataFrame([{
            "group": group_name,
            "scope": "LOSO",
            "n": len(d),
            "spearman_r": sp_r,
            "spearman_p": sp_p,
            "pearson_r": pr_r,
            "pearson_p": pr_p,
            "mean_shift": float(d["shift_distance"].mean()) if len(d) else np.nan,
            "median_shift": float(d["shift_distance"].median()) if len(d) else np.nan,
            "mean_auc": float(d["roc_auc"].mean()) if len(d) else np.nan,
            "median_auc": float(d["roc_auc"].median()) if len(d) else np.nan,
        }])
    else:
        loso_shift_summary_df = pd.DataFrame()

    return out_df, ols_df, loso_shift_summary_df

shift_summary_all = []
shift_ols_all = []
shift_loso_all = []

for g in available_groups:
    s1, s2, s3 = run_shift_analysis(pub_results[g], shift_results[g], loso_shift_results[g], g)
    if len(s1): shift_summary_all.append(s1)
    if len(s2): shift_ols_all.append(s2)
    if len(s3): shift_loso_all.append(s3)

shift_summary_all = pd.concat(shift_summary_all, ignore_index=True) if len(shift_summary_all) else pd.DataFrame()
shift_ols_all = pd.concat(shift_ols_all, ignore_index=True) if len(shift_ols_all) else pd.DataFrame()
shift_loso_all = pd.concat(shift_loso_all, ignore_index=True) if len(shift_loso_all) else pd.DataFrame()

shift_summary_all.to_csv(os.path.join(AUG_TAB_DIR, "shift_auc_summary_by_group_and_family.csv"), index=False)
shift_ols_all.to_csv(os.path.join(AUG_TAB_DIR, "shift_auc_family_adjusted_ols.csv"), index=False)
shift_loso_all.to_csv(os.path.join(AUG_TAB_DIR, "shift_auc_loso_summary.csv"), index=False)

print("\nSaved stronger shift-performance analyses.")
display(shift_summary_all.head(12))
display(shift_ols_all.head(12))
display(shift_loso_all)

for g in available_groups:
    d = shift_results[g]
    if not has_cols(d, ["shift_distance", "roc_auc"]):
        continue

    d = d.dropna(subset=["shift_distance", "roc_auc"]).copy()

    plt.figure(figsize=(6.5, 5.0))
    sns.regplot(data=d, x="shift_distance", y="roc_auc", scatter_kws={"s": 18}, line_kws={"lw": 2})
    plt.title(f"{g}: shift distance vs ROC-AUC (publication transfer)")
    plt.xlabel("Shift distance")
    plt.ylabel("ROC-AUC")
    save_fig(os.path.join(AUG_FIG_DIR, f"{g}_shift_vs_auc_overall"))

    if "family" in d.columns:
        plt.figure(figsize=(7.2, 5.5))
        sns.scatterplot(data=d, x="shift_distance", y="roc_auc", hue="family", s=18)
        plt.title(f"{g}: shift distance vs ROC-AUC by family")
        plt.xlabel("Shift distance")
        plt.ylabel("ROC-AUC")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        save_fig(os.path.join(AUG_FIG_DIR, f"{g}_shift_vs_auc_by_family"))

# ------------------------------------------------------------
# 3) CLEANER BIOLOGICAL SIGNATURE INTERPRETATION
# ------------------------------------------------------------
def parse_tax_feature(feature):
    genus = None
    species = None
    g = re.search(r"g__([^|]+)", str(feature))
    s = re.search(r"s__([^|]+)", str(feature))
    if g:
        genus = g.group(1)
    if s:
        species = s.group(1)
    if species and species not in ["", "_", "s__"]:
        label = f"{genus} {species}" if genus else species
    elif genus:
        label = genus
    else:
        label = str(feature)
    return label

def parse_pathway_feature(feature):
    txt = str(feature).split("::", 1)[-1]
    if ":" in txt:
        pid, desc = txt.split(":", 1)
        return pid.strip(), desc.strip()
    return txt.strip(), txt.strip()

def summarize_signatures(group_name, stab_df, sig_df):
    required = ["feature", "mean_coef"]
    if not has_cols(stab_df, required):
        return pd.DataFrame(), pd.DataFrame()

    d = stab_df.copy()
    if "stable" in d.columns:
        d = d[d["stable"] == True].copy()

    if len(d) == 0:
        return pd.DataFrame(), pd.DataFrame()

    if "sign_consistency" not in d.columns:
        d["sign_consistency"] = np.nan
    if "freq_nonzero" not in d.columns:
        d["freq_nonzero"] = np.nan

    d["abs_mean_coef"] = d["mean_coef"].abs()

    if group_name == "species":
        d["interpretable_label"] = d["feature"].apply(parse_tax_feature)
        d["feature_type"] = "taxon"
    elif group_name == "pathways":
        d[["pathway_id", "pathway_desc"]] = d["feature"].apply(lambda x: pd.Series(parse_pathway_feature(x)))
        d["interpretable_label"] = d["pathway_desc"]
        d["feature_type"] = d["feature"].astype(str).str.split("::").str[0]
    elif "marker" in group_name:
        d["interpretable_label"] = d["feature"].astype(str).str.split("::").str[-1]
        d["feature_type"] = group_name
    else:
        d["interpretable_label"] = d["feature"].astype(str)
        d["feature_type"] = group_name

    top_pos = d.sort_values("mean_coef", ascending=False).head(25).copy()
    top_neg = d.sort_values("mean_coef", ascending=True).head(25).copy()
    summary = pd.concat([top_pos.assign(direction="positive"), top_neg.assign(direction="negative")], axis=0)

    if group_name in ["species", "pathways"]:
        if group_name == "species":
            agg = (
                d.groupby("interpretable_label", as_index=False)
                .agg(
                    n_features=("feature", "size"),
                    mean_coef_mean=("mean_coef", "mean"),
                    mean_coef_abs_sum=("abs_mean_coef", "sum"),
                    sign_consistency_mean=("sign_consistency", "mean"),
                    freq_nonzero_mean=("freq_nonzero", "mean")
                )
                .sort_values("mean_coef_abs_sum", ascending=False)
            )
        else:
            agg = (
                d.groupby("feature_type", as_index=False)
                .agg(
                    n_features=("feature", "size"),
                    mean_coef_abs_sum=("abs_mean_coef", "sum"),
                    mean_coef_mean=("mean_coef", "mean"),
                    sign_consistency_mean=("sign_consistency", "mean")
                )
                .sort_values("mean_coef_abs_sum", ascending=False)
            )
    else:
        agg = (
            d.groupby("feature_type", as_index=False)
            .agg(
                n_features=("feature", "size"),
                mean_coef_abs_sum=("abs_mean_coef", "sum"),
                mean_coef_mean=("mean_coef", "mean"),
                sign_consistency_mean=("sign_consistency", "mean")
            )
            .sort_values("mean_coef_abs_sum", ascending=False)
        )

    return summary, agg

bio_summary_all = []
bio_agg_all = []

for g in available_groups:
    summary_df, agg_df = summarize_signatures(g, stab_results[g], sig_results[g])
    if len(summary_df):
        summary_df["group"] = g
        summary_df.to_csv(os.path.join(AUG_TAB_DIR, f"{g}_interpretable_signature_top_features.csv"), index=False)
        bio_summary_all.append(summary_df)
    if len(agg_df):
        agg_df["group"] = g
        agg_df.to_csv(os.path.join(AUG_TAB_DIR, f"{g}_interpretable_signature_aggregates.csv"), index=False)
        bio_agg_all.append(agg_df)

bio_summary_all = pd.concat(bio_summary_all, ignore_index=True) if len(bio_summary_all) else pd.DataFrame()
bio_agg_all = pd.concat(bio_agg_all, ignore_index=True) if len(bio_agg_all) else pd.DataFrame()

print("\nSaved cleaner biological signature interpretation tables.")
display(bio_summary_all.head(12))
display(bio_agg_all.head(12))

for g in available_groups:
    stab_df = stab_results[g]
    if not has_cols(stab_df, ["feature", "mean_coef"]):
        continue

    d = stab_df.copy()
    if "stable" in d.columns:
        d = d[d["stable"] == True].copy()
    if len(d) == 0:
        continue

    d["abs_mean_coef"] = d["mean_coef"].abs()

    if g == "species":
        d["label"] = d["feature"].apply(parse_tax_feature)
    elif g == "pathways":
        d["label"] = d["feature"].apply(lambda x: parse_pathway_feature(x)[1])
    else:
        d["label"] = d["feature"].astype(str).str.split("::").str[-1]

    top = pd.concat([
        d.sort_values("mean_coef", ascending=False).head(12),
        d.sort_values("mean_coef", ascending=True).head(12)
    ], axis=0).drop_duplicates(subset=["feature"]).copy()

    plt.figure(figsize=(9, max(6, 0.28 * len(top))))
    top = top.sort_values("mean_coef")
    colors = np.where(top["mean_coef"] > 0, "tab:red", "tab:blue")
    plt.barh(top["label"], top["mean_coef"], color=colors)
    plt.axvline(0, color="black", lw=1)
    plt.title(f"{g}: top positive and negative stable signature features")
    plt.xlabel("Mean coefficient across LOSO folds")
    plt.ylabel("")
    save_fig(os.path.join(AUG_FIG_DIR, f"{g}_top_stable_signature_features"))

# ------------------------------------------------------------
# 4) REPRESENTATION COMPARISON + PRACTICAL GUIDANCE
# ------------------------------------------------------------
repr_rows = []

for g in available_groups:
    pub_df = pub_results[g]
    loso_df = loso_metrics_results[g]
    shift_df = shift_results[g]
    stab_df = stab_results[g]

    row = {"group": g}

    for fam in ["1->1", "1->2", "1->3", "2->1", "2->2", "3->1"]:
        vals = pub_df.loc[pub_df["family"] == fam, "roc_auc"].dropna().values if has_cols(pub_df, ["family", "roc_auc"]) else np.array([])
        row[f"{fam}_median_auc"] = float(np.median(vals)) if len(vals) else np.nan
        row[f"{fam}_mean_auc"] = float(np.mean(vals)) if len(vals) else np.nan

    if has_cols(loso_df, ["roc_auc"]):
        row["loso_mean_auc"] = float(loso_df["roc_auc"].mean())
        row["loso_median_auc"] = float(loso_df["roc_auc"].median())
        row["loso_mean_brier"] = float(loso_df["brier"].mean()) if "brier" in loso_df.columns else np.nan
    else:
        row["loso_mean_auc"] = np.nan
        row["loso_median_auc"] = np.nan
        row["loso_mean_brier"] = np.nan

    if has_cols(shift_df, ["shift_distance", "roc_auc"]):
        dd = shift_df.dropna(subset=["shift_distance", "roc_auc"]).copy()
        sp_r, sp_p = safe_corr(dd["shift_distance"], dd["roc_auc"], method="spearman")
        row["shift_spearman_r"] = float(sp_r) if pd.notna(sp_r) else np.nan
        row["shift_spearman_p"] = float(sp_p) if pd.notna(sp_p) else np.nan
    else:
        row["shift_spearman_r"] = np.nan
        row["shift_spearman_p"] = np.nan

    if has_cols(stab_df, ["feature"]):
        if "stable" in stab_df.columns:
            row["n_stable_features"] = int((stab_df["stable"] == True).sum())
        else:
            row["n_stable_features"] = int(len(stab_df))
    else:
        row["n_stable_features"] = 0

    row["gain_2to1_vs_1to1"] = row["2->1_median_auc"] - row["1->1_median_auc"]
    row["gain_3to1_vs_1to1"] = row["3->1_median_auc"] - row["1->1_median_auc"]
    row["gain_2to2_vs_1to1"] = row["2->2_median_auc"] - row["1->1_median_auc"]

    repr_rows.append(row)

repr_df = pd.DataFrame(repr_rows)

def recommend_use_case(r):
    if pd.notna(r["3->1_median_auc"]) and r["3->1_median_auc"] >= 0.72:
        return "Best candidate for multi-source external transportability"
    if pd.notna(r["1->1_median_auc"]) and r["1->1_median_auc"] >= 0.65:
        return "Strong single-source baseline; useful when cohorts are limited"
    if pd.notna(r["gain_3to1_vs_1to1"]) and r["gain_3to1_vs_1to1"] >= 0.05:
        return "Benefits substantially from source aggregation"
    return "Moderate transfer; likely best as complementary representation"

repr_df["practical_guidance"] = repr_df.apply(recommend_use_case, axis=1)
repr_df.to_csv(os.path.join(AUG_TAB_DIR, "representation_comparison_practical_guidance.csv"), index=False)

print("\nSaved representation comparison with practical guidance.")
display(repr_df)

plot_cols = ["1->1_median_auc", "2->1_median_auc", "3->1_median_auc", "2->2_median_auc", "loso_mean_auc"]
melt_df = repr_df.melt(
    id_vars=["group"],
    value_vars=plot_cols,
    var_name="metric",
    value_name="value"
)
plt.figure(figsize=(9.5, 5.5))
sns.barplot(data=melt_df, x="group", y="value", hue="metric")
plt.axhline(0.5, ls="--", lw=1)
plt.title("Representation comparison across key transfer settings")
plt.xlabel("")
plt.ylabel("ROC-AUC")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_fig(os.path.join(AUG_FIG_DIR, "representation_comparison_key_metrics"))

# ------------------------------------------------------------
# 5) TRANSPORTABILITY SCORE FRAMEWORK
# ------------------------------------------------------------
def minmax_scale(s):
    s = pd.Series(s, dtype=float)
    if s.nunique(dropna=True) <= 1:
        return pd.Series(np.ones(len(s)) * 0.5, index=s.index)
    return (s - s.min()) / (s.max() - s.min())

score_df = repr_df.copy()

score_df["score_auc_single"] = minmax_scale(score_df["1->1_median_auc"])
score_df["score_auc_multi"] = minmax_scale(score_df["3->1_median_auc"])
score_df["score_multisource_gain"] = minmax_scale(score_df["gain_3to1_vs_1to1"])
score_df["score_calibration"] = minmax_scale(-score_df["loso_mean_brier"])
score_df["score_signature_stability"] = minmax_scale(np.log1p(score_df["n_stable_features"]))
score_df["score_shift_robustness"] = minmax_scale(-score_df["shift_spearman_r"].fillna(score_df["shift_spearman_r"].median()))

score_df["transportability_score"] = (
    0.20 * score_df["score_auc_single"] +
    0.25 * score_df["score_auc_multi"] +
    0.15 * score_df["score_multisource_gain"] +
    0.15 * score_df["score_calibration"] +
    0.10 * score_df["score_signature_stability"] +
    0.15 * score_df["score_shift_robustness"]
)

score_df = score_df.sort_values("transportability_score", ascending=False).reset_index(drop=True)
score_df.to_csv(os.path.join(AUG_TAB_DIR, "transportability_score_framework.csv"), index=False)

print("\nSaved composite transportability score framework.")
display(score_df[[
    "group",
    "transportability_score",
    "1->1_median_auc",
    "3->1_median_auc",
    "gain_3to1_vs_1to1",
    "loso_mean_auc",
    "loso_mean_brier",
    "shift_spearman_r",
    "n_stable_features"
]])

plt.figure(figsize=(8, 5))
sns.barplot(data=score_df, x="group", y="transportability_score")
plt.title("Composite transportability score by representation")
plt.xlabel("")
plt.ylabel("Composite score")
save_fig(os.path.join(AUG_FIG_DIR, "composite_transportability_score"))

# ------------------------------------------------------------
# 6) NOVELTY LAYER:
#    SOURCE-SIDE PLATT RECALIBRATION + UNSUPERVISED CORAL
# ------------------------------------------------------------
def family_train_test_index(studies, family):
    a, b = family
    out = []
    for tr in combinations(studies, a):
        rem = [s for s in studies if s not in set(tr)]
        tests = list(combinations(rem, b))
        out.append((tuple(tr), tests))
    return out

def crossfit_source_platt_calibrator(X_train, y_train, use_scaler, best_C, n_features, n_splits=3):
    y_train = np.asarray(y_train).astype(int)
    if len(np.unique(y_train)) < 2:
        return None

    n_pos = np.sum(y_train == 1)
    n_neg = np.sum(y_train == 0)
    splits = min(n_splits, int(max(2, min(n_pos, n_neg))))
    if splits < 2:
        return None

    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_STATE)
    oof_logits = np.zeros(len(y_train), dtype=float)

    for tr_idx, va_idx in skf.split(X_train, y_train):
        model = build_model(best_C, use_scaler=use_scaler, n_features=n_features)
        model.fit(X_train[tr_idx], y_train[tr_idx])
        p = clip_probs(model.predict_proba(X_train[va_idx])[:, 1])
        oof_logits[va_idx] = np.log(p / (1 - p))

    cal = LogisticRegression(
        penalty="l2",
        C=10.0,
        solver="lbfgs",
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    cal.fit(oof_logits.reshape(-1, 1), y_train)
    return cal

def apply_platt_calibrator(base_probs, calibrator):
    if calibrator is None:
        return base_probs
    p = clip_probs(base_probs)
    logits = np.log(p / (1 - p)).reshape(-1, 1)
    p_cal = clip_probs(calibrator.predict_proba(logits)[:, 1])
    return p_cal

def coral_align_target_to_source(Xs, Xt, eps=1e-3, max_full_features=1500, diagonal_fallback=True):
    Xs = np.asarray(Xs, dtype=np.float64)
    Xt = np.asarray(Xt, dtype=np.float64)

    mu_s = Xs.mean(axis=0, keepdims=True)
    mu_t = Xt.mean(axis=0, keepdims=True)

    Xs0 = Xs - mu_s
    Xt0 = Xt - mu_t
    d = Xs0.shape[1]

    # Fast diagonal fallback for very high-dimensional representations
    if d > max_full_features:
        if not diagonal_fallback:
            raise MemoryError(f"Too many features for full CORAL: d={d}")
        var_s = Xs0.var(axis=0) + eps
        var_t = Xt0.var(axis=0) + eps
        Xt_aligned = Xt0 * np.sqrt(var_s / var_t) + mu_s
        return Xt_aligned.astype(np.float32)

    Cs = np.cov(Xs0, rowvar=False) + eps * np.eye(d)
    Ct = np.cov(Xt0, rowvar=False) + eps * np.eye(d)

    es, Vs = np.linalg.eigh(Cs)
    et, Vt = np.linalg.eigh(Ct)

    es = np.maximum(es, eps)
    et = np.maximum(et, eps)

    Cs_sqrt = Vs @ np.diag(np.sqrt(es)) @ Vs.T
    Ct_inv_sqrt = Vt @ np.diag(1.0 / np.sqrt(et)) @ Vt.T

    Xt_aligned = Xt0 @ Ct_inv_sqrt @ Cs_sqrt + mu_s
    return Xt_aligned.astype(np.float32)

def evaluate_probs(y_true, p):
    if not has_both_classes(y_true):
        return {
            "roc_auc": np.nan,
            "pr_auc": np.nan,
            "brier": np.nan,
            "sens_at_90spec": np.nan,
            "spec_at_90sens": np.nan,
            "cal_intercept": np.nan,
            "cal_slope": np.nan
        }
    cal_i, cal_s = calibration_intercept_slope(y_true, p)
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "pr_auc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
        "sens_at_90spec": sens_at_spec(y_true, p, spec_target=0.90),
        "spec_at_90sens": spec_at_sens(y_true, p, sens_target=0.90),
        "cal_intercept": cal_i,
        "cal_slope": cal_s
    }

def run_novelty_experiments_for_group(group_name, families=NOVELTY_FAMILIES):
    if group_name not in group_studies or len(group_studies[group_name]) == 0:
        return pd.DataFrame()

    prep = GROUP_PREP[group_name]
    transform = prep["transform"]
    pseudocount = prep["pseudocount"]
    use_scaler = prep["use_scaler"]
    min_prev = prep["min_prev"]
    min_var = prep["min_var"]
    max_features = prep["max_features"]

    study_dict = group_study_data[group_name]
    studies = list(group_studies[group_name])

    rows = []

    for family in families:
        a, b = family
        family_label = family_to_str(a, b)

        if a + b > len(studies):
            continue

        family_index = family_train_test_index(studies, family)
        if MAX_TRAIN_COMBOS_PER_FAMILY is not None:
            family_index = family_index[:MAX_TRAIN_COMBOS_PER_FAMILY]

        for train_combo, test_combos in tqdm(family_index, desc=f"Novelty {group_name} {family_label}"):
            train_combo = list(train_combo)

            X_train, y_train, _, _, feat_list = align_stack_filtered(
                study_dict, train_combo,
                min_prev=min_prev, min_var=min_var, max_features=max_features,
                transform=transform, pseudocount=pseudocount
            )

            if not has_both_classes(y_train):
                continue

            n_feat = int(X_train.shape[1])
            best_C, _ = tune_C_nested_cv(X_train, y_train, use_scaler=use_scaler, C_grid=C_GRID, n_splits=N_INNER_CV)

            base_model = build_model(best_C, use_scaler=use_scaler, n_features=n_feat)
            base_model.fit(X_train, y_train)

            platt = crossfit_source_platt_calibrator(
                X_train=X_train,
                y_train=y_train,
                use_scaler=use_scaler,
                best_C=best_C,
                n_features=n_feat,
                n_splits=3
            )

            if MAX_TESTS_PER_TRAIN_COMBO is not None:
                test_combos = test_combos[:MAX_TESTS_PER_TRAIN_COMBO]

            for test_combo in test_combos:
                test_combo = list(test_combo)

                X_test_list = []
                y_test_list = []
                for st in test_combo:
                    X_df = study_dict[st]["X_raw"].reindex(columns=feat_list, fill_value=0.0)
                    X = X_df.to_numpy(dtype=np.float32, copy=False)
                    X = apply_transform(X, mode=transform, pseudocount=pseudocount)
                    y = study_dict[st]["y"].astype(int)
                    X_test_list.append(X)
                    y_test_list.append(y)

                X_test = np.concatenate(X_test_list, axis=0)
                y_test = np.concatenate(y_test_list, axis=0)

                if not has_both_classes(y_test):
                    continue

                p_base = clip_probs(base_model.predict_proba(X_test)[:, 1])
                m_base = evaluate_probs(y_test, p_base)

                p_platt = apply_platt_calibrator(p_base, platt)
                m_platt = evaluate_probs(y_test, p_platt)

                try:
                    X_test_coral = coral_align_target_to_source(
                        X_train, X_test,
                        eps=CORAL_EPS,
                        max_full_features=CORAL_MAX_FULL_FEATURES,
                        diagonal_fallback=USE_DIAGONAL_CORAL_FALLBACK
                    )
                    p_coral = clip_probs(base_model.predict_proba(X_test_coral)[:, 1])
                    m_coral = evaluate_probs(y_test, p_coral)

                    p_coral_platt = apply_platt_calibrator(p_coral, platt)
                    m_coral_platt = evaluate_probs(y_test, p_coral_platt)
                except Exception:
                    m_coral = {
                        "roc_auc": np.nan,
                        "pr_auc": np.nan,
                        "brier": np.nan,
                        "sens_at_90spec": np.nan,
                        "spec_at_90sens": np.nan,
                        "cal_intercept": np.nan,
                        "cal_slope": np.nan
                    }
                    m_coral_platt = {
                        "roc_auc": np.nan,
                        "pr_auc": np.nan,
                        "brier": np.nan,
                        "sens_at_90spec": np.nan,
                        "spec_at_90sens": np.nan,
                        "cal_intercept": np.nan,
                        "cal_slope": np.nan
                    }

                for method_name, metrics in [
                    ("baseline", m_base),
                    ("platt", m_platt),
                    ("coral", m_coral),
                    ("coral_platt", m_coral_platt)
                ]:
                    rows.append({
                        "group": group_name,
                        "family": family_label,
                        "train_combo": "||".join(train_combo),
                        "test_combo": "||".join(test_combo),
                        "n_train_studies": a,
                        "n_test_studies": b,
                        "n_train_samples": int(X_train.shape[0]),
                        "n_test_samples": int(X_test.shape[0]),
                        "n_feat": n_feat,
                        "method": method_name,
                        "best_C": float(best_C),
                        "roc_auc": metrics["roc_auc"],
                        "pr_auc": metrics["pr_auc"],
                        "brier": metrics["brier"],
                        "sens_at_90spec": metrics["sens_at_90spec"],
                        "spec_at_90sens": metrics["spec_at_90sens"],
                        "cal_intercept": metrics["cal_intercept"],
                        "cal_slope": metrics["cal_slope"],
                    })

    return pd.DataFrame(rows)

novelty_all = []
for g in NOVELTY_GROUPS:
    if g not in group_studies or len(group_studies[g]) < 3:
        continue
    nov_df = run_novelty_experiments_for_group(g, families=NOVELTY_FAMILIES)
    if len(nov_df):
        nov_df.to_csv(os.path.join(AUG_TAB_DIR, f"novelty_recalibration_domainadapt_{g}.csv"), index=False)
        novelty_all.append(nov_df)

novelty_all = pd.concat(novelty_all, ignore_index=True) if len(novelty_all) else pd.DataFrame()

if len(novelty_all):
    novelty_summary = (
        novelty_all.groupby(["group", "family", "method"], as_index=False)
        .agg(
            n_pairs=("roc_auc", "size"),
            roc_auc_mean=("roc_auc", "mean"),
            roc_auc_median=("roc_auc", "median"),
            pr_auc_mean=("pr_auc", "mean"),
            brier_mean=("brier", "mean"),
            cal_slope_mean=("cal_slope", "mean"),
        )
        .sort_values(["group", "family", "method"])
    )
    novelty_summary.to_csv(os.path.join(AUG_TAB_DIR, "novelty_recalibration_domainadapt_summary.csv"), index=False)

    base = novelty_summary[novelty_summary["method"] == "baseline"].copy()
    merged = novelty_summary.merge(
        base[["group", "family", "roc_auc_mean", "brier_mean"]].rename(
            columns={"roc_auc_mean": "baseline_roc_auc_mean", "brier_mean": "baseline_brier_mean"}
        ),
        on=["group", "family"],
        how="left"
    )
    merged["delta_roc_auc_vs_baseline"] = merged["roc_auc_mean"] - merged["baseline_roc_auc_mean"]
    merged["delta_brier_vs_baseline"] = merged["brier_mean"] - merged["baseline_brier_mean"]
    merged.to_csv(os.path.join(AUG_TAB_DIR, "novelty_recalibration_domainadapt_gain_vs_baseline.csv"), index=False)

    print("\nSaved novelty-layer recalibration/domain-adaptation results.")
    display(novelty_summary.head(20))
    display(merged.head(20))

    for g in sorted(novelty_all["group"].unique()):
        dg = merged[merged["group"] == g].copy()
        plt.figure(figsize=(8.5, 5.5))
        sns.barplot(data=dg, x="family", y="delta_roc_auc_vs_baseline", hue="method")
        plt.axhline(0, ls="--", lw=1)
        plt.title(f"{g}: novelty layer gain in ROC-AUC vs baseline")
        plt.xlabel("Transfer family")
        plt.ylabel("Δ ROC-AUC vs baseline")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
        save_fig(os.path.join(AUG_FIG_DIR, f"{g}_novelty_gain_vs_baseline"))
else:
    novelty_summary = pd.DataFrame()
    merged = pd.DataFrame()
    print("\nNo novelty-layer results produced.")

# ------------------------------------------------------------
# 7) CONCISE METHODOLOGICAL FRAMING + REPORT WRITER
# ------------------------------------------------------------
def build_method_framing_text():
    return f"""
# CRC Transportability Analysis: manuscript framing

## Core methodological framing
We evaluated cross-cohort transportability of CRC classifiers across multiple microbiome-derived representation types using a predefined family of disjoint source-target transfer settings. Rather than exhaustively enumerating all possible source-target subsets, we restricted the design to publication-oriented, interpretable transfer families: 1→1, 1→2, 1→3, 2→1, 3→1, and 2→2. This preserves ecological realism while avoiding an intractable combinatorial expansion.

## Why this framing is stronger
1. It separates single-source transfer from multi-source aggregation effects.
2. It isolates whether gains arise from increased source diversity rather than only increased sample size.
3. It allows direct comparison of representation robustness under comparable source-target complexity.
4. It enables a formal link between distributional shift and external discrimination.
5. It supports a translational message: which representation is most portable under realistic multi-cohort deployment.

## Formal statistical augmentation
Family-specific ROC-AUC distributions were compared using:
- Kruskal-Wallis tests across transfer families within each representation
- pairwise Mann-Whitney U tests with BH-adjusted p-values
- effect-size estimation using Cliff's delta
- bootstrap confidence intervals for family medians and means

## Stronger robustness layer
We quantified shift-performance coupling using:
- family-specific and overall Spearman/Pearson correlations
- family-adjusted OLS models with shift distance as the primary covariate
- comparison of LOSO and publication-transfer shift-performance trends

## Biological interpretation layer
Stable features were summarized from LOSO coefficient stability outputs and converted into more interpretable tables:
- taxa-oriented summaries for species
- pathway-oriented summaries for pathway representations
- top positive and negative stable features for marker representations

## Novelty layer
To move beyond benchmarking alone, we added two lightweight transportability-improving strategies:
1. source-side cross-fitted Platt recalibration
2. unsupervised CORAL-based target alignment

These provide an initial domain-adaptation benchmark and strengthen the paper beyond descriptive external validation.

## Composite transportability score
A composite score was constructed from:
- single-source external discrimination
- multi-source external discrimination
- multi-source gain
- calibration quality
- shift robustness
- signature stability

This score is not a replacement for the raw metrics; rather, it provides a concise ranking framework for practical representation selection.
""".strip()

method_framing_text = build_method_framing_text()
with open(os.path.join(AUG_TXT_DIR, "methodological_framing.md"), "w") as f:
    f.write(method_framing_text)

summary_lines = []
summary_lines.append("# IF10+ augmentation summary")
summary_lines.append("")
summary_lines.append("## Available representations")
summary_lines.append(", ".join(available_groups))
summary_lines.append("")

if len(score_df):
    top_repr = score_df.iloc[0]["group"]
    summary_lines.append("## Composite transportability ranking")
    for _, r in score_df.iterrows():
        summary_lines.append(
            f"- {r['group']}: score={r['transportability_score']:.3f}, "
            f"1->1 median AUC={r['1->1_median_auc']:.3f}, "
            f"3->1 median AUC={r['3->1_median_auc']:.3f}, "
            f"gain={r['gain_3to1_vs_1to1']:.3f}"
        )
    summary_lines.append("")
    summary_lines.append(f"Top-ranked representation by composite score: {top_repr}")
    summary_lines.append("")

if len(family_kw_all):
    summary_lines.append("## Global family effects")
    for _, r in family_kw_all.iterrows():
        summary_lines.append(
            f"- {r['group']}: Kruskal-Wallis p={r['p_value']:.3g}"
        )
    summary_lines.append("")

if len(shift_summary_all):
    summary_lines.append("## Shift-performance coupling")
    for g in available_groups:
        d = shift_summary_all[(shift_summary_all["group"] == g) & (shift_summary_all["scope"] == "publication_all")]
        if len(d):
            rr = d.iloc[0]
            summary_lines.append(
                f"- {g}: Spearman r={rr['spearman_r']:.3f}, p={rr['spearman_p']:.3g}"
            )
    summary_lines.append("")

if len(novelty_summary):
    summary_lines.append("## Novelty layer")
    gain_df = merged[merged["method"].isin(["platt", "coral", "coral_platt"])].copy()
    if len(gain_df):
        best_gain = gain_df.sort_values("delta_roc_auc_vs_baseline", ascending=False).iloc[0]
        summary_lines.append(
            f"Best observed mean ROC-AUC gain vs baseline: "
            f"{best_gain['group']} {best_gain['family']} {best_gain['method']} "
            f"(Δ={best_gain['delta_roc_auc_vs_baseline']:.4f})"
        )
    summary_lines.append("")

summary_lines.append("## Files written")
summary_lines.append(f"- Tables: {AUG_TAB_DIR}")
summary_lines.append(f"- Figures: {AUG_FIG_DIR}")
summary_lines.append(f"- Reports: {AUG_TXT_DIR}")

summary_text = "\n".join(summary_lines)
with open(os.path.join(AUG_TXT_DIR, "augmentation_summary.md"), "w") as f:
    f.write(summary_text)

print("\n" + "=" * 90)
print("AUGMENTATION COMPLETE")
print("=" * 90)
print(summary_text)
print("\nMain augmented outputs:")
print("Tables :", AUG_TAB_DIR)
print("Figures:", AUG_FIG_DIR)
print("Reports:", AUG_TXT_DIR)

Available groups: ['species', 'pathways', 'markers_abundance', 'markers_presence']

Saved family-level inferential statistics.


,group,family,n,roc_auc_mean,roc_auc_median,roc_auc_sd,roc_auc_mean_ci_low,roc_auc_mean_ci_high,roc_auc_median_ci_low,roc_auc_median_ci_high
0,species,1->1,72,0.628838,0.658678,0.136897,0.596849,0.659286,0.617979,0.678760
1,species,1->2,252,0.628000,0.644026,0.108510,0.614435,0.642310,0.638004,0.661306
2,species,1->3,504,0.626404,0.645152,0.099834,0.617888,0.634924,0.641243,0.651319
3,species,2->1,252,0.675464,0.683862,0.100093,0.662864,0.688020,0.669107,0.695402
4,species,3->1,504,0.690517,0.698955,0.098233,0.681591,0.699228,0.690405,0.707759
5,species,2->2,756,0.675852,0.671417,0.065336,0.671452,0.680602,0.666083,0.677632
6,pathways,1->1,110,0.609744,0.596722,0.096848,0.591737,0.629207,0.574808,0.622500
7,pathways,1->2,495,0.605232,0.597840,0.072559,0.598584,0.611746,0.589569,0.606106
8,pathways,1->3,1320,0.603378,0.598044,0.064993,0.599797,0.606691,0.594576,0.602589
9,pathways,2->1,495,0.637990,0.635424,0.093838,0.630046,0.645909,0.626250,0.645145


,group,test,n_families,statistic,p_value
0,species,Kruskal-Wallis,6,147.300791,5.011651e-30
1,pathways,Kruskal-Wallis,6,391.617191,1.900385e-82
2,markers_abundance,Kruskal-Wallis,6,790.699217,1.189403e-168
3,markers_presence,Kruskal-Wallis,6,554.170739,1.606890e-117


,group,family_1,family_2,n1,n2,median_1,median_2,mean_1,mean_2,delta_median,delta_mean,cliffs_delta,mw_statistic,p_value,p_adj_bh
0,species,1->1,1->2,72,252,0.658678,0.644026,0.628838,0.628000,0.014652,0.000839,0.033289,9374.0,6.671226e-01,7.147742e-01
1,species,1->1,1->3,72,504,0.658678,0.645152,0.628838,0.626404,0.013527,0.002434,0.050705,19064.0,4.863649e-01,5.611903e-01
2,species,1->1,2->1,72,252,0.658678,0.683862,0.628838,0.675464,-0.025184,-0.046625,-0.183532,7407.0,1.757414e-02,2.636120e-02
3,species,1->1,3->1,72,504,0.658678,0.698955,0.628838,0.690517,-0.040277,-0.061679,-0.266810,13303.0,2.478475e-04,4.647141e-04
4,species,1->1,2->2,72,756,0.658678,0.671417,0.628838,0.675852,-0.012739,-0.047014,-0.174475,22467.5,1.434596e-02,2.390993e-02
5,species,1->2,1->3,252,504,0.644026,0.645152,0.628000,0.626404,-0.001126,0.001596,0.014220,64407.0,7.498470e-01,7.498470e-01
6,species,1->2,2->1,252,252,0.644026,0.683862,0.628000,0.675464,-0.039836,-0.047464,-0.243512,24020.0,2.251577e-06,5.628942e-06
7,species,1->2,3->1,252,504,0.644026,0.698955,0.628000,0.690517,-0.054929,-0.062518,-0.340018,41911.5,2.381181e-14,1.190590e-13
8,species,1->2,2->2,252,756,0.644026,0.671417,0.628000,0.675852,-0.027391,-0.047852,-0.242840,72124.0,7.493830e-09,2.248149e-08
9,species,1->3,2->1,504,252,0.645152,0.683862,0.626404,0.675464,-0.038710,-0.049060,-0.270818,46306.0,1.234893e-09,4.630847e-09



Saved stronger shift-performance analyses.


...

Novelty markers_presence 3->1:   0%|          | 0/165 [00:00<?, ?it/s]


Saved novelty-layer recalibration/domain-adaptation results.


,group,family,method,n_pairs,roc_auc_mean,roc_auc_median,pr_auc_mean,brier_mean,cal_slope_mean
0,markers_abundance,1->1,baseline,110,0.596836,0.582810,0.616910,0.332314,0.280176
1,markers_abundance,1->1,coral,110,0.600952,0.591612,0.620457,0.296732,0.271450
2,markers_abundance,1->1,coral_platt,110,0.600952,0.591612,0.620457,0.261067,0.644441
3,markers_abundance,1->1,platt,110,0.596838,0.582810,0.616910,0.287403,0.669988
4,markers_abundance,2->1,baseline,495,0.650864,0.641282,0.673782,0.290115,0.533305
5,markers_abundance,2->1,coral,495,0.653948,0.644231,0.677542,0.268381,0.509689
6,markers_abundance,2->1,coral_platt,495,0.653948,0.644231,0.677542,0.240929,0.850608
7,markers_abundance,2->1,platt,495,0.650863,0.641282,0.673769,0.258824,0.888181
8,markers_abundance,3->1,baseline,1320,0.681693,0.671478,0.707098,0.262739,0.728559
9,markers_abundance,3->1,coral,1320,0.683758,0.672309,0.710119,0.247391,0.707967


,group,family,method,n_pairs,roc_auc_mean,roc_auc_median,pr_auc_mean,brier_mean,cal_slope_mean,baseline_roc_auc_mean,baseline_brier_mean,delta_roc_auc_vs_baseline,delta_brier_vs_baseline
0,markers_abundance,1->1,baseline,110,0.596836,0.582810,0.616910,0.332314,0.280176,0.596836,0.332314,0.000000e+00,0.000000
1,markers_abundance,1->1,coral,110,0.600952,0.591612,0.620457,0.296732,0.271450,0.596836,0.332314,4.116149e-03,-0.035582
2,markers_abundance,1->1,coral_platt,110,0.600952,0.591612,0.620457,0.261067,0.644441,0.596836,0.332314,4.116149e-03,-0.071247
3,markers_abundance,1->1,platt,110,0.596838,0.582810,0.616910,0.287403,0.669988,0.596836,0.332314,1.681011e-06,-0.044911
4,markers_abundance,2->1,baseline,495,0.650864,0.641282,0.673782,0.290115,0.533305,0.650864,0.290115,0.000000e+00,0.000000
5,markers_abundance,2->1,coral,495,0.653948,0.644231,0.677542,0.268381,0.509689,0.650864,0.290115,3.084396e-03,-0.021734
6,markers_abundance,2->1,coral_platt,495,0.653948,0.644231,0.677542,0.240929,0.850608,0.650864,0.290115,3.084396e-03,-0.049186
7,markers_abundance,2->1,platt,495,0.650863,0.641282,0.673769,0.258824,0.888181,0.650864,0.290115,-2.527780e-07,-0.031291
8,markers_abundance,3->1,baseline,1320,0.681693,0.671478,0.707098,0.262739,0.728559,0.681693,0.262739,0.000000e+00,0.000000
9,markers_abundance,3->1,coral,1320,0.683758,0.672309,0.710119,0.247391,0.707967,0.681693,0.262739,2.064181e-03,-0.015348



AUGMENTATION COMPLETE
# IF10+ augmentation summary

## Available representations
species, pathways, markers_abundance, markers_presence

## Composite transportability ranking
- markers_presence: score=0.727, 1->1 median AUC=0.658, 3->1 median AUC=0.737, gain=0.079
- species: score=0.616, 1->1 median AUC=0.659, 3->1 median AUC=0.699, gain=0.040
- pathways: score=0.343, 1->1 median AUC=0.597, 3->1 median AUC=0.657, gain=0.060
- markers_abundance: score=0.277, 1->1 median AUC=0.583, 3->1 median AUC=0.671, gain=0.089

Top-ranked representation by composite score: markers_presence

## Global family effects
- species: Kruskal-Wallis p=5.01e-30
- pathways: Kruskal-Wallis p=1.9e-82
- markers_abundance: Kruskal-Wallis p=1.19e-168
- markers_presence: Kruskal-Wallis p=1.61e-117

## Shift-performance coupling
- species: Spearman r=-0.385, p=2.16e-83
- pathways: Spearman r=-0.269, p=2.77e-95
- markers_abundance: Spearman r=-0.208, p=6.04e-57
- markers_presence: Spearman r=-0.276, p=3.39e-100

## N

##4. figure block

In [ ]:
!pip -q install adjustText

import os
import re
import math
import warnings
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist
from scipy.stats import spearmanr, pearsonr

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from adjustText import adjust_text

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_DIR = "/content/curatedMetagenomicData_EH"
OUT_DIR = os.path.join(DATA_DIR, "analysis_outputs_crc")
FIG_BASE = os.path.join(OUT_DIR, "ncomms_style_figures")
os.makedirs(FIG_BASE, exist_ok=True)

REPRESENTATIONS = ["species", "pathways", "markers_abundance", "markers_presence"]
PUBLICATION_FAMILIES = ["1->1", "1->2", "1->3", "2->1", "2->2", "3->1"]

sns.set_theme(style="white", context="paper")
mpl.rcParams["figure.dpi"] = 160
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["axes.labelsize"] = 10
mpl.rcParams["xtick.labelsize"] = 9
mpl.rcParams["ytick.labelsize"] = 9
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False

# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------
def fp_pub(group):
    return os.path.join(OUT_DIR, f"publication_transfer_auc_{group}.csv")

def fp_pub_summary(group):
    return os.path.join(OUT_DIR, f"publication_transfer_summary_{group}.csv")

def fp_shift_pub(group):
    return os.path.join(OUT_DIR, f"shift_vs_auc_publication_{group}.csv")

def fp_loso(group):
    return os.path.join(OUT_DIR, f"loso_{group}_metrics_next.csv")

def fp_loso_shift(group):
    return os.path.join(OUT_DIR, f"shift_vs_auc_loso_{group}.csv")

def fp_stability(group):
    return os.path.join(OUT_DIR, f"{group}_stability_summary.csv")

def fp_signature(group):
    return os.path.join(OUT_DIR, f"{group}_signature_stable.csv")

def safe_read_csv(path):
    if not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Could not read {path}: {e}")
        return None

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------
pub = {g: safe_read_csv(fp_pub(g)) for g in REPRESENTATIONS}
pub_summary = {g: safe_read_csv(fp_pub_summary(g)) for g in REPRESENTATIONS}
shift_pub = {g: safe_read_csv(fp_shift_pub(g)) for g in REPRESENTATIONS}
loso = {g: safe_read_csv(fp_loso(g)) for g in REPRESENTATIONS}
loso_shift = {g: safe_read_csv(fp_loso_shift(g)) for g in REPRESENTATIONS}
stability = {g: safe_read_csv(fp_stability(g)) for g in REPRESENTATIONS}
signature = {g: safe_read_csv(fp_signature(g)) for g in REPRESENTATIONS}

available_groups = [g for g in REPRESENTATIONS if pub[g] is not None and len(pub[g]) > 0]
print("Available groups:", available_groups)

# ------------------------------------------------------------
# GENERIC HELPERS
# ------------------------------------------------------------
def savefig(name):
    fig = plt.gcf()
    try:
        engine = fig.get_layout_engine()
    except Exception:
        engine = None

    if engine is None:
        try:
            fig.tight_layout()
        except Exception:
            pass

    fig.savefig(os.path.join(FIG_BASE, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG_BASE, f"{name}.pdf"), bbox_inches="tight")
    plt.close(fig)

def family_order_present(df):
    if df is None or "family" not in df.columns:
        return []
    return [f for f in PUBLICATION_FAMILIES if f in set(df["family"].astype(str))]

def parse_tax_label(feature):
    txt = str(feature)
    g = re.search(r"g__([^|]+)", txt)
    s = re.search(r"s__([^|]+)", txt)
    genus = g.group(1) if g else None
    species = s.group(1) if s else None
    if genus and species and species not in ["", "_", "s__"]:
        return f"{genus} {species}"
    if genus:
        return genus
    return txt.split("::")[-1][:80]

def parse_pathway_label(feature):
    txt = str(feature).split("::", 1)[-1]
    if ":" in txt:
        pid, desc = txt.split(":", 1)
        return desc.strip()
    return txt[:80]

def short_label(x, max_len=45):
    x = str(x)
    return x if len(x) <= max_len else x[:max_len - 3] + "..."

def clustered_order_from_matrix(mat):
    if mat is None or mat.shape[0] == 0 or mat.shape[1] == 0:
        return [], []
    if mat.shape[0] <= 2 or mat.shape[1] <= 2:
        return list(mat.index), list(mat.columns)
    try:
        filled = mat.copy()
        fill_value = np.nanmean(filled.values)
        if np.isnan(fill_value):
            fill_value = 0.0
        filled = filled.fillna(fill_value)

        row_link = linkage(pdist(filled.values), method="average")
        col_link = linkage(pdist(filled.T.values), method="average")
        row_order = filled.index[leaves_list(row_link)].tolist()
        col_order = filled.columns[leaves_list(col_link)].tolist()
        return row_order, col_order
    except Exception:
        return list(mat.index), list(mat.columns)

def build_pairwise_matrix_from_1to1(pub_df):
    if pub_df is None or len(pub_df) == 0:
        return None

    needed = {"family", "train_combo", "test_combo", "roc_auc"}
    if not needed.issubset(pub_df.columns):
        return None

    d = pub_df.copy()
    d = d[d["family"].astype(str) == "1->1"].copy()
    if len(d) == 0:
        return None

    d["train_label"] = d["train_combo"].astype(str)
    d["test_label"] = d["test_combo"].astype(str)
    mat = d.pivot_table(index="train_label", columns="test_label", values="roc_auc", aggfunc="mean")
    return mat

def top_features_for_group(group, n_each=12):
    stab = stability.get(group, None)
    if stab is None or len(stab) == 0:
        return pd.DataFrame()

    needed = {"feature", "mean_coef"}
    if not needed.issubset(stab.columns):
        return pd.DataFrame()

    d = stab.copy()
    if "stable" in d.columns:
        d = d[d["stable"] == True].copy()
    if len(d) == 0:
        return pd.DataFrame()

    top_pos = d.sort_values("mean_coef", ascending=False).head(n_each).copy()
    top_neg = d.sort_values("mean_coef", ascending=True).head(n_each).copy()
    out = pd.concat([top_pos, top_neg], axis=0).drop_duplicates(subset=["feature"]).copy()

    if group == "species":
        out["label"] = out["feature"].apply(parse_tax_label)
    elif group == "pathways":
        out["label"] = out["feature"].apply(parse_pathway_label)
    else:
        out["label"] = out["feature"].astype(str).str.split("::").str[-1].apply(short_label)

    return out

# ------------------------------------------------------------
# FIGURE 1
# ------------------------------------------------------------
overview_rows = []
for g in available_groups:
    if pub_summary[g] is None or len(pub_summary[g]) == 0:
        continue
    if not {"family", "roc_auc_mean", "roc_auc_median", "roc_auc_sd"}.issubset(pub_summary[g].columns):
        continue

    dd = pub_summary[g].copy()
    for fam in PUBLICATION_FAMILIES:
        sub = dd[dd["family"].astype(str) == fam]
        if len(sub):
            overview_rows.append({
                "representation": g,
                "family": fam,
                "mean_auc": float(sub["roc_auc_mean"].iloc[0]),
                "median_auc": float(sub["roc_auc_median"].iloc[0]),
                "sd_auc": float(sub["roc_auc_sd"].iloc[0]),
            })

overview_df = pd.DataFrame(overview_rows)

if len(overview_df):
    mat = overview_df.pivot(index="representation", columns="family", values="median_auc")
    mat = mat.reindex(index=available_groups, columns=[f for f in PUBLICATION_FAMILIES if f in mat.columns])

    plt.figure(figsize=(8.4, 3.8))
    sns.heatmap(
        mat, annot=True, fmt=".3f", cmap="mako",
        vmin=0.50, vmax=max(0.80, np.nanmax(mat.values)),
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": "Median ROC-AUC"}
    )
    plt.title("Fig. 1 | Cross-cohort transportability across representations and transfer families")
    plt.xlabel("Transfer family")
    plt.ylabel("")
    savefig("Fig1_overview_performance_heatmap")

# ------------------------------------------------------------
# FIGURE 2
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, g in zip(axes, available_groups):
    d = pub[g]
    if d is None or len(d) == 0 or "roc_auc" not in d.columns or "family" not in d.columns:
        ax.axis("off")
        continue

    d = d.dropna(subset=["roc_auc"]).copy()
    order = family_order_present(d)
    if len(d) == 0 or not order:
        ax.axis("off")
        continue

    sns.boxplot(data=d, x="family", y="roc_auc", order=order, ax=ax, color="#c7dcef", fliersize=1.8, linewidth=1)

    n_samp = min(len(d), 1200)
    d_plot = d.sample(n=n_samp, random_state=7) if len(d) > n_samp else d
    sns.stripplot(
        data=d_plot, x="family", y="roc_auc",
        order=order, ax=ax, color="black",
        alpha=0.18, size=1.6, jitter=0.22
    )
    ax.axhline(0.5, ls="--", lw=1, color="gray")
    ax.set_title(g)
    ax.set_xlabel("")
    ax.set_ylabel("ROC-AUC")
    ax.tick_params(axis="x", rotation=20)

for j in range(len(available_groups), len(axes)):
    axes[j].axis("off")

fig.suptitle("Fig. 2 | Distribution of external transport performance across predefined transfer families", y=1.02)
savefig("Fig2_transfer_family_distributions")

# ------------------------------------------------------------
# FIGURE 3
# ------------------------------------------------------------
for g in available_groups:
    mat = build_pairwise_matrix_from_1to1(pub[g])
    if mat is None or mat.shape[0] == 0:
        continue

    row_order, col_order = clustered_order_from_matrix(mat)
    if row_order and col_order:
        mat = mat.loc[row_order, col_order]

    plt.figure(figsize=(8.5, 7.2))
    sns.heatmap(
        mat,
        cmap="viridis",
        vmin=0.4,
        vmax=max(0.85, np.nanmax(mat.values)),
        linewidths=0.25,
        linecolor="white",
        cbar_kws={"label": "ROC-AUC"}
    )
    plt.title(f"Fig. 3 | {g}: pairwise single-source to single-target transfer heatmap")
    plt.xlabel("Test cohort")
    plt.ylabel("Train cohort")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    savefig(f"Fig3_{g}_pairwise_1to1_heatmap")

# ------------------------------------------------------------
# FIGURE 4
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

legend_handles = None
legend_labels = None

for ax, g in zip(axes, available_groups):
    d = shift_pub[g]
    if d is None or len(d) == 0 or not {"shift_distance", "roc_auc", "family"}.issubset(d.columns):
        ax.axis("off")
        continue

    d = d.dropna(subset=["shift_distance", "roc_auc"]).copy()
    sns.scatterplot(
        data=d, x="shift_distance", y="roc_auc",
        hue="family", palette="tab10", alpha=0.55,
        s=16, linewidth=0, ax=ax
    )
    sns.regplot(
        data=d, x="shift_distance", y="roc_auc",
        scatter=False, truncate=False,
        line_kws={"lw": 2, "color": "black"},
        ax=ax
    )

    if len(d) >= 3:
        sr, sp = spearmanr(d["shift_distance"], d["roc_auc"])
        ax.text(
            0.03, 0.03,
            f"Spearman r = {sr:.2f}\np = {sp:.2g}",
            transform=ax.transAxes,
            fontsize=9,
            va="bottom",
            ha="left",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.8", alpha=0.85)
        )

    ax.set_title(g)
    ax.set_xlabel("Shift distance")
    ax.set_ylabel("ROC-AUC")

    h, l = ax.get_legend_handles_labels()
    if h and l:
        legend_handles, legend_labels = h, l

for j in range(len(available_groups), len(axes)):
    axes[j].axis("off")

for ax in axes:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

if legend_handles is not None and legend_labels is not None:
    fig.legend(legend_handles, legend_labels, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

fig.suptitle("Fig. 4 | Distributional shift is associated with variation in external transportability", y=1.02)
savefig("Fig4_shift_vs_performance")

# ------------------------------------------------------------
# FIGURE 5
# ------------------------------------------------------------
def build_signature_matrix(group, n_top_each=10):
    if "group_study_data" not in globals() or "group_studies" not in globals():
        return None, None

    if group not in group_study_data or group not in group_studies:
        return None, None

    feat_df = top_features_for_group(group, n_each=n_top_each)
    if feat_df is None or len(feat_df) == 0:
        return None, None

    feat_list = feat_df["feature"].tolist()
    label_map = dict(zip(feat_df["feature"], feat_df["label"]))

    study_dict = group_study_data[group]
    studies = group_studies[group]

    rows = []
    for st in studies:
        if "X_raw" not in study_dict[st] or "y" not in study_dict[st]:
            continue

        X_raw = study_dict[st]["X_raw"]
        common = [f for f in feat_list if f in X_raw.columns]
        if len(common) == 0:
            continue

        X = X_raw.reindex(columns=feat_list, fill_value=0.0).copy()
        y = np.asarray(study_dict[st]["y"]).astype(int)

        for cls, cls_name in [(0, "control"), (1, "crc")]:
            idx = np.where(y == cls)[0]
            if len(idx) == 0:
                continue
            mu = X.iloc[idx].mean(axis=0)
            row = {"study_class": f"{st} | {cls_name}"}
            for f in feat_list:
                row[label_map[f]] = float(mu[f])
            rows.append(row)

    if len(rows) == 0:
        return None, None

    df = pd.DataFrame(rows).set_index("study_class")
    z = (df - df.mean(axis=0)) / (df.std(axis=0) + 1e-8)
    return z, feat_df

for g in REPRESENTATIONS:
    sig_mat, feat_meta = build_signature_matrix(g, n_top_each=8)
    if sig_mat is None or sig_mat.shape[0] == 0:
        continue

    row_order, col_order = clustered_order_from_matrix(sig_mat)
    if row_order and col_order:
        sig_mat = sig_mat.loc[row_order, col_order]

    plt.figure(figsize=(10.5, max(5, 0.28 * sig_mat.shape[0])))
    sns.heatmap(
        sig_mat, cmap="coolwarm", center=0,
        linewidths=0.2, linecolor="white",
        cbar_kws={"label": "Feature z-score"}
    )
    plt.title(f"Fig. 5 | {g}: stable discriminatory signature across cohorts")
    plt.xlabel("Stable features")
    plt.ylabel("")
    plt.xticks(rotation=45, ha="right")
    savefig(f"Fig5_{g}_stable_signature_heatmap")

# ------------------------------------------------------------
# FIGURE 6
# ------------------------------------------------------------
gain_rows = []
for g in available_groups:
    if pub_summary[g] is None or len(pub_summary[g]) == 0:
        continue
    if not {"family", "roc_auc_median"}.issubset(pub_summary[g].columns):
        continue

    d = pub_summary[g].copy().set_index("family")
    if "1->1" in d.index:
        baseline = float(d.loc["1->1", "roc_auc_median"])
        for fam in d.index:
            gain_rows.append({
                "representation": g,
                "family": fam,
                "delta_vs_1to1": float(d.loc[fam, "roc_auc_median"] - baseline)
            })

gain_df = pd.DataFrame(gain_rows)
if len(gain_df):
    mat = gain_df.pivot(index="representation", columns="family", values="delta_vs_1to1")
    mat = mat.reindex(index=available_groups, columns=[f for f in PUBLICATION_FAMILIES if f in mat.columns])

    plt.figure(figsize=(8.2, 3.8))
    sns.heatmap(
        mat, annot=True, fmt=".3f",
        cmap="RdBu_r", center=0,
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": "Δ median ROC-AUC vs 1→1"}
    )
    plt.title("Fig. 6 | Multi-source aggregation generally improves external transportability")
    plt.xlabel("Transfer family")
    plt.ylabel("")
    savefig("Fig6_multisource_gain_heatmap")

# ------------------------------------------------------------
# FIGURE 7
# ------------------------------------------------------------
summary_rows = []
for g in available_groups:
    row = {"representation": g}

    if loso[g] is not None and len(loso[g]) and "roc_auc" in loso[g].columns:
        row["LOSO mean"] = float(loso[g]["roc_auc"].mean())
        row["LOSO median"] = float(loso[g]["roc_auc"].median())
    else:
        row["LOSO mean"] = np.nan
        row["LOSO median"] = np.nan

    if pub_summary[g] is not None and len(pub_summary[g]) and {"family", "roc_auc_median"}.issubset(pub_summary[g].columns):
        d = pub_summary[g].set_index("family")
        for fam in ["1->1", "2->1", "3->1", "2->2"]:
            row[fam] = float(d.loc[fam, "roc_auc_median"]) if fam in d.index else np.nan

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
plot_df = summary_df.melt(id_vars="representation", var_name="setting", value_name="auc")

plt.figure(figsize=(9, 5.2))
sns.barplot(data=plot_df, x="representation", y="auc", hue="setting")
plt.axhline(0.5, ls="--", lw=1, color="gray")
plt.title("Fig. 7 | Comparison of LOSO performance and publication-oriented transfer settings")
plt.xlabel("")
plt.ylabel("ROC-AUC")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
savefig("Fig7_loso_vs_transfer_summary")

# ------------------------------------------------------------
# FIGURE 8
# ------------------------------------------------------------
loso_rows = []
for g in available_groups:
    d = loso[g]
    if d is None or len(d) == 0:
        continue
    if not {"heldout", "roc_auc"}.issubset(d.columns):
        continue

    for _, r in d.iterrows():
        loso_rows.append({
            "representation": g,
            "heldout": str(r["heldout"]),
            "roc_auc": float(r["roc_auc"])
        })

loso_long = pd.DataFrame(loso_rows)
if len(loso_long):
    mat = loso_long.pivot(index="representation", columns="heldout", values="roc_auc")
    col_order = mat.mean(axis=0).sort_values(ascending=False).index.tolist()
    mat = mat.reindex(index=available_groups, columns=col_order)

    plt.figure(figsize=(11, 3.8))
    sns.heatmap(
        mat, annot=True, fmt=".3f",
        cmap="YlGnBu",
        vmin=0.5, vmax=max(0.9, np.nanmax(mat.values)),
        linewidths=0.35, linecolor="white",
        cbar_kws={"label": "LOSO ROC-AUC"}
    )
    plt.title("Fig. 8 | Held-out cohort performance varies substantially across representations")
    plt.xlabel("Held-out cohort")
    plt.ylabel("")
    plt.xticks(rotation=45, ha="right")
    savefig("Fig8_loso_heldout_heatmap")

# ------------------------------------------------------------
# FIGURE 9
# ------------------------------------------------------------
coverage_rows = []
for g in available_groups:
    df = pub[g]
    if df is None or len(df) == 0:
        continue

    train_used = set()
    test_used = set()

    if "train_studies" in df.columns:
        for x in df["train_studies"].dropna().astype(str):
            train_used.update([s for s in x.split("||") if s.strip()])
    elif "train_combo" in df.columns:
        for x in df["train_combo"].dropna().astype(str):
            train_used.update([s for s in x.split("||") if s.strip()])

    if "test_studies" in df.columns:
        for x in df["test_studies"].dropna().astype(str):
            test_used.update([s for s in x.split("||") if s.strip()])
    elif "test_combo" in df.columns:
        for x in df["test_combo"].dropna().astype(str):
            test_used.update([s for s in x.split("||") if s.strip()])

    all_used = sorted(train_used | test_used)

    for fam in family_order_present(df):
        n_rows = int((df["family"].astype(str) == fam).sum())
        coverage_rows.append({
            "representation": g,
            "family": fam,
            "n_pairs": n_rows,
            "n_cohorts": len(all_used)
        })

coverage_df = pd.DataFrame(coverage_rows)
if len(coverage_df):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))  # removed constrained_layout=True

    sns.barplot(
        data=coverage_df,
        x="representation",
        y="n_cohorts",
        ax=axes[0],
        color="#8ecae6"
    )
    axes[0].set_title("Cohorts represented in transfer analysis")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Number of cohorts")

    cnt_mat = coverage_df.pivot(index="representation", columns="family", values="n_pairs")
    cnt_mat = cnt_mat.reindex(index=available_groups, columns=[f for f in PUBLICATION_FAMILIES if f in cnt_mat.columns])

    sns.heatmap(
        cnt_mat,
        annot=True,
        fmt=".0f",
        cmap="Greys",
        linewidths=0.35,
        linecolor="white",
        ax=axes[1],
        cbar_kws={"label": "Number of evaluated transfers"}
    )
    axes[1].set_title("Evaluated transfer count by family")
    axes[1].set_xlabel("Family")
    axes[1].set_ylabel("")

    fig.suptitle("Fig. 9 | Cohort coverage and evaluation density across representations", y=1.03)
    fig.tight_layout()
    savefig("Fig9_coverage_and_family_count_summary")

# ------------------------------------------------------------
# FIGURE 10
# ------------------------------------------------------------
rank_rows = []
for g in available_groups:
    if pub_summary[g] is None or len(pub_summary[g]) == 0:
        continue
    if not {"family", "roc_auc_median"}.issubset(pub_summary[g].columns):
        continue

    d = pub_summary[g].set_index("family")
    rank_rows.append({
        "representation": g,
        "1->1": float(d.loc["1->1", "roc_auc_median"]) if "1->1" in d.index else np.nan,
        "2->1": float(d.loc["2->1", "roc_auc_median"]) if "2->1" in d.index else np.nan,
        "3->1": float(d.loc["3->1", "roc_auc_median"]) if "3->1" in d.index else np.nan,
        "2->2": float(d.loc["2->2", "roc_auc_median"]) if "2->2" in d.index else np.nan,
    })

rank_df = pd.DataFrame(rank_rows)
if len(rank_df):
    metric_cols = ["1->1", "2->1", "3->1", "2->2"]

    plt.figure(figsize=(8.5, 5.0))
    x = np.arange(len(rank_df))
    width = 0.18

    palette = ["#264653", "#2a9d8f", "#e9c46a", "#e76f51"]
    for i, c in enumerate(metric_cols):
        plt.bar(x + (i - 1.5) * width, rank_df[c], width=width, label=c, color=palette[i])

    plt.xticks(x, rank_df["representation"])
    plt.ylabel("Median ROC-AUC")
    plt.xlabel("")
    plt.title("Fig. 10 | Comparative representation ranking across core deployment scenarios")
    plt.legend(title="Setting", frameon=False)
    plt.axhline(0.5, ls="--", lw=1, color="gray")
    savefig("Fig10_representation_ranked_bar_panel")

# ------------------------------------------------------------
# EXPORT FIGURE INDEX
# ------------------------------------------------------------
fig_index = pd.DataFrame({
    "figure_file": sorted([f for f in os.listdir(FIG_BASE) if f.endswith(".png")]),
})
fig_index.to_csv(os.path.join(FIG_BASE, "figure_index.csv"), index=False)

print("\nDone.")
print("Figures saved to:", FIG_BASE)
print("Generated figure files:")
display(fig_index.head(30))

Available groups: ['species', 'pathways', 'markers_abundance', 'markers_presence']

Done.
Figures saved to: /content/curatedMetagenomicData_EH/analysis_outputs_crc/ncomms_style_figures
Generated figure files:


,figure_file
0,Fig10_representation_ranked_bar_panel.png
1,Fig1_overview_performance_heatmap.png
2,Fig2_transfer_family_distributions.png
3,Fig3_markers_abundance_pairwise_1to1_heatmap.png
4,Fig3_markers_presence_pairwise_1to1_heatmap.png
5,Fig3_pathways_pairwise_1to1_heatmap.png
6,Fig3_species_pairwise_1to1_heatmap.png
7,Fig4_shift_vs_performance.png
8,Fig5_markers_abundance_stable_signature_heatma...
9,Fig5_markers_presence_stable_signature_heatmap...


##5. Recalibration / add-on analysis

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn

import os, json, math, warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# -------------------------
# CONFIG
# -------------------------
OUT_ROOT = "/content/curatedMetagenomicData_EH/analysis_outputs_crc"
FIG_DIR = os.path.join(OUT_ROOT, "figures_addons")
TAB_DIR = os.path.join(OUT_ROOT, "tables_addons")
CACHE_DIR = os.path.join(OUT_ROOT, "matrix_cache")  # optional

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

RNG = np.random.default_rng(42)

# Figures: journal defaults
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 400,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

# Analysis knobs
ECE_BINS = 10
BOOTSTRAP_N = 2000
PERM_N = 2000
DECISION_THRESHOLDS = np.linspace(0.01, 0.99, 99)

FEWSHOT_K = [0, 5, 10, 20, 50, 100]
FEWSHOT_REPEATS = 200

# Signature-only controls (requires matrix cache)
RANDOM_CONTROL_REPEATS = 50

# C2ST (requires matrix cache)
C2ST_FOLDS = 5

# Heatmaps
HEATMAP_ANNOTATE_MAX_N = 25   # only annotate values if <= this many cohorts
HEATMAP_MIN_AUC = 0.45
HEATMAP_MAX_AUC = 0.95

# -------------------------
# Utilities
# -------------------------
def _infer_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def _clip_probs(p, eps=1e-6):
    p = np.asarray(p, dtype=float)
    return np.clip(p, eps, 1.0 - eps)

def logit(p):
    p = _clip_probs(p)
    return np.log(p / (1.0 - p))

def inv_logit(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def compute_ece(y, p, n_bins=10):
    y = np.asarray(y).astype(int)
    p = _clip_probs(p)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    mce = 0.0
    records = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins - 1 else (p >= lo) & (p <= hi)
        if mask.sum() == 0:
            continue
        acc = y[mask].mean()
        conf = p[mask].mean()
        gap = abs(acc - conf)
        w = mask.mean()
        ece += w * gap
        mce = max(mce, gap)
        records.append({"bin_lo": lo, "bin_hi": hi, "n": int(mask.sum()), "acc": acc, "conf": conf, "gap": gap})
    return float(ece), float(mce), pd.DataFrame(records)

def calibration_intercept_slope(y, p):
    """
    Fit: y ~ a + b * logit(p) (logistic regression on single predictor logit(p))
    Returns (a, b). If fails, returns (nan, nan).
    """
    y = np.asarray(y).astype(int)
    x = logit(p).reshape(-1, 1)
    try:
        lr = LogisticRegression(penalty="none", solver="lbfgs", max_iter=2000)
        lr.fit(x, y)
        return float(lr.intercept_[0]), float(lr.coef_[0, 0])
    except Exception:
        try:
            lr = LogisticRegression(penalty="l2", C=1e6, solver="lbfgs", max_iter=2000)
            lr.fit(x, y)
            return float(lr.intercept_[0]), float(lr.coef_[0, 0])
        except Exception:
            return np.nan, np.nan

def net_benefit(y, p, thresholds):
    y = np.asarray(y).astype(int)
    p = _clip_probs(p)
    N = len(y)
    prev = y.mean()
    rows = []
    for t in thresholds:
        pred = (p >= t).astype(int)
        TP = ((pred == 1) & (y == 1)).sum()
        FP = ((pred == 1) & (y == 0)).sum()
        nb_model = TP / N - FP / N * (t / (1.0 - t))
        nb_all = prev - (1.0 - prev) * (t / (1.0 - t))
        rows.append({"threshold": t, "nb_model": nb_model, "nb_all": nb_all, "nb_none": 0.0})
    return pd.DataFrame(rows)

def bootstrap_auc_ci(y, p, n_boot=2000, seed=42):
    y = np.asarray(y).astype(int)
    p = _clip_probs(p)
    if len(np.unique(y)) < 2:
        return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    aucs = []
    n = len(y)
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yy = y[idx]
        pp = p[idx]
        if len(np.unique(yy)) < 2:
            continue
        aucs.append(roc_auc_score(yy, pp))
    if len(aucs) < 50:
        return (float(roc_auc_score(y, p)), np.nan, np.nan)
    aucs = np.array(aucs, dtype=float)
    return (float(np.median(aucs)), float(np.quantile(aucs, 0.025)), float(np.quantile(aucs, 0.975)))

def perm_test_auc(y, p, n_perm=2000, seed=42):
    y = np.asarray(y).astype(int)
    p = _clip_probs(p)
    if len(np.unique(y)) < 2:
        return (np.nan, np.nan, np.nan, np.nan)
    obs = roc_auc_score(y, p)
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_perm):
        yy = rng.permutation(y)
        if len(np.unique(yy)) < 2:
            continue
        aucs.append(roc_auc_score(yy, p))
    aucs = np.array(aucs, dtype=float)
    if len(aucs) == 0:
        return (obs, np.nan, np.nan, np.nan)
    pval = (1.0 + np.sum(aucs >= obs)) / (len(aucs) + 1.0)
    return (float(obs), float(aucs.mean()), float(aucs.std(ddof=1)), float(pval))

def hc3_ols(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    n, p = X.shape
    XtX = X.T @ X
    XtX_inv = np.linalg.pinv(XtX)
    beta = XtX_inv @ (X.T @ y)
    yhat = X @ beta
    resid = (y - yhat).ravel()

    H = (X @ XtX_inv) * X
    h = H.sum(axis=1)
    adj = resid / np.maximum(1e-12, (1.0 - h))
    S = (X.T * (adj**2)) @ X
    V = XtX_inv @ S @ XtX_inv
    se = np.sqrt(np.maximum(1e-18, np.diag(V))).reshape(-1, 1)

    tvals = (beta / se).ravel()
    df = max(1, n - p)
    pvals = 2.0 * stats.t.sf(np.abs(tvals), df=df)

    ss_res = np.sum(resid**2)
    ss_tot = np.sum((y.ravel() - y.mean())**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return beta.ravel(), se.ravel(), tvals, pvals, float(r2)

def savefig(basepath):
    plt.tight_layout()
    plt.savefig(basepath + ".png", bbox_inches="tight")
    plt.savefig(basepath + ".pdf", bbox_inches="tight")
    plt.close()

# -------------------------
# Load core outputs
# -------------------------
def load_required(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

loso_species_metrics = load_required(os.path.join(OUT_ROOT, "loso_species_metrics_next.csv"))
loso_pathways_metrics = load_required(os.path.join(OUT_ROOT, "loso_pathways_metrics_next.csv"))
pred_species = load_required(os.path.join(OUT_ROOT, "loso_species_predictions_next.csv"))
pred_pathways = load_required(os.path.join(OUT_ROOT, "loso_pathways_predictions_next.csv"))

shift_loso_species = load_required(os.path.join(OUT_ROOT, "shift_vs_auc_loso_species.csv"))
shift_loso_pathways = load_required(os.path.join(OUT_ROOT, "shift_vs_auc_loso_pathways.csv"))
pair_auc_species = load_required(os.path.join(OUT_ROOT, "train_test_auc_species.csv"))
pair_auc_pathways = load_required(os.path.join(OUT_ROOT, "train_test_auc_pathways.csv"))
shift_pair_species = load_required(os.path.join(OUT_ROOT, "shift_vs_auc_pairwise_species.csv"))
shift_pair_pathways = load_required(os.path.join(OUT_ROOT, "shift_vs_auc_pairwise_pathways.csv"))

transfer_scores = load_required(os.path.join(OUT_ROOT, "transferability_source_target_scores.csv"))

sig_species = load_required(os.path.join(OUT_ROOT, "species_signature_stable.csv"))
sig_pathways = load_required(os.path.join(OUT_ROOT, "pathways_signature_stable.csv"))

print("Loaded core outputs from:", OUT_ROOT)

# -------------------------
# Infer prediction columns
# -------------------------
def standardize_pred_df(df, group_name):
    heldout_col = _infer_col(df, ["heldout", "study", "cohort", "study_name"])
    y_col = _infer_col(df, ["y_true", "y", "label", "outcome"])
    p_col = _infer_col(df, ["p_hat", "prob", "proba", "pred_prob", "prediction", "p", "y_pred_prob"])

    if heldout_col is None or y_col is None or p_col is None:
        raise ValueError(
            f"[{group_name}] Could not infer columns. "
            f"Found heldout={heldout_col}, y={y_col}, p={p_col}. "
            f"Columns: {list(df.columns)}"
        )

    out = df.copy()
    out = out.rename(columns={heldout_col: "heldout", y_col: "y_true", p_col: "p_hat"})
    out["y_true"] = out["y_true"].astype(int)
    out["p_hat"] = _clip_probs(out["p_hat"].astype(float).values)

    sid = _infer_col(out, ["sample_id", "sampleID", "sample", "run", "subject_id", "participant_id"])
    if sid is not None and sid != "sample_id":
        out = out.rename(columns={sid: "sample_id"})
    return out

pred_species_std = standardize_pred_df(pred_species, "species")
pred_pathways_std = standardize_pred_df(pred_pathways, "pathways")

# -------------------------
# 1) Calibration + ECE/MCE + Decision Curves + Bootstrapped CI + Permutation controls
# -------------------------
def per_cohort_pred_summary(pred_df, group_label):
    rows = []
    ece_bins_long = []
    dec_long = []
    perm_rows = []

    cohorts = sorted(pred_df["heldout"].unique())
    for cohort in cohorts:
        sub = pred_df[pred_df["heldout"] == cohort].copy()
        y = sub["y_true"].values
        p = sub["p_hat"].values
        if len(np.unique(y)) < 2:
            continue

        auc = roc_auc_score(y, p)
        ap = average_precision_score(y, p)
        brier = brier_score_loss(y, p)
        ece, mce, ece_bins = compute_ece(y, p, n_bins=ECE_BINS)
        a, b = calibration_intercept_slope(y, p)
        prev = float(y.mean())

        auc_med, auc_lo, auc_hi = bootstrap_auc_ci(y, p, n_boot=BOOTSTRAP_N, seed=42)
        obs, perm_mean, perm_sd, p_perm = perm_test_auc(y, p, n_perm=PERM_N, seed=123)

        rows.append({
            "group": group_label,
            "heldout": cohort,
            "n": int(len(y)),
            "prevalence": prev,
            "roc_auc": float(auc),
            "pr_auc": float(ap),
            "brier": float(brier),
            "ece": float(ece),
            "mce": float(mce),
            "cal_intercept": float(a) if np.isfinite(a) else np.nan,
            "cal_slope": float(b) if np.isfinite(b) else np.nan,
            "auc_boot_median": auc_med,
            "auc_boot_lo": auc_lo,
            "auc_boot_hi": auc_hi,
        })

        ece_bins["group"] = group_label
        ece_bins["heldout"] = cohort
        ece_bins_long.append(ece_bins)

        dec = net_benefit(y, p, DECISION_THRESHOLDS)
        dec["group"] = group_label
        dec["heldout"] = cohort
        dec_long.append(dec)

        perm_rows.append({
            "group": group_label,
            "heldout": cohort,
            "roc_auc_obs": obs,
            "roc_auc_perm_mean": perm_mean,
            "roc_auc_perm_sd": perm_sd,
            "p_perm_one_sided": p_perm
        })

    summary = pd.DataFrame(rows).sort_values(["group", "roc_auc"], ascending=[True, False])
    ece_bins_long = pd.concat(ece_bins_long, ignore_index=True) if len(ece_bins_long) else pd.DataFrame()
    dec_long = pd.concat(dec_long, ignore_index=True) if len(dec_long) else pd.DataFrame()
    perm_df = pd.DataFrame(perm_rows).sort_values(["group", "p_perm_one_sided"])

    summary.to_csv(os.path.join(TAB_DIR, f"{group_label}_calibration_summary.csv"), index=False)
    perm_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_permutation_auc_controls.csv"), index=False)
    if len(ece_bins_long):
        ece_bins_long.to_csv(os.path.join(TAB_DIR, f"{group_label}_ece_bins_long.csv"), index=False)
    if len(dec_long):
        dec_long.to_csv(os.path.join(TAB_DIR, f"{group_label}_decision_curves_long.csv"), index=False)

    print(f"Saved tables for {group_label} ->", TAB_DIR)
    return summary, ece_bins_long, dec_long, perm_df

species_calib, species_ece_bins, species_dec, species_perm = per_cohort_pred_summary(pred_species_std, "species")
pathways_calib, pathways_ece_bins, pathways_dec, pathways_perm = per_cohort_pred_summary(pred_pathways_std, "pathways")

def plot_auc_ci(calib_df, title, out_base):
    df = calib_df.copy().sort_values("roc_auc", ascending=True)
    plt.figure(figsize=(8, max(3.8, 0.35 * len(df))))
    y = np.arange(len(df))
    plt.errorbar(df["roc_auc"], y,
                 xerr=[df["roc_auc"] - df["auc_boot_lo"], df["auc_boot_hi"] - df["roc_auc"]],
                 fmt="o", capsize=3)
    plt.yticks(y, df["heldout"])
    plt.axvline(0.5, linestyle="--", linewidth=1)
    plt.xlabel("ROC-AUC (bootstrap 95% CI)")
    plt.title(title)
    savefig(os.path.join(FIG_DIR, out_base))

plot_auc_ci(species_calib, "Species LOSO ROC-AUC with bootstrap CI", "auc_ci_species")
plot_auc_ci(pathways_calib, "Pathways LOSO ROC-AUC with bootstrap CI", "auc_ci_pathways")

from sklearn.calibration import calibration_curve

def plot_calibration_small_multiples(pred_df, group_label, out_base, ncols=3):
    cohorts = sorted(pred_df["heldout"].unique())
    n = len(cohorts)
    nrows = int(math.ceil(n / ncols))
    plt.figure(figsize=(4.6 * ncols, 3.6 * nrows))
    for i, cohort in enumerate(cohorts, start=1):
        sub = pred_df[pred_df["heldout"] == cohort].copy()
        y = sub["y_true"].values
        p = sub["p_hat"].values
        if len(np.unique(y)) < 2:
            continue
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax = plt.subplot(nrows, ncols, i)
        ax.plot(mean_pred, frac_pos, marker="o", linewidth=1)
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
        ax.set_title(cohort)
        ax.set_xlabel("Mean predicted risk")
        ax.set_ylabel("Observed risk")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    plt.suptitle(f"{group_label.capitalize()} calibration curves (LOSO predictions)", y=1.02)
    savefig(os.path.join(FIG_DIR, out_base))

plot_calibration_small_multiples(pred_species_std, "species", "calibration_curves_species")
plot_calibration_small_multiples(pred_pathways_std, "pathways", "calibration_curves_pathways")

def plot_decision_curves(dec_long, title, out_base):
    if dec_long is None or len(dec_long) == 0:
        return
    plt.figure(figsize=(8, 5))
    for cohort, sub in dec_long.groupby("heldout"):
        plt.plot(sub["threshold"], sub["nb_model"], linewidth=1, alpha=0.25)
    mean_nb = dec_long.groupby("threshold")["nb_model"].mean().reset_index()
    plt.plot(mean_nb["threshold"], mean_nb["nb_model"], linewidth=2, label="Mean model net benefit")
    pooled = dec_long.groupby("threshold")[["nb_all", "nb_none"]].mean().reset_index()
    plt.plot(pooled["threshold"], pooled["nb_all"], linestyle="--", linewidth=1.5, label="Treat-all (avg)")
    plt.plot(pooled["threshold"], pooled["nb_none"], linestyle=":", linewidth=1.5, label="Treat-none")
    plt.axhline(0, linewidth=1)
    plt.xlabel("Decision threshold")
    plt.ylabel("Net benefit")
    plt.title(title)
    plt.legend(frameon=False)
    savefig(os.path.join(FIG_DIR, out_base))

plot_decision_curves(species_dec, "Species decision curve analysis (LOSO)", "decision_curves_species")
plot_decision_curves(pathways_dec, "Pathways decision curve analysis (LOSO)", "decision_curves_pathways")

# -------------------------
# 2) Few-shot target recalibration (predictions + labels only)
# -------------------------
def stratified_sample_indices(y, k, rng):
    y = np.asarray(y).astype(int)
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    if k == 0:
        return np.array([], dtype=int)
    k_pos = max(1, int(round(k * (len(pos_idx) / len(y)))))
    k_neg = k - k_pos
    k_pos = min(k_pos, len(pos_idx))
    k_neg = min(k_neg, len(neg_idx))
    if k_pos + k_neg < 2:
        return np.array([], dtype=int)
    sel_pos = rng.choice(pos_idx, size=k_pos, replace=False) if k_pos > 0 else np.array([], dtype=int)
    sel_neg = rng.choice(neg_idx, size=k_neg, replace=False) if k_neg > 0 else np.array([], dtype=int)
    sel = np.concatenate([sel_pos, sel_neg])
    rng.shuffle(sel)
    return sel

def fewshot_recalibration(pred_df, group_label):
    rows = []
    cohorts = sorted(pred_df["heldout"].unique())
    for cohort in cohorts:
        sub = pred_df[pred_df["heldout"] == cohort].copy()
        y = sub["y_true"].values
        p = sub["p_hat"].values
        if len(np.unique(y)) < 2:
            continue
        n = len(y)
        for k in FEWSHOT_K:
            if k >= n - 5:
                continue
            for rep in range(FEWSHOT_REPEATS):
                rng = np.random.default_rng(1000 + rep)
                cal_idx = stratified_sample_indices(y, k, rng)
                test_mask = np.ones(n, dtype=bool)
                test_mask[cal_idx] = False

                y_test = y[test_mask]
                p_test = p[test_mask]

                if k == 0 or len(cal_idx) < 2 or len(np.unique(y[cal_idx])) < 2:
                    p_cal = p_test
                else:
                    x_cal = logit(p[cal_idx]).reshape(-1, 1)
                    y_cal = y[cal_idx]
                    x_test = logit(p_test).reshape(-1, 1)
                    lr = LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=2000)
                    lr.fit(x_cal, y_cal)
                    p_cal = _clip_probs(lr.predict_proba(x_test)[:, 1])

                auc = roc_auc_score(y_test, p_test)
                brier_raw = brier_score_loss(y_test, p_test)
                ece_raw, _, _ = compute_ece(y_test, p_test, n_bins=ECE_BINS)

                brier_cal = brier_score_loss(y_test, p_cal)
                ece_cal, _, _ = compute_ece(y_test, p_cal, n_bins=ECE_BINS)

                rows.append({
                    "group": group_label,
                    "heldout": cohort,
                    "k_cal": int(k),
                    "rep": int(rep),
                    "n_total": int(n),
                    "n_test": int(len(y_test)),
                    "roc_auc": float(auc),
                    "brier_raw": float(brier_raw),
                    "ece_raw": float(ece_raw),
                    "brier_cal": float(brier_cal),
                    "ece_cal": float(ece_cal),
                })
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(TAB_DIR, f"fewshot_recalibration_{group_label}.csv"), index=False)
    return out

few_species = fewshot_recalibration(pred_species_std, "species")
few_pathways = fewshot_recalibration(pred_pathways_std, "pathways")

def plot_fewshot(df, metric, title, out_base):
    if df is None or len(df) == 0:
        return
    agg = df.groupby(["k_cal"])[[f"{metric}_raw", f"{metric}_cal"]].agg(["mean", "sem"])
    agg.columns = ["_".join(c) for c in agg.columns]
    agg = agg.reset_index()

    plt.figure(figsize=(7, 4.5))
    plt.errorbar(agg["k_cal"], agg[f"{metric}_raw_mean"], yerr=agg[f"{metric}_raw_sem"],
                 marker="o", capsize=3, label=f"Raw {metric}")
    plt.errorbar(agg["k_cal"], agg[f"{metric}_cal_mean"], yerr=agg[f"{metric}_cal_sem"],
                 marker="o", capsize=3, label=f"Recalibrated {metric}")
    plt.xlabel("Target labeled samples used for recalibration (k)")
    plt.ylabel(metric.upper())
    plt.title(title)
    plt.legend(frameon=False)
    savefig(os.path.join(FIG_DIR, out_base))

plot_fewshot(few_species, "brier", "Species: few-shot target recalibration improves Brier", "fewshot_brier_species")
plot_fewshot(few_species, "ece", "Species: few-shot target recalibration improves ECE", "fewshot_ece_species")
plot_fewshot(few_pathways, "brier", "Pathways: few-shot target recalibration improves Brier", "fewshot_brier_pathways")
plot_fewshot(few_pathways, "ece", "Pathways: few-shot target recalibration improves ECE", "fewshot_ece_pathways")

print("Saved few-shot recalibration tables/figures.")

# -------------------------
# 3) Hardness index vs LOSO AUC + cross-representation LOSO comparison
# -------------------------
def hardness_vs_loso(group_label, loso_metrics_df):
    tr = transfer_scores[transfer_scores["group"] == group_label].copy()
    held = _infer_col(loso_metrics_df, ["heldout", "study", "cohort", "study_name"])
    auc_col = _infer_col(loso_metrics_df, ["roc_auc", "auc"])
    if held is None or auc_col is None:
        raise ValueError(f"Could not infer columns in LOSO metrics for {group_label}")
    lm = loso_metrics_df[[held, auc_col]].copy()
    lm.columns = ["cohort", "loso_auc"]
    m = tr.merge(lm, on="cohort", how="inner")

    pear_r, pear_p = stats.pearsonr(m["mean_auc_as_target"], m["loso_auc"])
    spear_r, spear_p = stats.spearmanr(m["mean_auc_as_target"], m["loso_auc"])

    m["group"] = group_label
    m.to_csv(os.path.join(TAB_DIR, f"hardness_vs_loso_{group_label}.csv"), index=False)

    plt.figure(figsize=(6.5, 5))
    sns.regplot(data=m, x="mean_auc_as_target", y="loso_auc", scatter_kws={"s": 60})
    plt.title(f"{group_label.capitalize()}: target hardness vs LOSO AUC\nSpearman ρ={spear_r:.2f} (p={spear_p:.3g})")
    plt.xlabel("Mean AUC as target (higher = easier target)")
    plt.ylabel("LOSO ROC-AUC")
    savefig(os.path.join(FIG_DIR, f"hardness_vs_loso_{group_label}"))
    return m, (pear_r, pear_p, spear_r, spear_p)

hard_species, corr_species = hardness_vs_loso("species", loso_species_metrics)
hard_pathways, corr_pathways = hardness_vs_loso("pathways", loso_pathways_metrics)

sp_loso = species_calib[["heldout", "roc_auc"]].rename(columns={"roc_auc": "auc_species"})
pw_loso = pathways_calib[["heldout", "roc_auc"]].rename(columns={"roc_auc": "auc_pathways"})
overlap = sp_loso.merge(pw_loso, on="heldout", how="inner")
overlap.to_csv(os.path.join(TAB_DIR, "loso_auc_species_vs_pathways_overlap.csv"), index=False)

if len(overlap) >= 3:
    r, p = stats.pearsonr(overlap["auc_species"], overlap["auc_pathways"])
    plt.figure(figsize=(6, 5))
    sns.regplot(data=overlap, x="auc_species", y="auc_pathways", scatter_kws={"s": 60})
    for _, row in overlap.iterrows():
        plt.text(row["auc_species"] + 0.002, row["auc_pathways"] + 0.002, row["heldout"], fontsize=9)
    plt.title(f"LOSO AUC: species vs pathways (overlap cohorts)\nPearson r={r:.2f} (p={p:.3g})")
    plt.xlabel("Species LOSO ROC-AUC")
    plt.ylabel("Pathways LOSO ROC-AUC")
    savefig(os.path.join(FIG_DIR, "loso_species_vs_pathways_auc"))

print("Saved hardness + cross-representation comparisons.")

# -------------------------
# 4) Adjusted regressions with robust SE (shift + log(ntrain) + log(ntest))
#    FIX: will NOT crash if n_train/n_test are absent.
#    Fallback: if matrix cache exists, infer sizes per cohort and attach to pairwise tables.
# -------------------------
def _infer_n_cols(df):
    ntrain = _infer_col(df, ["n_train", "train_n", "ntrain", "n_tr", "n_trn", "train_samples", "n_samples_train"])
    ntest  = _infer_col(df, ["n_test", "test_n", "ntest", "n_te", "n_tst", "test_samples", "n_samples_test"])
    return ntrain, ntest

def cache_available(group_label):
    d = os.path.join(CACHE_DIR, group_label)
    return os.path.isdir(d) and any(f.endswith(".npz") for f in os.listdir(d))

def cohort_sizes_from_cache(group_label):
    """
    Returns dict: {study_name: n_samples} using cached NPZ files.
    """
    d = os.path.join(CACHE_DIR, group_label)
    if not os.path.isdir(d):
        return {}
    out = {}
    for fn in os.listdir(d):
        if not fn.endswith(".npz"):
            continue
        study = fn.replace(".npz", "")
        z = np.load(os.path.join(d, fn), allow_pickle=True)
        X = z["X"]
        out[str(study)] = int(X.shape[0])
    return out

def adjusted_regression_any(shift_df, label, group_label=None, is_pairwise=False):
    """
    shift_df: dataframe with at least auc + shift columns; for pairwise it should also have train/test columns.
    FIX: if sizes are missing everywhere, run regression without them (shift-only), instead of raising.
    """
    df = shift_df.copy()

    auc_col = _infer_col(df, ["roc_auc", "auc"])
    shift_col = _infer_col(df, ["shift_distance", "shift", "distance", "mean_shift", "shift_mean"])
    if auc_col is None or shift_col is None:
        raise ValueError(f"{label}: could not infer auc/shift columns in {list(df.columns)}")

    ntrain_col, ntest_col = _infer_n_cols(df)
    sizes_source = "in_table"

    if is_pairwise:
        train_col = _infer_col(df, ["train", "source", "train_study", "study_train"])
        test_col  = _infer_col(df, ["test", "target", "test_study", "study_test"])
        if train_col is None or test_col is None:
            raise ValueError(f"{label}: pairwise requires train/test columns; found: {list(df.columns)}")

        df = df.rename(columns={train_col: "train", test_col: "test"})

        # If n_train/n_test missing, try cache-derived cohort sizes
        if (ntrain_col is None or ntest_col is None) and group_label is not None:
            if cache_available(group_label):
                sz = cohort_sizes_from_cache(group_label)
                if len(sz):
                    df["n_train"] = df["train"].astype(str).map(sz)
                    df["n_test"]  = df["test"].astype(str).map(sz)
                    ntrain_col, ntest_col = "n_train", "n_test"
                    sizes_source = "matrix_cache"
            # If still missing, try pair_auc file (may only have train/test/roc_auc)
            if ntrain_col is None or ntest_col is None:
                pair_path = os.path.join(OUT_ROOT, f"train_test_auc_{group_label}.csv")
                if os.path.exists(pair_path):
                    pair_auc = pd.read_csv(pair_path)
                    p_train = _infer_col(pair_auc, ["train", "source", "train_study", "study_train"])
                    p_test  = _infer_col(pair_auc, ["test", "target", "test_study", "study_test"])
                    p_ntr, p_nte = _infer_n_cols(pair_auc)
                    if p_train is not None and p_test is not None and p_ntr is not None and p_nte is not None:
                        pair_auc = pair_auc.rename(columns={p_train: "train", p_test: "test", p_ntr: "n_train", p_nte: "n_test"})
                        df = df.merge(pair_auc[["train", "test", "n_train", "n_test"]], on=["train", "test"], how="left")
                        ntrain_col, ntest_col = "n_train", "n_test"
                        sizes_source = "pair_auc_file"

    # Build design matrix (shift always; sizes optional)
    df = df.dropna(subset=[auc_col, shift_col]).copy()
    y = df[auc_col].astype(float).values

    cols = [np.ones(len(df)), df[shift_col].astype(float).values]
    names = ["intercept", "shift"]

    used_sizes = False
    if ntrain_col is not None and ntrain_col in df.columns and df[ntrain_col].notna().any():
        df["log_ntrain"] = np.log(np.maximum(2.0, df[ntrain_col].astype(float).values))
        cols.append(df["log_ntrain"].values)
        names.append("log_ntrain")
        used_sizes = True

    if ntest_col is not None and ntest_col in df.columns and df[ntest_col].notna().any():
        df["log_ntest"] = np.log(np.maximum(2.0, df[ntest_col].astype(float).values))
        cols.append(df["log_ntest"].values)
        names.append("log_ntest")
        used_sizes = True

    X = np.column_stack(cols)
    coef, se, tvals, pvals, r2 = hc3_ols(X, y)

    out = {
        "label": label,
        "n": int(len(df)),
        "r2": float(r2),
        "sizes_included": bool(used_sizes),
        "sizes_source": sizes_source if used_sizes else "none_available",
    }
    for i, nm in enumerate(names):
        out[f"coef_{nm}"] = float(coef[i])
        out[f"se_{nm}"] = float(se[i])
        out[f"p_{nm}"] = float(pvals[i])

    return pd.DataFrame([out])

reg_rows = []
reg_rows.append(adjusted_regression_any(shift_loso_species, "species_loso_adjusted", is_pairwise=False))
reg_rows.append(adjusted_regression_any(shift_loso_pathways, "pathways_loso_adjusted", is_pairwise=False))
reg_rows.append(adjusted_regression_any(shift_pair_species, "species_pairwise_adjusted", group_label="species", is_pairwise=True))
reg_rows.append(adjusted_regression_any(shift_pair_pathways, "pathways_pairwise_adjusted", group_label="pathways", is_pairwise=True))

reg_df = pd.concat(reg_rows, ignore_index=True)
reg_df.to_csv(os.path.join(TAB_DIR, "adjusted_regression_hc3_results.csv"), index=False)
print("Saved:", os.path.join(TAB_DIR, "adjusted_regression_hc3_results.csv"))
print(reg_df)

# -------------------------
# 5) Signature interpretability summaries
# -------------------------
def parse_taxonomy(feature):
    s = str(feature)
    if "::" in s:
        s = s.split("::", 1)[1]
    parts = s.split("|")
    out = {}
    for p in parts:
        if "__" in p:
            k, v = p.split("__", 1)
            out[k] = v
    return out

def species_signature_summary(sig_df):
    feat_col = _infer_col(sig_df, ["feature", "name"])
    if feat_col is None:
        raise ValueError("species signature: no feature col")
    feats = sig_df[feat_col].astype(str).values
    tax = [parse_taxonomy(f) for f in feats]
    phylum = [t.get("p", "NA") for t in tax]
    genus = [t.get("g", "NA") for t in tax]
    df = pd.DataFrame({"feature": feats, "phylum": phylum, "genus": genus})
    ph = df["phylum"].value_counts().reset_index()
    ph.columns = ["phylum", "count"]
    ge = df["genus"].value_counts().reset_index()
    ge.columns = ["genus", "count"]
    ph.to_csv(os.path.join(TAB_DIR, "species_signature_phylum_counts.csv"), index=False)
    ge.to_csv(os.path.join(TAB_DIR, "species_signature_genus_counts.csv"), index=False)

    plt.figure(figsize=(7, 4))
    sns.barplot(data=ph.head(12), x="count", y="phylum")
    plt.title("Stable species signature: phylum composition (top 12)")
    plt.xlabel("Number of stable features")
    plt.ylabel("Phylum")
    savefig(os.path.join(FIG_DIR, "species_signature_phylum_counts_top12"))

def pathway_keyword_bins(sig_df):
    feat_col = _infer_col(sig_df, ["feature", "name"])
    if feat_col is None:
        raise ValueError("pathways signature: no feature col")
    feats = sig_df[feat_col].astype(str).values

    bins = {
        "Carbohydrate metabolism": ["glycol", "pentose", "starch", "sugar", "glucon", "cellulose"],
        "Amino acid metabolism": ["valine", "leucine", "isoleuc", "lysine", "histidine", "tryptophan", "argin", "proline"],
        "Nucleotide metabolism": ["aden", "guanin", "purine", "pyrimid", "ribonucle", "deoxyrib"],
        "Vitamin/cofactor": ["thiamin", "riboflav", "biotin", "folate", "cobal", "pyridox", "menaquin", "quinone"],
        "Cell wall/peptidoglycan": ["peptidoglycan", "dtdp", "rham", "lipopolys", "teichoic"],
        "Energy/respiration": ["tca", "calvin", "electron", "respir", "ferment"],
        "Other/uncategorized": []
    }

    def assign_bin(name):
        n = name.lower()
        for k, kws in bins.items():
            if k == "Other/uncategorized":
                continue
            if any(kw in n for kw in kws):
                return k
        return "Other/uncategorized"

    df = pd.DataFrame({"feature": feats})
    df["bin"] = df["feature"].apply(assign_bin)
    counts = df["bin"].value_counts().reset_index()
    counts.columns = ["bin", "count"]
    counts.to_csv(os.path.join(TAB_DIR, "pathways_signature_keyword_bins.csv"), index=False)

    plt.figure(figsize=(7.5, 4.2))
    sns.barplot(data=counts, x="count", y="bin")
    plt.title("Stable pathway signature: keyword-based functional bins")
    plt.xlabel("Number of stable features")
    plt.ylabel("")
    savefig(os.path.join(FIG_DIR, "pathways_signature_keyword_bins"))

species_signature_summary(sig_species)
pathway_keyword_bins(sig_pathways)
print("Saved signature interpretability summaries.")

# -------------------------
# BIG HEATMAPS (species + optional pathways)
# -------------------------
def _pivot_matrix(df, row_col, col_col, val_col):
    m = df.pivot_table(index=row_col, columns=col_col, values=val_col, aggfunc="mean")
    return m

def _order_by_mean(mat, axis=0, descending=True):
    if axis == 0:
        means = mat.mean(axis=1, skipna=True)
        return means.sort_values(ascending=not descending).index.tolist()
    else:
        means = mat.mean(axis=0, skipna=True)
        return means.sort_values(ascending=not descending).index.tolist()

def plot_big_heatmap(mat, title, out_base, vmin=None, vmax=None, annotate=None, figsize=None):
    mat = mat.copy()
    n = mat.shape[0]
    if annotate is None:
        annotate = (n <= HEATMAP_ANNOTATE_MAX_N and mat.shape[1] <= HEATMAP_ANNOTATE_MAX_N)
    if figsize is None:
        # scale to matrix size; cap to prevent absurd sizes
        w = min(28, max(10, 0.55 * mat.shape[1] + 4))
        h = min(28, max(10, 0.55 * mat.shape[0] + 4))
        figsize = (w, h)

    plt.figure(figsize=figsize)
    ax = sns.heatmap(
        mat,
        vmin=vmin,
        vmax=vmax,
        annot=annotate,
        fmt=".2f" if annotate else "",
        linewidths=0.2,
        linecolor="white",
        cbar_kws={"label": "Value"}
    )
    ax.set_title(title)
    ax.set_xlabel(mat.columns.name if mat.columns.name else "")
    ax.set_ylabel(mat.index.name if mat.index.name else "")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    savefig(os.path.join(FIG_DIR, out_base))

def make_pairwise_heatmaps(group_label, pair_auc_df, shift_pair_df):
    # infer columns
    tr_col = _infer_col(pair_auc_df, ["train", "source", "train_study", "study_train"])
    te_col = _infer_col(pair_auc_df, ["test", "target", "test_study", "study_test"])
    auc_col = _infer_col(pair_auc_df, ["roc_auc", "auc"])
    if tr_col is None or te_col is None or auc_col is None:
        raise ValueError(f"[{group_label}] Could not infer train/test/auc in pair_auc: {list(pair_auc_df.columns)}")

    # AUC matrix
    auc_mat = _pivot_matrix(pair_auc_df.rename(columns={tr_col:"train", te_col:"test"}), "train", "test", auc_col)
    auc_mat.index.name = "train"
    auc_mat.columns.name = "test"

    # Order by mean AUC
    row_order = _order_by_mean(auc_mat, axis=0, descending=True)
    col_order = _order_by_mean(auc_mat, axis=1, descending=True)
    auc_mat = auc_mat.loc[row_order, col_order]

    plot_big_heatmap(
        auc_mat,
        title=f"{group_label.capitalize()} pairwise transfer ROC-AUC (train → test)",
        out_base=f"heatmap_pairwise_auc_{group_label}",
        vmin=HEATMAP_MIN_AUC,
        vmax=HEATMAP_MAX_AUC,
        annotate=None
    )
    auc_mat.to_csv(os.path.join(TAB_DIR, f"heatmap_pairwise_auc_{group_label}_matrix.csv"))

    # Shift matrix (if available)
    tr2 = _infer_col(shift_pair_df, ["train", "source", "train_study", "study_train"])
    te2 = _infer_col(shift_pair_df, ["test", "target", "test_study", "study_test"])
    sh_col = _infer_col(shift_pair_df, ["shift_distance", "shift", "distance", "mean_shift", "shift_mean"])
    if tr2 is not None and te2 is not None and sh_col is not None:
        sh_mat = _pivot_matrix(shift_pair_df.rename(columns={tr2:"train", te2:"test"}), "train", "test", sh_col)
        sh_mat.index.name = "train"
        sh_mat.columns.name = "test"
        # align to same order as AUC matrix for easy visual comparison
        sh_mat = sh_mat.reindex(index=auc_mat.index, columns=auc_mat.columns)
        plot_big_heatmap(
            sh_mat,
            title=f"{group_label.capitalize()} pairwise shift distance (train → test)",
            out_base=f"heatmap_pairwise_shift_{group_label}",
            vmin=float(np.nanpercentile(sh_mat.values, 5)),
            vmax=float(np.nanpercentile(sh_mat.values, 95)),
            annotate=None
        )
        sh_mat.to_csv(os.path.join(TAB_DIR, f"heatmap_pairwise_shift_{group_label}_matrix.csv"))

        # Also: AUC residualized by row/col means (simple centering) for pattern discovery
        row_mean = auc_mat.mean(axis=1, skipna=True)
        col_mean = auc_mat.mean(axis=0, skipna=True)
        grand = np.nanmean(auc_mat.values)
        resid = auc_mat.copy()
        for r in resid.index:
            for c in resid.columns:
                if pd.isna(resid.loc[r, c]):
                    continue
                resid.loc[r, c] = resid.loc[r, c] - (row_mean[r] + col_mean[c] - grand)
        plot_big_heatmap(
            resid,
            title=f"{group_label.capitalize()} pairwise AUC (double-centered residuals)",
            out_base=f"heatmap_pairwise_auc_residual_{group_label}",
            vmin=float(np.nanpercentile(resid.values, 5)),
            vmax=float(np.nanpercentile(resid.values, 95)),
            annotate=None
        )
        resid.to_csv(os.path.join(TAB_DIR, f"heatmap_pairwise_auc_residual_{group_label}_matrix.csv"))

    print(f"Saved big heatmaps for {group_label} -> {FIG_DIR}")

# Requested: BIG heatmaps including species
make_pairwise_heatmaps("species", pair_auc_species, shift_pair_species)
# Optional, but usually useful
make_pairwise_heatmaps("pathways", pair_auc_pathways, shift_pair_pathways)

# -------------------------
# OPTIONAL: MATRIX-CACHE-BASED SIGNATURE EFFECT-SIZE HEATMAP (species-focused)
#   - Heatmap of Cohen's d for stable signature features across cohorts
# -------------------------
def load_cache_group(group_label):
    d = os.path.join(CACHE_DIR, group_label)
    mats = {}
    for fn in os.listdir(d):
        if not fn.endswith(".npz"):
            continue
        path = os.path.join(d, fn)
        study = fn.replace(".npz", "")
        z = np.load(path, allow_pickle=True)
        X = z["X"].astype(np.float32, copy=False)
        y = z["y"].astype(int, copy=False)
        feats = z["features"].astype(object)
        mats[study] = {"X": X, "y": y, "features": feats}
    return mats

def align_to_union(mats):
    all_feats = []
    for v in mats.values():
        all_feats.extend(list(v["features"]))
    union = pd.Index(pd.unique(pd.Series(all_feats).astype(str)))
    idx_map = {f: i for i, f in enumerate(union)}
    aligned = {}
    for study, v in mats.items():
        feats = pd.Index(pd.Series(v["features"]).astype(str))
        X = v["X"]
        A = np.zeros((X.shape[0], len(union)), dtype=np.float32)
        cols = [idx_map[f] for f in feats]
        A[:, cols] = X
        aligned[study] = {"X": A, "y": v["y"].astype(int), "features": union.values.astype(object)}
    return aligned, union

def cohen_d_vector(X, y):
    y = np.asarray(y).astype(int)
    pos = X[y == 1]
    neg = X[y == 0]
    if len(pos) < 2 or len(neg) < 2:
        return np.full(X.shape[1], np.nan, dtype=float)
    m1 = pos.mean(axis=0)
    m0 = neg.mean(axis=0)
    s1 = pos.std(axis=0, ddof=1)
    s0 = neg.std(axis=0, ddof=1)
    sp = np.sqrt((s1**2 + s0**2) / 2.0)
    sp = np.maximum(sp, 1e-8)
    return (m1 - m0) / sp

def signature_effectsize_heatmap(group_label, signature_df, top_k=60):
    if not cache_available(group_label):
        print(f"[SKIP] No matrix cache for {group_label}; skipping signature effect-size heatmap.")
        return

    mats = load_cache_group(group_label)
    aligned, union = align_to_union(mats)
    studies = sorted(aligned.keys())

    feat_col = _infer_col(signature_df, ["feature", "name"])
    sig_set = set(signature_df[feat_col].astype(str).tolist())
    sig_idx = [i for i, f in enumerate(union.astype(str)) if f in sig_set]
    if len(sig_idx) < 5:
        print(f"[SKIP] Signature features in union too few (n={len(sig_idx)}).")
        return

    # Compute d per cohort for signature features
    D = []
    for s in studies:
        dvec = cohen_d_vector(aligned[s]["X"][:, sig_idx], aligned[s]["y"])
        D.append(dvec)
    D = np.vstack(D)  # (cohorts, features)
    # pick top_k by mean abs d
    mean_abs = np.nanmean(np.abs(D), axis=0)
    order = np.argsort(-mean_abs)[:min(top_k, len(sig_idx))]

    mat = pd.DataFrame(
        D[:, order],
        index=studies,
        columns=[str(union[sig_idx[i]]) for i in order]
    )
    # transpose: features x cohorts tends to read better
    matT = mat.T

    # order cohorts by average |d|
    cohort_order = matT.abs().mean(axis=0).sort_values(ascending=False).index.tolist()
    feature_order = matT.abs().mean(axis=1).sort_values(ascending=False).index.tolist()
    matT = matT.loc[feature_order, cohort_order]

    # plot
    plot_big_heatmap(
        matT,
        title=f"{group_label.capitalize()} stable signature effect sizes (Cohen's d) across cohorts (top {len(feature_order)})",
        out_base=f"heatmap_signature_effectsizes_{group_label}",
        vmin=float(np.nanpercentile(matT.values, 5)),
        vmax=float(np.nanpercentile(matT.values, 95)),
        annotate=False,
        figsize=(min(28, max(12, 0.25 * matT.shape[1] + 10)), min(28, max(12, 0.18 * matT.shape[0] + 10)))
    )
    matT.to_csv(os.path.join(TAB_DIR, f"heatmap_signature_effectsizes_{group_label}.csv"))
    print(f"Saved signature effect-size heatmap for {group_label}.")

# Species-focused, as requested
signature_effectsize_heatmap("species", sig_species, top_k=60)

# -------------------------
# 6) Cross-representation ensemble (if sample IDs match) -- minor robustness fix
# -------------------------
def try_ensemble(pred_sp, pred_pw):
    if "sample_id" not in pred_sp.columns or "sample_id" not in pred_pw.columns:
        print("[SKIP] Ensemble: sample_id not found in both predictions files.")
        return
    m = pred_sp.merge(pred_pw, on=["heldout", "sample_id"], suffixes=("_sp", "_pw"))
    if len(m) == 0:
        print("[SKIP] Ensemble: no matching (heldout, sample_id) pairs.")
        return

    if not np.all(m["y_true_sp"].values == m["y_true_pw"].values):
        print("[WARN] Ensemble: y labels mismatch after merge; using species labels.")

    rows = []
    for cohort, sub in m.groupby("heldout"):
        yy = sub["y_true_sp"].values.astype(int)
        p_sp = _clip_probs(sub["p_hat_sp"].values)
        p_pw = _clip_probs(sub["p_hat_pw"].values)
        p_ens = _clip_probs(0.5 * (p_sp + p_pw))
        if len(np.unique(yy)) < 2:
            continue
        rows.append({
            "heldout": cohort,
            "auc_species": roc_auc_score(yy, p_sp),
            "auc_pathways": roc_auc_score(yy, p_pw),
            "auc_ensemble": roc_auc_score(yy, p_ens),
            "n": int(len(yy))
        })
    ens = pd.DataFrame(rows).sort_values("auc_ensemble", ascending=False)
    ens.to_csv(os.path.join(TAB_DIR, "ensemble_auc_by_cohort.csv"), index=False)

    if len(ens) >= 3:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=ens, x="auc_species", y="auc_pathways", size="n", sizes=(40, 200))
        for _, r in ens.iterrows():
            plt.text(r["auc_species"] + 0.002, r["auc_pathways"] + 0.002, r["heldout"], fontsize=9)
        plt.title("Overlap cohorts: species vs pathways AUC (size ~ n)")
        plt.xlabel("Species AUC")
        plt.ylabel("Pathways AUC")
        savefig(os.path.join(FIG_DIR, "species_vs_pathways_auc_scatter_ens"))

        plt.figure(figsize=(7, 4.5))
        ens_m = ens.melt(id_vars=["heldout"], value_vars=["auc_species", "auc_pathways", "auc_ensemble"],
                         var_name="model", value_name="auc")
        sns.boxplot(data=ens_m, x="model", y="auc")
        plt.title("Overlap cohorts: AUC distribution (species vs pathways vs ensemble)")
        plt.xlabel("")
        plt.ylabel("ROC-AUC")
        savefig(os.path.join(FIG_DIR, "ensemble_auc_boxplot"))

    print("Saved ensemble results (if overlaps exist).")

try_ensemble(pred_species_std, pred_pathways_std)

# -------------------------
# Final index of outputs
# -------------------------
index_rows = []
for root, _, files in os.walk(TAB_DIR):
    for f in sorted(files):
        if f.endswith(".csv"):
            index_rows.append({"type": "table", "path": os.path.join(root, f)})
for root, _, files in os.walk(FIG_DIR):
    for f in sorted(files):
        if f.endswith(".png") or f.endswith(".pdf"):
            index_rows.append({"type": "figure", "path": os.path.join(root, f)})

index_df = pd.DataFrame(index_rows)
index_df.to_csv(os.path.join(OUT_ROOT, "addons_output_index.csv"), index=False)

print("\nDone.")
print("Add-on tables:", TAB_DIR)
print("Add-on figures:", FIG_DIR)
print("Index:", os.path.join(OUT_ROOT, "addons_output_index.csv"))

if not cache_available("species") and not cache_available("pathways"):
    print("\nNOTE: Matrix-cache based effect-size heatmaps and size-augmented pairwise regressions use NPZ caches.")
    print("To enable, export per-cohort NPZ caches (X,y,features) into:")
    print("  /content/curatedMetagenomicData_EH/analysis_outputs_crc/matrix_cache/{group}/{study}.npz")


FileNotFoundError: Missing required file: /content/curatedMetagenomicData_EH/analysis_outputs_crc/transferability_source_target_scores.csv

##6. Final source-selection / deployment suite

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn umap-learn

import os, math, shutil, zipfile, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    mean_absolute_error, r2_score, silhouette_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG
# -------------------------
ZIP_CANDIDATES = [
    "/content/analysis_outputs_crc.zip",
    "/content/analysis_outputs_crc.zip.zip",
    "/content/analysis_outputs_crc (1).zip",
]

OUT_ROOT = "/content/curatedMetagenomicData_EH/analysis_outputs_crc"
TAB_DIR  = os.path.join(OUT_ROOT, "tables_pub_safe_suite_v2")
FIG_DIR  = os.path.join(OUT_ROOT, "figures_pub_safe_suite_v2")
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RNG = np.random.default_rng(42)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 400,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

def savefig(path_base):
    plt.tight_layout()
    plt.savefig(path_base + ".png", bbox_inches="tight")
    plt.savefig(path_base + ".pdf", bbox_inches="tight")
    plt.close()

# -------------------------
# (Optional) unzip if needed
# -------------------------
def maybe_unzip_to_outroot():
    if os.path.exists(OUT_ROOT) and len(os.listdir(OUT_ROOT)) > 0:
        print("Found existing OUT_ROOT:", OUT_ROOT)
        return

    zip_path = None
    for z in ZIP_CANDIDATES:
        if os.path.exists(z):
            zip_path = z
            break

    if zip_path is None:
        raise FileNotFoundError(
            "Could not find inputs. Either upload analysis_outputs_crc.zip to /content, "
            "or ensure OUT_ROOT exists with the required CSVs."
        )

    print("Unzipping:", zip_path)
    tmp = "/content/_tmp_unzip_crc"
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.makedirs(tmp, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp)

    # Your zip likely contains: content/curatedMetagenomicData_EH/analysis_outputs_crc/...
    candidate = os.path.join(tmp, "content", "curatedMetagenomicData_EH", "analysis_outputs_crc")
    if not os.path.exists(candidate):
        # fallback: search for analysis_outputs_crc inside tmp
        found = None
        for root, dirs, files in os.walk(tmp):
            if os.path.basename(root) == "analysis_outputs_crc":
                found = root
                break
        if found is None:
            raise FileNotFoundError("Could not locate analysis_outputs_crc inside the zip.")
        candidate = found

    os.makedirs(os.path.dirname(OUT_ROOT), exist_ok=True)
    if os.path.exists(OUT_ROOT):
        shutil.rmtree(OUT_ROOT)
    shutil.copytree(candidate, OUT_ROOT)
    print("Extracted to:", OUT_ROOT)

maybe_unzip_to_outroot()

# -------------------------
# Robust utilities
# -------------------------
def _infer_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None

def _clip_probs(p, eps=1e-6):
    p = np.asarray(p, dtype=float)
    return np.clip(p, eps, 1.0 - eps)

def load_csv_req(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path)

def standardize_pred_df(df, group_name):
    heldout_col = _infer_col(df, ["heldout", "study", "cohort", "study_name"])
    y_col = _infer_col(df, ["y_true", "y", "label", "outcome"])
    p_col = _infer_col(df, ["p_hat", "prob", "proba", "pred_prob", "prediction", "p", "y_pred_prob"])

    if heldout_col is None or y_col is None or p_col is None:
        raise ValueError(
            f"[{group_name}] Could not infer columns. "
            f"Found heldout={heldout_col}, y={y_col}, p={p_col}. "
            f"Columns: {list(df.columns)}"
        )

    out = df.copy()
    out = out.rename(columns={heldout_col: "heldout", y_col: "y_true", p_col: "p_hat"})
    out["heldout"] = out["heldout"].astype(str)
    out["y_true"]  = out["y_true"].astype(int)
    out["p_hat"]   = _clip_probs(out["p_hat"].astype(float).values)
    return out

def cohort_stats_from_pred(pred_df):
    g = pred_df.groupby("heldout")["y_true"].agg(["count", "mean"]).reset_index()
    g = g.rename(columns={"heldout": "cohort", "count": "n", "mean": "prevalence"})
    g["cohort"] = g["cohort"].astype(str)
    return g

def loso_auc_table(pred_df, group_label):
    rows = []
    for coh, g in pred_df.groupby("heldout"):
        y = g["y_true"].values
        p = g["p_hat"].values
        try:
            auc = roc_auc_score(y, p)
        except Exception:
            auc = np.nan
        try:
            pr = average_precision_score(y, p)
        except Exception:
            pr = np.nan
        rows.append({
            "group": group_label,
            "heldout": str(coh),
            "n": int(len(g)),
            "prevalence": float(np.mean(y)),
            "roc_auc": float(auc) if np.isfinite(auc) else np.nan,
            "pr_auc": float(pr) if np.isfinite(pr) else np.nan,
        })
    return pd.DataFrame(rows)

def bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order]
    q = np.empty(m, dtype=float)
    prev = 1.0
    for i in range(m - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * m / rank
        prev = min(prev, val)
        q[i] = prev
    q_adj = np.empty(m, dtype=float)
    q_adj[order] = np.clip(q, 0, 1)
    return q_adj

# -------------------------
# Build pairwise dataset (AUC + shift + cohort covariates)
# -------------------------
def build_pairwise_dataset(pair_auc_df, shift_pair_df, cohort_stats_df, group_label):
    tr_col_auc = _infer_col(pair_auc_df, ["train", "source", "train_study", "study_train"])
    te_col_auc = _infer_col(pair_auc_df, ["test", "target", "test_study", "study_test"])
    auc_col    = _infer_col(pair_auc_df, ["roc_auc", "auc"])
    if tr_col_auc is None or te_col_auc is None or auc_col is None:
        raise ValueError(f"[{group_label}] train/test/auc cols not found in pair_auc: {list(pair_auc_df.columns)}")

    A = pair_auc_df[[tr_col_auc, te_col_auc, auc_col]].copy()
    A.columns = ["train", "test", "roc_auc"]
    A["train"] = A["train"].astype(str)
    A["test"]  = A["test"].astype(str)
    A["roc_auc"] = A["roc_auc"].astype(float)

    tr_col_sh = _infer_col(shift_pair_df, ["train", "source", "train_study", "study_train"])
    te_col_sh = _infer_col(shift_pair_df, ["test", "target", "test_study", "study_test"])
    sh_col    = _infer_col(shift_pair_df, ["shift_distance", "shift", "distance", "mean_shift", "shift_mean"])
    if tr_col_sh is None or te_col_sh is None or sh_col is None:
        raise ValueError(f"[{group_label}] train/test/shift cols not found in shift_pair: {list(shift_pair_df.columns)}")

    S = shift_pair_df[[tr_col_sh, te_col_sh, sh_col]].copy()
    S.columns = ["train", "test", "shift_distance"]
    S["train"] = S["train"].astype(str)
    S["test"]  = S["test"].astype(str)
    S["shift_distance"] = S["shift_distance"].astype(float)

    df = A.merge(S, on=["train", "test"], how="inner")

    cs = cohort_stats_df.copy()
    df = df.merge(cs.rename(columns={"cohort": "train", "n": "n_train", "prevalence": "prev_train"}), on="train", how="left")
    df = df.merge(cs.rename(columns={"cohort": "test",  "n": "n_test",  "prevalence": "prev_test"}),  on="test",  how="left")

    df["abs_prev_diff"] = (df["prev_train"] - df["prev_test"]).abs()
    df["log_n_train"] = np.log(np.maximum(2.0, df["n_train"].astype(float)))
    df["log_n_test"]  = np.log(np.maximum(2.0, df["n_test"].astype(float)))

    # remove diagonal if present
    df = df[df["train"] != df["test"]].copy()

    # safety: drop non-finite
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["roc_auc", "shift_distance", "n_train", "n_test", "prev_train", "prev_test"]).copy()

    df["group"] = group_label
    df.to_csv(os.path.join(TAB_DIR, f"pairwise_dataset_{group_label}.csv"), index=False)
    return df

# -------------------------
# Heatmaps (safe)
# -------------------------
def pivot_matrix(df, value_col):
    mat = df.pivot_table(index="train", columns="test", values=value_col, aggfunc="mean")
    idx = sorted(set(mat.index) | set(mat.columns))
    return mat.reindex(index=idx, columns=idx)

def clean_matrix_for_clustermap(mat):
    M = mat.copy().astype(float)
    M = M.replace([np.inf, -np.inf], np.nan)

    vals = M.values
    row_mean = np.nanmean(vals, axis=1)
    col_mean = np.nanmean(vals, axis=0)
    global_mean = np.nanmean(vals)

    # fill NaNs row-wise
    for i in range(vals.shape[0]):
        mask = ~np.isfinite(vals[i, :])
        if mask.any():
            fill = row_mean[i] if np.isfinite(row_mean[i]) else global_mean
            vals[i, mask] = fill

    # fill remaining by column mean
    mask = ~np.isfinite(vals)
    if mask.any():
        vals[mask] = np.take(col_mean, np.where(mask)[1])

    # fill any remaining with global mean
    vals[~np.isfinite(vals)] = global_mean
    return pd.DataFrame(vals, index=M.index, columns=M.columns)

def heatmap_clustermap_safe(mat, title, out_base, cmap="viridis", vmin=None, vmax=None):
    M = clean_matrix_for_clustermap(mat)
    cg = sns.clustermap(
        M,
        cmap=cmap,
        figsize=(12, 10),
        linewidths=0.0,
        vmin=vmin,
        vmax=vmax,
        cbar_kws={"label": title},
        dendrogram_ratio=(.12, .12)
    )
    cg.fig.suptitle(title, y=1.02)
    cg.fig.savefig(os.path.join(FIG_DIR, out_base + ".png"), dpi=400, bbox_inches="tight")
    cg.fig.savefig(os.path.join(FIG_DIR, out_base + ".pdf"), dpi=400, bbox_inches="tight")
    plt.close(cg.fig)

def make_big_heatmaps(pair_df, group_label):
    auc_mat = pivot_matrix(pair_df, "roc_auc")
    sh_mat  = pivot_matrix(pair_df, "shift_distance")

    auc_mat.to_csv(os.path.join(TAB_DIR, f"{group_label}_auc_matrix.csv"))
    sh_mat.to_csv(os.path.join(TAB_DIR, f"{group_label}_shift_matrix.csv"))

    heatmap_clustermap_safe(
        auc_mat,
        title=f"{group_label.capitalize()} pairwise transfer ROC-AUC (train→test)",
        out_base=f"{group_label}_heatmap_auc",
        cmap="viridis",
        vmin=0.4, vmax=1.0
    )

    # robust vmin/vmax
    sh_vals = clean_matrix_for_clustermap(sh_mat).values
    vmin = float(np.nanpercentile(sh_vals, 2))
    vmax = float(np.nanpercentile(sh_vals, 98))

    heatmap_clustermap_safe(
        sh_mat,
        title=f"{group_label.capitalize()} pairwise shift distance (train↔test)",
        out_base=f"{group_label}_heatmap_shift",
        cmap="magma",
        vmin=vmin, vmax=vmax
    )

    # scatter shift vs AUC
    plt.figure(figsize=(6.8, 5.4))
    sns.regplot(
        data=pair_df, x="shift_distance", y="roc_auc",
        scatter_kws={"s": 22, "alpha": 0.45},
        line_kws={"linewidth": 2}
    )
    r, p = stats.pearsonr(pair_df["shift_distance"], pair_df["roc_auc"])
    plt.title(f"{group_label.capitalize()}: shift vs transfer AUC\nPearson r={r:.2f} (p={p:.3g})")
    plt.xlabel("Shift distance")
    plt.ylabel("Transfer ROC-AUC")
    savefig(os.path.join(FIG_DIR, f"{group_label}_shift_vs_auc_scatter"))

# -------------------------
# Predictor + ablations
# -------------------------
def fit_predictor(train_df, feature_cols, model_name="ridge", seed=42):
    X = train_df[feature_cols].values
    y = train_df["roc_auc"].values

    if model_name == "ridge":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("reg", Ridge(alpha=1.0, random_state=seed))
        ])
    elif model_name == "elasticnet":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("reg", ElasticNet(alpha=0.01, l1_ratio=0.2, random_state=seed, max_iter=5000))
        ])
    elif model_name == "rf":
        # keep it fast (datasets are tiny; 150–250 trees is enough)
        model = RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=2,
            random_state=seed,
            n_jobs=-1
        )
    else:
        raise ValueError("model_name must be one of: ridge, elasticnet, rf")

    model.fit(X, y)
    return model

def leave_one_target_out_recommendations(pair_df, group_label, feature_cols, model_name="ridge"):
    targets = sorted(pair_df["test"].unique())
    rec_rows = []
    pred_rows = []

    cohorts = sorted(set(pair_df["train"]).union(set(pair_df["test"])))
    pred_mat = pd.DataFrame(index=cohorts, columns=cohorts, dtype=float)

    for tgt in targets:
        train_df = pair_df[pair_df["test"] != tgt].copy()
        test_df  = pair_df[pair_df["test"] == tgt].copy()

        if len(test_df) < 3 or len(train_df) < 10:
            continue

        model = fit_predictor(train_df, feature_cols, model_name=model_name, seed=42)
        yhat  = model.predict(test_df[feature_cols].values)
        test_df["pred_auc"] = yhat

        for _, r in test_df.iterrows():
            pred_mat.loc[r["train"], r["test"]] = float(r["pred_auc"])

        best_pred = test_df.loc[test_df["pred_auc"].idxmax()]
        chosen_source = str(best_pred["train"])
        chosen_pred_auc = float(best_pred["pred_auc"])
        chosen_obs_auc = float(test_df[test_df["train"] == chosen_source]["roc_auc"].iloc[0])

        oracle_row = test_df.loc[test_df["roc_auc"].idxmax()]
        oracle_source = str(oracle_row["train"])
        oracle_auc = float(oracle_row["roc_auc"])

        rec_rows.append({
            "group": group_label,
            "target": tgt,
            "model": model_name,
            "features": "+".join(feature_cols),
            "chosen_source": chosen_source,
            "chosen_pred_auc": chosen_pred_auc,
            "chosen_obs_auc": chosen_obs_auc,
            "oracle_source": oracle_source,
            "oracle_auc": oracle_auc,
            "regret": oracle_auc - chosen_obs_auc,
            "top1_hit": int(chosen_source == oracle_source),
            "n_candidates": int(len(test_df))
        })

        pred_rows.append(test_df.assign(group=group_label, model=model_name))

    rec_df = pd.DataFrame(rec_rows)
    pred_long = pd.concat(pred_rows, ignore_index=True) if len(pred_rows) else pd.DataFrame()

    if len(pred_long):
        mae = mean_absolute_error(pred_long["roc_auc"], pred_long["pred_auc"])
        r2  = r2_score(pred_long["roc_auc"], pred_long["pred_auc"])
    else:
        mae, r2 = np.nan, np.nan

    summary = pd.DataFrame([{
        "group": group_label,
        "model": model_name,
        "features": "+".join(feature_cols),
        "n_targets_evaluated": int(rec_df["target"].nunique()) if len(rec_df) else 0,
        "pairwise_mae": float(mae) if np.isfinite(mae) else np.nan,
        "pairwise_r2": float(r2) if np.isfinite(r2) else np.nan,
        "top1_hit_rate": float(rec_df["top1_hit"].mean()) if len(rec_df) else np.nan,
        "mean_regret": float(rec_df["regret"].mean()) if len(rec_df) else np.nan,
        "median_regret": float(rec_df["regret"].median()) if len(rec_df) else np.nan,
    }])

    return rec_df, summary, pred_mat, pred_long

def save_predicted_auc_heatmap(pred_mat, group_label, setting_name):
    heatmap_clustermap_safe(
        pred_mat,
        title=f"{group_label.capitalize()} predicted transfer AUC ({setting_name})",
        out_base=f"{group_label}_heatmap_predicted_auc__{setting_name}",
        cmap="viridis",
        vmin=0.4, vmax=1.0
    )

def run_ablation_grid(pair_df, group_label):
    SETTINGS = {
        "shift_only":  ["shift_distance"],
        "shift_prev":  ["shift_distance", "abs_prev_diff"],
        "shift_sizes": ["shift_distance", "log_n_train", "log_n_test"],
        "all":         ["shift_distance", "abs_prev_diff", "log_n_train", "log_n_test"],
    }
    MODELS = ["ridge", "rf"]  # keep it simple + fast; you can add elasticnet if you want

    grid_rows = []
    rec_out = {}
    pred_mats = {}

    for sname, feats in SETTINGS.items():
        for m in MODELS:
            rec_df, summ_df, pred_mat, pred_long = leave_one_target_out_recommendations(
                pair_df, group_label, feats, model_name=m
            )
            setting_key = f"{sname}__{m}"
            rec_out[setting_key] = rec_df
            pred_mats[setting_key] = pred_mat

            row = summ_df.iloc[0].to_dict()
            row["setting"] = sname
            row["model"] = m
            row["setting_key"] = setting_key
            grid_rows.append(row)

            # save per-setting outputs
            rec_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_recommendations_{setting_key}.csv"), index=False)
            summ_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_recommendations_summary_{setting_key}.csv"), index=False)
            pred_mat.to_csv(os.path.join(TAB_DIR, f"{group_label}_predicted_auc_matrix_{setting_key}.csv"))

    comp = pd.DataFrame(grid_rows)
    comp.to_csv(os.path.join(TAB_DIR, f"{group_label}_ablation_model_grid.csv"), index=False)

    # choose best: lowest mean_regret, then lowest MAE
    best = comp.sort_values(["mean_regret", "pairwise_mae"], ascending=[True, True]).iloc[0]
    best_key = best["setting_key"]
    print(f"[{group_label}] Best setting:", best_key)

    # plot comparisons
    plt.figure(figsize=(8.5, 4.6))
    sns.barplot(data=comp, x="setting", y="mean_regret", hue="model")
    plt.title(f"{group_label.capitalize()}: Mean regret (lower is better)")
    plt.xlabel("")
    plt.ylabel("Mean regret (oracle AUC − chosen AUC)")
    plt.legend(frameon=False, title="model")
    savefig(os.path.join(FIG_DIR, f"{group_label}_ablation_mean_regret"))

    plt.figure(figsize=(8.5, 4.6))
    sns.barplot(data=comp, x="setting", y="pairwise_mae", hue="model")
    plt.title(f"{group_label.capitalize()}: Pairwise MAE of predicted AUC (lower is better)")
    plt.xlabel("")
    plt.ylabel("MAE")
    plt.legend(frameon=False, title="model")
    savefig(os.path.join(FIG_DIR, f"{group_label}_ablation_mae"))

    return comp, best_key, rec_out, pred_mats

# -------------------------
# Baselines
# -------------------------
def compute_baselines(pair_df, loso_df, group_label):
    # loso_df: output of loso_auc_table()
    pooled = loso_df[["heldout", "roc_auc"]].rename(columns={"heldout": "target", "roc_auc": "pooled_baseline_auc"}).copy()

    rows = []
    for tgt in sorted(pair_df["test"].unique()):
        cand = pair_df[pair_df["test"] == tgt].copy()
        if len(cand) == 0:
            continue

        # oracle
        o = cand.loc[cand["roc_auc"].idxmax()]
        oracle_source = str(o["train"])
        oracle_auc = float(o["roc_auc"])

        # min-shift
        s = cand.loc[cand["shift_distance"].idxmin()]
        shift_source = str(s["train"])
        shift_auc = float(s["roc_auc"])
        shift_val = float(s["shift_distance"])

        # random baseline: expectation across candidates
        rand_mean = float(cand["roc_auc"].mean())
        rand_sd   = float(cand["roc_auc"].std(ddof=1)) if len(cand) > 1 else 0.0

        # pooled (LOSO)
        pooled_auc = float(pooled[pooled["target"] == tgt]["pooled_baseline_auc"].iloc[0]) \
                     if (pooled["target"] == tgt).any() else np.nan

        rows.append({
            "group": group_label,
            "target": tgt,
            "n_candidates": int(len(cand)),
            "pooled_baseline_auc": pooled_auc,
            "oracle_source": oracle_source,
            "oracle_auc": oracle_auc,
            "shift_baseline_source": shift_source,
            "shift_baseline_auc": shift_auc,
            "shift_baseline_shift": shift_val,
            "random_mean_auc": rand_mean,
            "random_sd_auc": rand_sd,
            "oracle_gain_over_pooled": oracle_auc - pooled_auc,
            "shift_gain_over_pooled": shift_auc - pooled_auc,
        })

    by_target = pd.DataFrame(rows)
    by_target.to_csv(os.path.join(TAB_DIR, f"{group_label}_baselines_by_target.csv"), index=False)

    summary = pd.DataFrame([{
        "group": group_label,
        "n_targets": int(by_target["target"].nunique()) if len(by_target) else 0,
        "pooled_mean_auc": float(by_target["pooled_baseline_auc"].mean()),
        "oracle_mean_auc": float(by_target["oracle_auc"].mean()),
        "shift_mean_auc": float(by_target["shift_baseline_auc"].mean()),
        "random_mean_auc": float(by_target["random_mean_auc"].mean()),
        "oracle_mean_gain_over_pooled": float(by_target["oracle_gain_over_pooled"].mean()),
        "shift_mean_gain_over_pooled": float(by_target["shift_gain_over_pooled"].mean()),
    }])
    summary.to_csv(os.path.join(TAB_DIR, f"{group_label}_baselines_summary.csv"), index=False)

    # baseline boxplot
    long = by_target.melt(
        id_vars=["target"],
        value_vars=["pooled_baseline_auc", "oracle_auc", "shift_baseline_auc", "random_mean_auc"],
        var_name="baseline",
        value_name="roc_auc"
    )
    plt.figure(figsize=(8.2, 4.8))
    sns.boxplot(data=long, x="baseline", y="roc_auc")
    plt.title(f"{group_label.capitalize()}: baseline transfer performance across targets")
    plt.xlabel("")
    plt.ylabel("ROC-AUC")
    savefig(os.path.join(FIG_DIR, f"{group_label}_baseline_boxplot"))

    return by_target, summary

# -------------------------
# Multiple-comparisons (per-target tests + BH-FDR)
# -------------------------
def per_target_shift_auc_tests(pair_df, group_label):
    rows = []
    for tgt, df in pair_df.groupby("test"):
        if len(df) < 4:
            continue
        r, p = stats.pearsonr(df["shift_distance"], df["roc_auc"])
        rows.append({"group": group_label, "target": str(tgt), "n_pairs": int(len(df)), "r": float(r), "p": float(p)})
    out = pd.DataFrame(rows)
    if len(out):
        out["q_bh_fdr"] = bh_fdr(out["p"].values)
    out.to_csv(os.path.join(TAB_DIR, f"{group_label}_per_target_shift_vs_auc_bh_fdr.csv"), index=False)

    # quick plot
    if len(out):
        plt.figure(figsize=(8.5, 4.6))
        sns.scatterplot(data=out, x="r", y="q_bh_fdr", s=80)
        plt.axhline(0.05, linewidth=1.5)
        plt.title(f"{group_label.capitalize()}: per-target shift~AUC correlations with BH-FDR")
        plt.xlabel("Pearson r (shift vs AUC)")
        plt.ylabel("BH-FDR q-value")
        savefig(os.path.join(FIG_DIR, f"{group_label}_per_target_shift_auc_bh_fdr"))
    return out

# -------------------------
# Robustness / sensitivity
# -------------------------
def global_shift_auc(pair_df):
    r, p = stats.pearsonr(pair_df["shift_distance"], pair_df["roc_auc"])
    return float(r), float(p)

def robustness_suite(pair_df, cohort_stats_df, group_label, B=500, winsor=(0.02, 0.98)):
    rows = []

    r0, p0 = global_shift_auc(pair_df)
    rows.append({"scenario": "all_pairs", "r": r0, "p": p0})

    # remove largest/smallest target cohort (by n)
    cs = cohort_stats_df.copy()
    largest = cs.loc[cs["n"].idxmax()]["cohort"]
    smallest = cs.loc[cs["n"].idxmin()]["cohort"]

    df1 = pair_df[pair_df["test"] != largest].copy()
    r1, p1 = global_shift_auc(df1)
    rows.append({"scenario": f"remove_largest_target({largest})", "r": r1, "p": p1})

    df2 = pair_df[pair_df["test"] != smallest].copy()
    r2, p2 = global_shift_auc(df2)
    rows.append({"scenario": f"remove_smallest_target({smallest})", "r": r2, "p": p2})

    # winsorize shift
    lo = float(pair_df["shift_distance"].quantile(winsor[0]))
    hi = float(pair_df["shift_distance"].quantile(winsor[1]))
    dfw = pair_df.copy()
    dfw["shift_w"] = dfw["shift_distance"].clip(lo, hi)
    r3, p3 = stats.pearsonr(dfw["shift_w"], dfw["roc_auc"])
    rows.append({"scenario": f"winsor_shift({winsor[0]}-{winsor[1]})", "r": float(r3), "p": float(p3)})

    # leave-one-target-out meta check
    loo = []
    targets = sorted(pair_df["test"].unique())
    for tgt in targets:
        d = pair_df[pair_df["test"] != tgt]
        if len(d) < 10:
            continue
        r, p = global_shift_auc(d)
        loo.append({"left_out_target": tgt, "r": r, "p": p})
    loo_df = pd.DataFrame(loo)
    loo_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_robustness_leave_one_target_out.csv"), index=False)

    # bootstrap over targets (cohort-level stability)
    boot = []
    if len(targets) >= 3:
        for b in range(B):
            sampled = RNG.choice(targets, size=len(targets), replace=True)
            d = pair_df[pair_df["test"].isin(sampled)]
            if len(d) < 10:
                continue
            r, _ = global_shift_auc(d)
            boot.append(r)
    boot = np.asarray(boot, dtype=float)
    boot_df = pd.DataFrame({"bootstrap_r": boot})
    boot_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_robustness_bootstrap_r.csv"), index=False)

    # plots
    if len(loo_df):
        plt.figure(figsize=(7.2, 4.6))
        sns.histplot(loo_df["r"], bins=12)
        plt.title(f"{group_label.capitalize()}: LOO distribution of global r(shift,AUC)")
        plt.xlabel("Pearson r")
        plt.ylabel("count")
        savefig(os.path.join(FIG_DIR, f"{group_label}_robustness_loo_r_hist"))

    if len(boot_df):
        plt.figure(figsize=(7.2, 4.6))
        sns.histplot(boot_df["bootstrap_r"], bins=18)
        plt.title(f"{group_label.capitalize()}: Bootstrap distribution of r(shift,AUC)")
        plt.xlabel("Pearson r")
        plt.ylabel("count")
        savefig(os.path.join(FIG_DIR, f"{group_label}_robustness_bootstrap_r_hist"))

    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(TAB_DIR, f"{group_label}_robustness_summary.csv"), index=False)
    return summary, loo_df, boot_df

# -------------------------
# Unsupervised structure (UMAP/PCA + KMeans) on transfer profiles
# -------------------------
def get_umap_or_pca_embed(X, seed=42):
    try:
        import umap
        reducer = umap.UMAP(
            n_neighbors=10,
            min_dist=0.15,
            metric="euclidean",
            random_state=seed
        )
        Z = reducer.fit_transform(X)
        return Z, "umap"
    except Exception:
        pca = PCA(n_components=2, random_state=seed)
        Z = pca.fit_transform(X)
        return Z, "pca"

def choose_k_by_silhouette(Z, k_min=2, k_max=8, seed=42):
    best_k, best_s = None, -np.inf
    rows = []
    for k in range(k_min, k_max + 1):
        km = KMeans(n_clusters=k, random_state=seed, n_init=20)
        lab = km.fit_predict(Z)
        s = silhouette_score(Z, lab) if len(np.unique(lab)) > 1 else np.nan
        rows.append({"k": k, "silhouette": s})
        if np.isfinite(s) and s > best_s:
            best_s, best_k = s, k
    return best_k, pd.DataFrame(rows)

def unsupervised_cohort_structure(pair_df, cohort_stats_df, group_label):
    auc_mat = pivot_matrix(pair_df, "roc_auc")
    M = auc_mat.values.astype(float)

    # fill missing
    row_mean = np.nanmean(M, axis=1, keepdims=True)
    M2 = np.where(np.isnan(M), row_mean, M)
    global_mean = np.nanmean(M2)
    M2 = np.where(np.isnan(M2), global_mean, M2)

    X = StandardScaler().fit_transform(M2)
    Z, method = get_umap_or_pca_embed(X, seed=42)

    best_k, sil_df = choose_k_by_silhouette(Z, k_min=2, k_max=min(8, len(auc_mat)-1), seed=42)
    if best_k is None:
        best_k = 3

    km = KMeans(n_clusters=best_k, random_state=42, n_init=30)
    labels = km.fit_predict(Z)

    emb = pd.DataFrame({
        "cohort": auc_mat.index.astype(str),
        f"{method}_x": Z[:, 0],
        f"{method}_y": Z[:, 1],
        "cluster": labels.astype(int)
    }).merge(cohort_stats_df.rename(columns={"cohort":"cohort"}), on="cohort", how="left")

    sil_df.to_csv(os.path.join(TAB_DIR, f"{group_label}_kmeans_silhouette_{method}.csv"), index=False)
    emb.to_csv(os.path.join(TAB_DIR, f"{group_label}_cohort_embedding_{method}_kmeans.csv"), index=False)

    # plot silhouette
    plt.figure(figsize=(6.8, 4.6))
    sns.lineplot(data=sil_df, x="k", y="silhouette", marker="o")
    plt.title(f"{group_label.capitalize()}: KMeans silhouette on {method.upper()} embedding")
    plt.xlabel("k")
    plt.ylabel("Silhouette")
    savefig(os.path.join(FIG_DIR, f"{group_label}_silhouette_{method}"))

    # plot embedding
    plt.figure(figsize=(7.2, 6.0))
    sns.scatterplot(data=emb, x=f"{method}_x", y=f"{method}_y", hue="cluster", s=85)
    for _, r in emb.iterrows():
        plt.text(r[f"{method}_x"] + 0.02, r[f"{method}_y"] + 0.02, r["cohort"], fontsize=8)
    plt.title(f"{group_label.capitalize()} cohort structure ({method.upper()} + KMeans, k={best_k})")
    plt.xlabel(f"{method.upper()}-1")
    plt.ylabel(f"{method.upper()}-2")
    plt.legend(frameon=False, title="cluster")
    savefig(os.path.join(FIG_DIR, f"{group_label}_embedding_{method}_kmeans"))

    return emb, sil_df

# -------------------------
# Case studies (fixed: no 'ece' KeyError)
# -------------------------
def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def make_case_studies(pair_df, baselines_by_target, best_setting_key, rec_dict, group_label):
    # pick easy / medium / hard based on regret (oracle - chosen) if available,
    # else based on oracle_gain_over_pooled
    rec_df = rec_dict.get(best_setting_key, pd.DataFrame()).copy()
    by = baselines_by_target.copy()

    if len(rec_df):
        tmp = rec_df[["target", "chosen_source", "oracle_source", "regret"]].copy()
        tmp["target"] = tmp["target"].astype(str)
        by = by.merge(tmp, on="target", how="left")
        score_col = "regret"
    else:
        score_col = "oracle_gain_over_pooled"

    by = by.dropna(subset=[score_col]).sort_values(score_col, ascending=True).copy()
    if len(by) < 3:
        print(f"[{group_label}] Not enough targets for 3 case studies.")
        return

    easy   = by.iloc[0]["target"]
    medium = by.iloc[len(by)//2]["target"]
    hard   = by.iloc[-1]["target"]
    targets = [easy, medium, hard]

    # few-shot recalibration file (optional)
    fewshot_path = os.path.join(OUT_ROOT, "tables_addons", f"fewshot_recalibration_{group_label}.csv")
    fewshot = pd.read_csv(fewshot_path) if os.path.exists(fewshot_path) else pd.DataFrame()

    # choose ece columns robustly
    ece_raw_col = first_existing_col(fewshot, ["ece_raw", "ece"])
    ece_cal_col = first_existing_col(fewshot, ["ece_cal", "ece_calibrated", "ece_platt"])
    brier_raw_col = first_existing_col(fewshot, ["brier_raw", "brier"])
    brier_cal_col = first_existing_col(fewshot, ["brier_cal", "brier_calibrated"])

    for tgt in targets:
        cand = pair_df[pair_df["test"] == tgt].copy().sort_values("roc_auc", ascending=False)
        if len(cand) == 0:
            continue

        chosen_src = None
        oracle_src = str(cand.iloc[0]["train"])

        if len(rec_df) and (rec_df["target"].astype(str) == str(tgt)).any():
            chosen_src = str(rec_df.loc[rec_df["target"].astype(str) == str(tgt), "chosen_source"].iloc[0])

        plt.figure(figsize=(12.2, 4.4))

        # Panel A: transfer bars
        ax1 = plt.subplot(1, 3, 1)
        sns.barplot(data=cand, x="roc_auc", y="train", ax=ax1)
        ax1.set_title(f"{group_label}: transfer AUC to {tgt}")
        ax1.set_xlabel("ROC-AUC")
        ax1.set_ylabel("Source cohort")
        for i, (src, auc) in enumerate(zip(cand["train"], cand["roc_auc"])):
            if str(src) == oracle_src:
                ax1.text(auc + 0.004, i, "oracle", va="center", fontsize=9)
            if chosen_src is not None and str(src) == chosen_src:
                ax1.text(auc + 0.004, i, "chosen", va="center", fontsize=9)

        # Panel B: shift vs AUC scatter
        ax2 = plt.subplot(1, 3, 2)
        sns.scatterplot(data=cand, x="shift_distance", y="roc_auc", ax=ax2)
        ax2.set_title("Shift vs AUC (sources)")
        ax2.set_xlabel("Shift distance")
        ax2.set_ylabel("ROC-AUC")
        # annotate top/bottom 2
        ann = pd.concat([cand.head(2), cand.tail(2)], axis=0)
        for _, r in ann.iterrows():
            ax2.text(r["shift_distance"], r["roc_auc"], str(r["train"]), fontsize=8)

        # Panel C: few-shot recalibration curve (ECE vs k_cal)
        ax3 = plt.subplot(1, 3, 3)
        f = fewshot[fewshot["heldout"].astype(str) == str(tgt)].copy() if len(fewshot) else pd.DataFrame()
        if len(f) and ece_raw_col and ece_cal_col:
            agg = f.groupby("k_cal").agg({
                ece_raw_col: "mean",
                ece_cal_col: "mean",
                brier_raw_col: "mean" if brier_raw_col else "mean",
                brier_cal_col: "mean" if brier_cal_col else "mean",
            }).reset_index()

            ax3.plot(agg["k_cal"], agg[ece_raw_col], marker="o", label="ECE raw")
            ax3.plot(agg["k_cal"], agg[ece_cal_col], marker="o", label="ECE recal")
            ax3.set_title("Few-shot recalibration")
            ax3.set_xlabel("k_cal")
            ax3.set_ylabel("ECE")
            ax3.legend(frameon=False, fontsize=8)
        else:
            ax3.axis("off")
            ax3.text(0.05, 0.55, "Few-shot recalibration table\nnot found for this target.", transform=ax3.transAxes)

        out_base = os.path.join(FIG_DIR, f"{group_label}_case_study__{tgt}")
        savefig(out_base)

    # save chosen targets table
    pd.DataFrame({"group": group_label, "case_targets": targets}).to_csv(
        os.path.join(TAB_DIR, f"{group_label}_case_study_targets.csv"), index=False
    )
    print(f"[{group_label}] Case-study figures saved:", targets)

# -------------------------
# Shift distance definition summary (what column + descriptive stats)
# -------------------------
def save_shift_definition_summary(shift_pair_df, group_label):
    tr = _infer_col(shift_pair_df, ["train","source","train_study","study_train"])
    te = _infer_col(shift_pair_df, ["test","target","test_study","study_test"])
    sh = _infer_col(shift_pair_df, ["shift_distance","shift","distance","mean_shift","shift_mean"])

    x = shift_pair_df[sh].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    summary = pd.DataFrame([{
        "group": group_label,
        "train_col": tr,
        "test_col": te,
        "shift_col_used": sh,
        "n_pairs": int(len(x)),
        "shift_min": float(x.min()),
        "shift_p05": float(np.quantile(x, 0.05)),
        "shift_median": float(np.median(x)),
        "shift_mean": float(np.mean(x)),
        "shift_p95": float(np.quantile(x, 0.95)),
        "shift_max": float(x.max()),
    }])
    return summary

# ============================================================
# RUN
# ============================================================
print("Loaded core files from:", OUT_ROOT)

# Load core outputs
pred_species = load_csv_req(os.path.join(OUT_ROOT, "loso_species_predictions_next.csv"))
pred_pathways = load_csv_req(os.path.join(OUT_ROOT, "loso_pathways_predictions_next.csv"))

pair_auc_species = load_csv_req(os.path.join(OUT_ROOT, "train_test_auc_species.csv"))
pair_auc_pathways = load_csv_req(os.path.join(OUT_ROOT, "train_test_auc_pathways.csv"))

shift_pair_species = load_csv_req(os.path.join(OUT_ROOT, "shift_vs_auc_pairwise_species.csv"))
shift_pair_pathways = load_csv_req(os.path.join(OUT_ROOT, "shift_vs_auc_pairwise_pathways.csv"))

pred_sp = standardize_pred_df(pred_species, "species")
pred_pw = standardize_pred_df(pred_pathways, "pathways")

stats_sp = cohort_stats_from_pred(pred_sp)
stats_pw = cohort_stats_from_pred(pred_pw)

stats_sp.to_csv(os.path.join(TAB_DIR, "cohort_stats_species_from_predictions.csv"), index=False)
stats_pw.to_csv(os.path.join(TAB_DIR, "cohort_stats_pathways_from_predictions.csv"), index=False)

pair_sp = build_pairwise_dataset(pair_auc_species, shift_pair_species, stats_sp, "species")
pair_pw = build_pairwise_dataset(pair_auc_pathways, shift_pair_pathways, stats_pw, "pathways")

print("Pairwise datasets built:",
      "species", pair_sp.shape, "pathways", pair_pw.shape)

# Shift definition summary
shift_def = pd.concat([
    save_shift_definition_summary(shift_pair_species, "species"),
    save_shift_definition_summary(shift_pair_pathways, "pathways"),
], ignore_index=True)
shift_def_path = os.path.join(TAB_DIR, "shift_distance_definition_summary.csv")
shift_def.to_csv(shift_def_path, index=False)
print("Saved shift definition summary:", shift_def_path)

# Heatmaps
make_big_heatmaps(pair_sp, "species")
make_big_heatmaps(pair_pw, "pathways")
print("Saved big heatmaps (safe) + scatterplots.")

# LOSO pooled baseline table
loso_sp = loso_auc_table(pred_sp, "species")
loso_pw = loso_auc_table(pred_pw, "pathways")
loso_sp.to_csv(os.path.join(TAB_DIR, "loso_auc_species.csv"), index=False)
loso_pw.to_csv(os.path.join(TAB_DIR, "loso_auc_pathways.csv"), index=False)

# Baselines
base_sp_by, base_sp_sum = compute_baselines(pair_sp, loso_sp, "species")
base_pw_by, base_pw_sum = compute_baselines(pair_pw, loso_pw, "pathways")
print("Saved baseline tables + baseline boxplots.")

# Ablation grid + best setting + predicted AUC heatmap
comp_sp, best_sp_key, rec_sp, predm_sp = run_ablation_grid(pair_sp, "species")
comp_pw, best_pw_key, rec_pw, predm_pw = run_ablation_grid(pair_pw, "pathways")

save_predicted_auc_heatmap(predm_sp[best_sp_key], "species", best_sp_key)
save_predicted_auc_heatmap(predm_pw[best_pw_key], "pathways", best_pw_key)
print("Saved predicted-AUC heatmaps for best settings.")

# Multiple comparisons (BH-FDR)
per_target_shift_auc_tests(pair_sp, "species")
per_target_shift_auc_tests(pair_pw, "pathways")
print("Saved per-target BH-FDR tables and plots.")

# Robustness
rob_sp = robustness_suite(pair_sp, stats_sp, "species", B=500)
rob_pw = robustness_suite(pair_pw, stats_pw, "pathways", B=500)
print("Saved robustness outputs (largest/smallest, winsor, LOO, bootstrap).")

# Unsupervised structure (UMAP/PCA + KMeans)
unsupervised_cohort_structure(pair_sp, stats_sp, "species")
unsupervised_cohort_structure(pair_pw, stats_pw, "pathways")
print("Saved unsupervised cohort structure outputs.")

# Case studies (fixed)
make_case_studies(pair_sp, base_sp_by, best_sp_key, rec_sp, "species")
make_case_studies(pair_pw, base_pw_by, best_pw_key, rec_pw, "pathways")
print("Case-study figures saved.")

# Output index
index_rows = []
for root, _, files in os.walk(TAB_DIR):
    for f in sorted(files):
        if f.endswith(".csv"):
            index_rows.append({"type": "table", "path": os.path.join(root, f)})
for root, _, files in os.walk(FIG_DIR):
    for f in sorted(files):
        if f.endswith(".png") or f.endswith(".pdf"):
            index_rows.append({"type": "figure", "path": os.path.join(root, f)})

index_df = pd.DataFrame(index_rows)
index_path = os.path.join(OUT_ROOT, "pub_safe_suite_v2_output_index.csv")
index_df.to_csv(index_path, index=False)

print("\nDone.")
print("Tables:", TAB_DIR)
print("Figures:", FIG_DIR)
print("Index:", index_path)


##adjusted regression / extra analyses



In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

OUT_DIR = "/content/curatedMetagenomicData_EH/analysis_outputs_crc"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def read_csv(name):
    path = os.path.join(OUT_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def safe_log(x):
    x = np.asarray(x, dtype=float)
    x = np.clip(x, 1.0, None)  # avoid log(0)
    return np.log(x)

def fit_adjusted_regression(df, x_col, y_col, ntrain_col, ntest_col, label):
    """
    Fits OLS via sklearn LinearRegression:
      y ~ shift + log(n_train) + log(n_test)
    Returns coefficients, standardized coefficients, and R^2.
    """
    d = df[[x_col, y_col, ntrain_col, ntest_col]].dropna().copy()
    if len(d) < 5:
        return pd.DataFrame([{
            "label": label,
            "n": len(d),
            "r2": np.nan,
            "coef_shift": np.nan,
            "coef_log_ntrain": np.nan,
            "coef_log_ntest": np.nan,
            "stdcoef_shift": np.nan,
            "stdcoef_log_ntrain": np.nan,
            "stdcoef_log_ntest": np.nan
        }])

    X = pd.DataFrame({
        "shift": d[x_col].astype(float).values,
        "log_n_train": safe_log(d[ntrain_col].values),
        "log_n_test": safe_log(d[ntest_col].values),
    })
    y = d[y_col].astype(float).values

    # unstandardized OLS
    lr = LinearRegression()
    lr.fit(X, y)
    r2 = lr.score(X, y)

    # standardized betas (z-score X and y)
    Xs = StandardScaler().fit_transform(X.values)
    ys = (y - y.mean()) / (y.std(ddof=0) + 1e-12)
    lrs = LinearRegression()
    lrs.fit(Xs, ys)

    return pd.DataFrame([{
        "label": label,
        "n": int(len(d)),
        "r2": float(r2),
        "coef_shift": float(lr.coef_[0]),
        "coef_log_ntrain": float(lr.coef_[1]),
        "coef_log_ntest": float(lr.coef_[2]),
        "stdcoef_shift": float(lrs.coef_[0]),
        "stdcoef_log_ntrain": float(lrs.coef_[1]),
        "stdcoef_log_ntest": float(lrs.coef_[2]),
        "intercept": float(lr.intercept_)
    }])

def corr_table(x, y, label):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(d) < 3:
        return pd.DataFrame([{
            "label": label, "n": len(d),
            "pearson_r": np.nan, "pearson_p": np.nan,
            "spearman_r": np.nan, "spearman_p": np.nan
        }])
    pr, pp = pearsonr(d["x"].values, d["y"].values)
    sr, sp = spearmanr(d["x"].values, d["y"].values)
    return pd.DataFrame([{
        "label": label, "n": int(len(d)),
        "pearson_r": float(pr), "pearson_p": float(pp),
        "spearman_r": float(sr), "spearman_p": float(sp)
    }])

# -------------------------
# Load inputs
# -------------------------
loso_species = read_csv("shift_vs_auc_loso_species.csv")   # columns: heldout, shift_distance, roc_auc, n_train, n_test, n_feat
loso_pathways = read_csv("shift_vs_auc_loso_pathways.csv")

pair_species = read_csv("shift_vs_auc_pairwise_species.csv")  # columns: train, test, shift_distance, roc_auc
pair_pathways = read_csv("shift_vs_auc_pairwise_pathways.csv")

# For pairwise we need n_train and n_test per pair. We can derive from train/test sizes using LOSO tables:
# LOSO tables contain n_test per heldout cohort; build mapping cohort->n
def cohort_sizes_from_loso(loso_df):
    m = {}
    for _, r in loso_df.iterrows():
        m[str(r["heldout"])] = int(r["n_test"])
    return m

n_species = cohort_sizes_from_loso(loso_species)
n_pathways = cohort_sizes_from_loso(loso_pathways)

def add_pair_sizes(pair_df, n_map):
    d = pair_df.copy()
    d["n_train"] = d["train"].map(lambda s: n_map.get(str(s), np.nan))
    d["n_test"]  = d["test"].map(lambda s: n_map.get(str(s), np.nan))
    return d

pair_species2 = add_pair_sizes(pair_species, n_species)
pair_pathways2 = add_pair_sizes(pair_pathways, n_pathways)

# -------------------------
# A) Adjusted regressions
# -------------------------
reg_results = pd.concat([
    fit_adjusted_regression(loso_species, "shift_distance", "roc_auc", "n_train", "n_test", "species_loso_adjusted"),
    fit_adjusted_regression(loso_pathways, "shift_distance", "roc_auc", "n_train", "n_test", "pathways_loso_adjusted"),
    fit_adjusted_regression(pair_species2, "shift_distance", "roc_auc", "n_train", "n_test", "species_pairwise_adjusted"),
    fit_adjusted_regression(pair_pathways2, "shift_distance", "roc_auc", "n_train", "n_test", "pathways_pairwise_adjusted"),
], axis=0)

reg_path = os.path.join(OUT_DIR, "adjusted_regression_results.csv")
reg_results.to_csv(reg_path, index=False)
print("Saved:", reg_path)
display(reg_results)

# -------------------------
# B) Hardness index vs LOSO AUC
# -------------------------
st = read_csv("transferability_source_target_scores.csv")  # cohort, mean_auc_as_source, mean_auc_as_target, group

# Build LOSO AUC lookup per group
loso_auc_species = loso_species[["heldout", "roc_auc"]].rename(columns={"heldout": "cohort"}).copy()
loso_auc_pathways = loso_pathways[["heldout", "roc_auc"]].rename(columns={"heldout": "cohort"}).copy()

st_species = st[st["group"] == "species"].copy()
st_pathways = st[st["group"] == "pathways"].copy()

m_species = st_species.merge(loso_auc_species, on="cohort", how="inner")
m_pathways = st_pathways.merge(loso_auc_pathways, on="cohort", how="inner")

hard_corrs = pd.concat([
    corr_table(m_species["mean_auc_as_target"], m_species["roc_auc"], "species_hardness_vs_loso_auc"),
    corr_table(m_pathways["mean_auc_as_target"], m_pathways["roc_auc"], "pathways_hardness_vs_loso_auc"),
], axis=0)

hard_path = os.path.join(OUT_DIR, "hardness_vs_loso_correlations.csv")
hard_corrs.to_csv(hard_path, index=False)
print("Saved:", hard_path)
display(hard_corrs)

# -------------------------
# Plot hardness vs LOSO AUC
# -------------------------
def plot_hardness_vs_loso(df, title, outname):
    d = df.dropna(subset=["mean_auc_as_target", "roc_auc"]).copy()
    plt.figure(figsize=(6.8, 5.2))
    sns.regplot(data=d, x="mean_auc_as_target", y="roc_auc", scatter_kws={"s": 55}, line_kws={"linewidth": 2})
    for _, r in d.iterrows():
        plt.annotate(str(r["cohort"]), (r["mean_auc_as_target"], r["roc_auc"]), fontsize=8)
    plt.xlabel("Mean external AUC as target (hardness index)")
    plt.ylabel("LOSO ROC-AUC")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, outname + ".png"), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, outname + ".pdf"))
    plt.close()

plot_hardness_vs_loso(m_species, "Species: target hardness vs LOSO ROC-AUC", "hardness_vs_loso_species")
plot_hardness_vs_loso(m_pathways, "Pathways: target hardness vs LOSO ROC-AUC", "hardness_vs_loso_pathways")

print("Figures saved in:", FIG_DIR)
